In [ ]:
# ============================================================================
# MAL-PhishNet — Cell 1: Environment Setup (Enterprise/Research-Grade)
# Multimodal Adversarial Learning Phishing Detection Network
# ============================================================================
# This cell initializes the entire runtime environment for the notebook:
#   - Package auto-installation + version logging
#   - Logging infrastructure (console + training.log + error.log)
#   - Global CONFIG dictionary (single source of truth for all hyperparameters)
#   - Config SHA256 hashing + config export (versioned, reproducible)
#   - Experiment UUID generation (persisted across reruns)
#   - Deterministic seeding + random-state save/restore (Python/NumPy/Torch/CUDA)
#   - GPU detection, AMP capability detection, cuDNN benchmark configuration
#   - Google Drive mount with retry and failure handling
#   - Canonical MAL-PhishNet directory structure creation (incl. backups)
#   - Disk space checker, RAM/GPU resource monitor
#   - Checkpoint versioning, integrity verification, and rotating backups
#   - Metrics CSV logger (append-safe) + history.json (append mode)
#   - last_status.json experiment-state tracker + global error hook
#
# All subsequent cells (2-8) depend exclusively on objects defined here:
#   CONFIG, DEVICE, LOGGER, PATHS, EXPERIMENT_UUID, CONFIG_HASH,
#   set_global_seed(), get_rng_state(), set_rng_state(),
#   verify_checkpoint_integrity(), get_latest_checkpoint_path(),
#   backup_checkpoint(), MetricsCSVLogger, append_history(),
#   write_status(), get_system_resource_snapshot(), check_disk_space()
# ============================================================================

import os
import sys
import io
import json
import time
import uuid
import random
import shutil
import socket
import logging
import hashlib
import pickle
import platform
import warnings
import traceback
import subprocess
import importlib
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, Optional, Tuple, List

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# ----------------------------------------------------------------------------
# 0. PACKAGE AUTO-INSTALLER (runs before third-party imports)
# ----------------------------------------------------------------------------

REQUIRED_PACKAGES: Dict[str, str] = {
    # import_name : pip_package_name
    "psutil": "psutil",
    "gdown": "gdown",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "PIL": "pillow",
}


def ensure_packages_installed(packages: Dict[str, str],
                               quiet: bool = True) -> Dict[str, str]:
    """
    Verify that required third-party packages are importable; install any
    that are missing via pip, without ever raising on a working Colab image.

    Args:
        packages: Mapping of {import_name: pip_install_name}.
        quiet: If True, suppresses pip's stdout/stderr chatter.

    Returns:
        A dict of {import_name: status} where status is "already_installed",
        "installed", or "install_failed:<reason>".
    """
    results: Dict[str, str] = {}
    for import_name, pip_name in packages.items():
        try:
            importlib.import_module(import_name)
            results[import_name] = "already_installed"
            continue
        except ImportError:
            pass

        cmd = [sys.executable, "-m", "pip", "install", "-q" if quiet else "", pip_name]
        cmd = [c for c in cmd if c]
        try:
            subprocess.check_call(cmd, stdout=subprocess.DEVNULL if quiet else None,
                                   stderr=subprocess.DEVNULL if quiet else None)
            importlib.import_module(import_name)
            results[import_name] = "installed"
        except (subprocess.CalledProcessError, ImportError) as exc:
            results[import_name] = f"install_failed:{exc}"
    return results


_PACKAGE_INSTALL_RESULTS: Dict[str, str] = ensure_packages_installed(REQUIRED_PACKAGES)

import numpy as np
import psutil
import torch
import torch.backends.cudnn as cudnn


# ----------------------------------------------------------------------------
# 1. RUNTIME CONSTANTS
# ----------------------------------------------------------------------------

IS_COLAB: bool = "google.colab" in sys.modules
PROJECT_NAME: str = "MAL-PhishNet"
DRIVE_MOUNT_POINT: str = "/content/drive"
DRIVE_ROOT: str = os.path.join(DRIVE_MOUNT_POINT, "MyDrive", PROJECT_NAME)
NOTEBOOK_START_TIME: float = time.time()
NOTEBOOK_RUN_ID: str = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


# ----------------------------------------------------------------------------
# 2. CONFIG — SINGLE SOURCE OF TRUTH FOR ALL HYPERPARAMETERS / PATHS / FLAGS
# ----------------------------------------------------------------------------
# Every subsequent cell reads from this dictionary exclusively.
# No hardcoded hyperparameters or paths are permitted anywhere else
# in the notebook. To change behavior, edit CONFIG only.
# ----------------------------------------------------------------------------

CONFIG: Dict[str, Any] = {
    # ---- Experiment identity -------------------------------------------
    "project_name": PROJECT_NAME,
    "experiment_name": "malphishnet_v1",
    "seed": 42,
    "experiment_uuid": None,   # populated in section 6b
    "config_hash": None,       # populated in section 9

    # ---- Paths (populated after Drive mount in section 7) --------------
    "drive_root": DRIVE_ROOT,
    "paths": {},  # filled by `build_directory_structure()`

    # ---- Dataset ---------------------------------------------------------
    "dataset": {
        "primary_source": "mendeley_phishing_url",
        "secondary_source": "phishtank",
        "tertiary_source": "nazario_email_corpus",
        "train_split": 0.7,
        "val_split": 0.15,
        "test_split": 0.15,
        "max_url_length": 200,
        "max_email_tokens": 512,
        "image_size": 224,
    },

    # ---- Dataloader ------------------------------------------------------
    "dataloader": {
        "batch_size": 32,
        "num_workers": 2,          # Colab free tier: 2 vCPUs typical
        "pin_memory": True,
        "persistent_workers": True,
        "prefetch_factor": 4,
        "drop_last": False,
    },

    # ---- Model -------------------------------------------------------------
    "model": {
        "url_embed_dim": 128,
        "text_hidden_dim": 256,
        "vision_backbone": "resnet18",
        "fusion_dim": 512,
        "num_classes": 2,
        "dropout": 0.3,
    },

    # ---- Optimization ------------------------------------------------------
    "train": {
        "epochs": 50,
        "learning_rate": 3e-4,
        "weight_decay": 1e-4,
        "optimizer": "adamw",
        "scheduler": "cosine_annealing_warm_restarts",
        "scheduler_t0": 10,
        "scheduler_tmult": 2,
        "grad_clip_norm": 1.0,
        "amp_enabled": True,       # overridden automatically if GPU unsupported
        "early_stopping_patience": 10,
        "resume": True,            # NEVER restart from epoch 0 unless False
        "save_every_epoch": True,
        "log_every_n_steps": 25,
    },

    # ---- Checkpointing -------------------------------------------------------
    "checkpoint": {
        "keep_last_n_backups": 5,
        "versioned_filename_template": "checkpoint_epoch{epoch:04d}_{timestamp}.pt",
        "latest_filename": "checkpoint_latest.pt",
        "backup_subdir": "backups",
    },

    # ---- Reproducibility / performance -------------------------------------
    "system": {
        "cudnn_benchmark": True,
        "cudnn_deterministic": False,   # benchmark=True & deterministic=True conflict
        "deterministic_seed": True,
        "num_gpus_expected": 1,
        "gpu_name_hint": "Tesla T4",
        "min_free_disk_gb": 2.0,
        "min_free_ram_gb": 1.0,
    },

    # ---- Logging ------------------------------------------------------------
    "logging": {
        "log_filename": "training.log",
        "error_log_filename": "error.log",
        "metrics_filename": "metrics.csv",
        "history_filename": "history.json",
        "status_filename": "last_status.json",
        "log_level": "INFO",
    },
}


# ----------------------------------------------------------------------------
# 3. LOGGING INFRASTRUCTURE (console + training.log + error.log)
# ----------------------------------------------------------------------------

def build_logger(name: str, log_file: Optional[str] = None,
                  error_log_file: Optional[str] = None,
                  level: str = "INFO") -> logging.Logger:
    """
    Construct a professional multi-handler logger (console + training log +
    a dedicated error-only log).

    Args:
        name: Logger namespace, typically the project name.
        log_file: Absolute path to the general log file. If None, general
            file logging is skipped (used before Drive is mounted).
        error_log_file: Absolute path to a WARNING+/ERROR+ only log file.
        level: Logging verbosity level name (e.g. "INFO", "DEBUG").

    Returns:
        A fully configured `logging.Logger` instance. Idempotent: calling
        this multiple times with the same `name` will not duplicate handlers.
    """
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level.upper(), logging.INFO))
    logger.propagate = False

    if logger.handlers:
        # Already configured — avoid duplicate handlers on notebook re-run.
        logger.handlers.clear()

    formatter = logging.Formatter(
        fmt="[%(asctime)s] [%(levelname)-8s] [%(name)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console_handler = logging.StreamHandler(stream=sys.stdout)
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    if log_file is not None:
        try:
            os.makedirs(os.path.dirname(log_file), exist_ok=True)
            file_handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
            file_handler.setFormatter(formatter)
            logger.addHandler(file_handler)
        except OSError as exc:
            logger.warning(f"Could not attach file handler at {log_file}: {exc}")

    if error_log_file is not None:
        try:
            os.makedirs(os.path.dirname(error_log_file), exist_ok=True)
            error_handler = logging.FileHandler(error_log_file, mode="a", encoding="utf-8")
            error_handler.setFormatter(formatter)
            error_handler.setLevel(logging.WARNING)
            logger.addHandler(error_handler)
        except OSError as exc:
            logger.warning(f"Could not attach error-log handler at {error_log_file}: {exc}")

    return logger


# Bootstrap logger before Drive is mounted (file handlers attached later).
LOGGER: logging.Logger = build_logger(
    PROJECT_NAME, log_file=None, error_log_file=None,
    level=CONFIG["logging"]["log_level"],
)
LOGGER.info(f"{PROJECT_NAME} — Cell 1: Environment Setup starting.")
LOGGER.info(f"Python version: {platform.python_version()} | "
            f"PyTorch version: {torch.__version__} | Colab: {IS_COLAB}")
for pkg, status in _PACKAGE_INSTALL_RESULTS.items():
    level_fn = LOGGER.warning if "failed" in status else LOGGER.info
    level_fn(f"Package check — {pkg}: {status}")


def install_uncaught_exception_hook(logger: logging.Logger) -> None:
    """
    Redirect uncaught exceptions to the logger (and hence error.log) instead
    of only printing to stderr, so failures are captured for postmortem
    analysis even after the Colab runtime is recycled.

    Args:
        logger: The logger instance to route uncaught exceptions through.
    """
    def _hook(exc_type: type, exc_value: BaseException, exc_tb: Any) -> None:
        logger.error(
            "Uncaught exception:\n" +
            "".join(traceback.format_exception(exc_type, exc_value, exc_tb))
        )
        sys.__excepthook__(exc_type, exc_value, exc_tb)

    sys.excepthook = _hook


install_uncaught_exception_hook(LOGGER)


# ----------------------------------------------------------------------------
# 4. REPRODUCIBILITY — GLOBAL SEEDING + RANDOM STATE SAVE/RESTORE
# ----------------------------------------------------------------------------

def set_global_seed(seed: int, deterministic: bool = True) -> None:
    """
    Seed all relevant RNGs (Python, NumPy, PyTorch CPU/CUDA) for reproducibility.

    Args:
        seed: Integer seed value applied uniformly across all libraries.
        deterministic: If True, forces deterministic cuDNN algorithms
            (may reduce throughput). If False, cuDNN is allowed to
            autotune (used together with `cudnn.benchmark=True`).

    Raises:
        ValueError: If `seed` is negative.
    """
    if seed < 0:
        raise ValueError(f"Seed must be non-negative, got {seed}.")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if deterministic:
        cudnn.deterministic = True
        cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)
    else:
        cudnn.deterministic = False

    LOGGER.info(f"Global seed set to {seed} (deterministic={deterministic}).")


def get_rng_state() -> Dict[str, Any]:
    """
    Capture the complete RNG state (Python, NumPy, Torch CPU, Torch CUDA)
    so training can be resumed bit-for-bit deterministically.

    Returns:
        A dictionary suitable for embedding directly inside a checkpoint.
    """
    return {
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_cpu_state": torch.get_rng_state(),
        "torch_cuda_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def set_rng_state(state: Dict[str, Any]) -> None:
    """
    Restore a previously captured RNG state (see `get_rng_state`).

    Args:
        state: Dictionary produced by `get_rng_state`. Missing or malformed
            keys are skipped individually with a warning rather than
            aborting the whole restoration.
    """
    try:
        if state.get("python_random_state") is not None:
            random.setstate(state["python_random_state"])
        if state.get("numpy_random_state") is not None:
            np.random.set_state(state["numpy_random_state"])
        if state.get("torch_cpu_state") is not None:
            torch.set_rng_state(state["torch_cpu_state"].to(torch.uint8) if isinstance(
                state["torch_cpu_state"], torch.Tensor) else state["torch_cpu_state"])
        if state.get("torch_cuda_state") is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(state["torch_cuda_state"])
        LOGGER.info("RNG state successfully restored from checkpoint.")
    except (RuntimeError, ValueError, TypeError) as exc:
        LOGGER.warning(f"Partial/failed RNG state restoration: {exc}. "
                        f"Continuing with current (unrestored) RNG state.")


# ----------------------------------------------------------------------------
# 5. DEVICE / GPU / AMP DETECTION
# ----------------------------------------------------------------------------

@dataclass
class DeviceInfo:
    """Container describing the detected compute environment."""
    device: torch.device
    device_name: str
    is_cuda: bool
    supports_amp: bool
    compute_capability: Optional[Tuple[int, int]]
    total_memory_gb: Optional[float]


def detect_device() -> DeviceInfo:
    """
    Detect available compute device and its AMP (mixed precision) support.

    Falls back gracefully to CPU with a warning if no GPU is available,
    rather than raising — the notebook must remain runnable on any tier.

    Returns:
        A populated `DeviceInfo` dataclass instance.
    """
    if torch.cuda.is_available():
        try:
            index = torch.cuda.current_device()
            props = torch.cuda.get_device_properties(index)
            name = props.name
            cc = (props.major, props.minor)
            mem_gb = round(props.total_memory / (1024 ** 3), 2)
            # AMP (fp16) requires compute capability >= 7.0 for full benefit;
            # Tesla T4 (Turing, cc 7.5) fully supports it.
            supports_amp = cc[0] >= 7
            device = torch.device("cuda")
            LOGGER.info(f"GPU detected: {name} | Compute Capability: "
                        f"{cc[0]}.{cc[1]} | VRAM: {mem_gb} GB | AMP supported: {supports_amp}")
            return DeviceInfo(device, name, True, supports_amp, cc, mem_gb)
        except (RuntimeError, AssertionError) as exc:
            LOGGER.error(f"CUDA reported available but device query failed: {exc}")
            LOGGER.warning("Falling back to CPU.")
    else:
        LOGGER.warning("No CUDA-capable GPU detected. Falling back to CPU. "
                        "Training will be significantly slower.")

    return DeviceInfo(torch.device("cpu"), "CPU", False, False, None, None)


DEVICE_INFO: DeviceInfo = detect_device()
DEVICE: torch.device = DEVICE_INFO.device

# AMP is only actually enabled if BOTH the config requests it AND hardware supports it.
CONFIG["train"]["amp_enabled"] = bool(
    CONFIG["train"]["amp_enabled"] and DEVICE_INFO.supports_amp
)
if CONFIG["train"]["amp_enabled"]:
    LOGGER.info("Automatic Mixed Precision (AMP) ENABLED for this session.")
else:
    LOGGER.warning("AMP DISABLED (unsupported hardware or explicitly turned off).")

# cuDNN performance configuration.
cudnn.benchmark = bool(CONFIG["system"]["cudnn_benchmark"] and DEVICE_INFO.is_cuda)
LOGGER.info(f"cuDNN benchmark mode: {cudnn.benchmark}")

# Apply reproducible seeding now that device/backends are configured.
set_global_seed(
    seed=CONFIG["seed"],
    deterministic=CONFIG["system"]["deterministic_seed"] and not cudnn.benchmark,
)


# ----------------------------------------------------------------------------
# 6. GOOGLE DRIVE MOUNT (WITH RETRY + GRACEFUL DEGRADATION)
# ----------------------------------------------------------------------------

def mount_google_drive(mount_point: str = DRIVE_MOUNT_POINT,
                        max_retries: int = 3,
                        retry_delay_sec: float = 3.0) -> bool:
    """
    Mount Google Drive inside the Colab runtime, retrying on transient failure.

    If not running inside Colab, this is a no-op that returns False so the
    notebook can gracefully fall back to local (ephemeral) storage.

    Args:
        mount_point: Target mount path inside the VM.
        max_retries: Number of mount attempts before giving up.
        retry_delay_sec: Seconds to wait between retries.

    Returns:
        True if Drive is mounted and accessible, False otherwise.
    """
    if not IS_COLAB:
        LOGGER.warning("Not running inside Google Colab — skipping Drive mount. "
                        "Falling back to local filesystem storage.")
        return False

    from google.colab import drive as colab_drive  # noqa: import-outside-toplevel

    for attempt in range(1, max_retries + 1):
        try:
            if os.path.ismount(mount_point) or os.path.exists(
                os.path.join(mount_point, "MyDrive")
            ):
                LOGGER.info("Google Drive already mounted.")
                return True

            LOGGER.info(f"Mounting Google Drive (attempt {attempt}/{max_retries})...")
            colab_drive.mount(mount_point, force_remount=False)

            if os.path.exists(os.path.join(mount_point, "MyDrive")):
                LOGGER.info("Google Drive mounted successfully.")
                return True
            raise RuntimeError("Mount call succeeded but MyDrive path not found.")

        except Exception as exc:  # noqa: broad-except — mounting can fail many ways
            LOGGER.error(f"Drive mount attempt {attempt} failed: {exc}")
            if attempt < max_retries:
                time.sleep(retry_delay_sec)
            else:
                LOGGER.error("All Drive mount attempts exhausted. "
                             "Falling back to local ephemeral storage.")
                return False
    return False


DRIVE_MOUNTED: bool = mount_google_drive()

# If Drive is unavailable, redirect the entire project root to local disk so
# the notebook remains fully functional (with a clear warning that state
# will NOT persist across Colab runtime restarts).
if not DRIVE_MOUNTED:
    DRIVE_ROOT = os.path.join("/content", PROJECT_NAME)
    CONFIG["drive_root"] = DRIVE_ROOT
    LOGGER.warning(f"Persistent storage unavailable. Using local path: {DRIVE_ROOT} "
                    f"(NOT persistent across sessions).")


# ----------------------------------------------------------------------------
# 6b. EXPERIMENT UUID (persisted across reruns of the same experiment)
# ----------------------------------------------------------------------------

def get_or_create_experiment_uuid(root: str, experiment_name: str) -> str:
    """
    Retrieve a persisted experiment UUID for `experiment_name`, or mint a
    new one if none exists yet. This guarantees the same experiment retains
    a stable identity across Colab session restarts, while distinct
    experiment names never collide.

    Args:
        root: Project root directory (Drive or local fallback).
        experiment_name: Logical name of the experiment (from CONFIG).

    Returns:
        A UUID4 string uniquely identifying this experiment.
    """
    registry_path = os.path.join(root, ".experiment_registry.json")
    registry: Dict[str, str] = {}

    if os.path.isfile(registry_path):
        try:
            with open(registry_path, "r", encoding="utf-8") as fh:
                registry = json.load(fh)
        except (json.JSONDecodeError, OSError) as exc:
            LOGGER.warning(f"Experiment registry unreadable ({exc}); starting fresh.")
            registry = {}

    if experiment_name in registry:
        return registry[experiment_name]

    new_uuid = str(uuid.uuid4())
    registry[experiment_name] = new_uuid
    try:
        os.makedirs(root, exist_ok=True)
        with open(registry_path, "w", encoding="utf-8") as fh:
            json.dump(registry, fh, indent=2)
    except OSError as exc:
        LOGGER.warning(f"Could not persist experiment registry: {exc}")

    return new_uuid


EXPERIMENT_UUID: str = get_or_create_experiment_uuid(DRIVE_ROOT, CONFIG["experiment_name"])
CONFIG["experiment_uuid"] = EXPERIMENT_UUID
LOGGER.info(f"Experiment UUID: {EXPERIMENT_UUID}")


# ----------------------------------------------------------------------------
# 7. CANONICAL DIRECTORY STRUCTURE
# ----------------------------------------------------------------------------

def build_directory_structure(root: str) -> Dict[str, str]:
    """
    Create (idempotently) the full MAL-PhishNet directory tree.

    Structure:
        root/
        ├── datasets/{raw,processed,cache}
        ├── checkpoints/{backups}
        ├── outputs/{figures,metrics,predictions}
        ├── logs/
        └── models/

    Args:
        root: Absolute path to the project root (Drive or local fallback).

    Returns:
        A flat dictionary mapping logical path keys to absolute directory paths.

    Raises:
        OSError: If a directory cannot be created due to permissions or
            a full disk, after logging the underlying cause.
    """
    layout = {
        "root": root,
        "datasets_raw": os.path.join(root, "datasets", "raw"),
        "datasets_processed": os.path.join(root, "datasets", "processed"),
        "datasets_cache": os.path.join(root, "datasets", "cache"),
        "checkpoints": os.path.join(root, "checkpoints"),
        "checkpoints_backups": os.path.join(
            root, "checkpoints", CONFIG["checkpoint"]["backup_subdir"]
        ),
        "outputs_figures": os.path.join(root, "outputs", "figures"),
        "outputs_metrics": os.path.join(root, "outputs", "metrics"),
        "outputs_predictions": os.path.join(root, "outputs", "predictions"),
        "logs": os.path.join(root, "logs"),
        "models": os.path.join(root, "models"),
    }

    for key, path in layout.items():
        try:
            os.makedirs(path, exist_ok=True)
        except OSError as exc:
            LOGGER.error(f"Failed to create directory '{key}' at {path}: {exc}")
            raise

    LOGGER.info(f"Directory structure verified/created under: {root}")
    return layout


PATHS: Dict[str, str] = build_directory_structure(DRIVE_ROOT)
CONFIG["paths"] = PATHS

# Re-attach the logger with persistent file handlers now that `logs/` exists.
LOGGER = build_logger(
    PROJECT_NAME,
    log_file=os.path.join(PATHS["logs"], CONFIG["logging"]["log_filename"]),
    error_log_file=os.path.join(PATHS["logs"], CONFIG["logging"]["error_log_filename"]),
    level=CONFIG["logging"]["log_level"],
)
install_uncaught_exception_hook(LOGGER)  # re-bind hook to the upgraded logger
LOGGER.info("File-based logging attached (training.log + error.log). "
            "All subsequent messages persist to disk.")


# ----------------------------------------------------------------------------
# 8. DISK SPACE + RAM/GPU RESOURCE MONITORING
# ----------------------------------------------------------------------------

def check_disk_space(path: str, min_free_gb: float) -> Tuple[bool, float]:
    """
    Verify sufficient free disk space is available at `path`.

    Args:
        path: Directory to check (must exist).
        min_free_gb: Minimum acceptable free space in gigabytes.

    Returns:
        Tuple of (is_sufficient, free_gb).
    """
    try:
        usage = shutil.disk_usage(path)
        free_gb = round(usage.free / (1024 ** 3), 2)
        sufficient = free_gb >= min_free_gb
        if not sufficient:
            LOGGER.warning(f"Low disk space at {path}: {free_gb} GB free "
                            f"(minimum required: {min_free_gb} GB).")
        else:
            LOGGER.info(f"Disk space OK at {path}: {free_gb} GB free.")
        return sufficient, free_gb
    except OSError as exc:
        LOGGER.error(f"Disk space check failed for {path}: {exc}")
        return False, -1.0


def get_system_resource_snapshot() -> Dict[str, Any]:
    """
    Capture a point-in-time snapshot of RAM and GPU memory usage.

    Returns:
        Dictionary with RAM (total/used/available/percent, GB) and, if CUDA
        is available, GPU memory (allocated/reserved/total, GB).
    """
    vm = psutil.virtual_memory()
    snapshot: Dict[str, Any] = {
        "ram_total_gb": round(vm.total / (1024 ** 3), 2),
        "ram_used_gb": round(vm.used / (1024 ** 3), 2),
        "ram_available_gb": round(vm.available / (1024 ** 3), 2),
        "ram_percent_used": vm.percent,
        "gpu_allocated_gb": None,
        "gpu_reserved_gb": None,
        "gpu_total_gb": DEVICE_INFO.total_memory_gb,
    }

    if torch.cuda.is_available():
        snapshot["gpu_allocated_gb"] = round(torch.cuda.memory_allocated() / (1024 ** 3), 3)
        snapshot["gpu_reserved_gb"] = round(torch.cuda.memory_reserved() / (1024 ** 3), 3)

    if vm.available / (1024 ** 3) < CONFIG["system"]["min_free_ram_gb"]:
        LOGGER.warning(f"Low system RAM available: {snapshot['ram_available_gb']} GB.")

    return snapshot


DISK_OK, DISK_FREE_GB = check_disk_space(DRIVE_ROOT, CONFIG["system"]["min_free_disk_gb"])
RESOURCE_SNAPSHOT: Dict[str, Any] = get_system_resource_snapshot()
LOGGER.info(f"Resource snapshot: {RESOURCE_SNAPSHOT}")


# ----------------------------------------------------------------------------
# 9. CONFIG HASHING (SHA256) + CONFIG EXPORT
# ----------------------------------------------------------------------------

def compute_config_hash(config: Dict[str, Any]) -> str:
    """
    Compute a stable SHA256 hash of the CONFIG dictionary for reproducibility
    tracking (e.g. detecting whether a resumed run's hyperparameters drifted
    from the run that produced an existing checkpoint).

    Args:
        config: The CONFIG dictionary. The volatile 'config_hash' and
            'experiment_uuid' fields are excluded before hashing so the hash
            reflects hyperparameters only, not run identity.

    Returns:
        Hex-encoded SHA256 digest string.
    """
    hashable = {k: v for k, v in config.items() if k not in ("config_hash", "experiment_uuid")}
    serialized = json.dumps(hashable, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(serialized).hexdigest()


def export_config(config: Dict[str, Any], output_dir: str, config_hash: str) -> str:
    """
    Persist the fully-resolved CONFIG dictionary to disk as JSON, tagged
    with its SHA256 hash for provenance and reproducibility auditing.

    Args:
        config: The CONFIG dictionary to export.
        output_dir: Directory in which to write the config file.
        config_hash: Precomputed SHA256 hash of `config`.

    Returns:
        Absolute path to the written config JSON file.
    """
    os.makedirs(output_dir, exist_ok=True)
    export_path = os.path.join(output_dir, f"config_{config_hash[:12]}.json")
    payload = dict(config)
    payload["_exported_at"] = datetime.utcnow().isoformat() + "Z"
    with open(export_path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2, default=str)
    LOGGER.info(f"Config exported to {export_path} (hash={config_hash[:12]}...).")
    return export_path


CONFIG_HASH: str = compute_config_hash(CONFIG)
CONFIG["config_hash"] = CONFIG_HASH
CONFIG_EXPORT_PATH: str = export_config(CONFIG, PATHS["logs"], CONFIG_HASH)
LOGGER.info(f"CONFIG SHA256: {CONFIG_HASH}")


# ----------------------------------------------------------------------------
# 10. CHECKPOINT VERSIONING, INTEGRITY VERIFICATION, ROTATING BACKUPS
# ----------------------------------------------------------------------------

def verify_checkpoint_integrity(checkpoint_path: str) -> Tuple[bool, Optional[Dict[str, Any]]]:
    """
    Validate that a checkpoint file exists, loads, and contains required keys.

    Used by the training cell (Cell 5) to safely decide whether to resume
    or to fall back to a fresh run without crashing on a corrupted file.

    Required keys: {'epoch', 'model_state_dict', 'optimizer_state_dict',
    'scheduler_state_dict', 'scaler_state_dict', 'best_accuracy', 'best_f1',
    'history', 'rng_state', 'config_hash'}

    Args:
        checkpoint_path: Absolute path to the checkpoint `.pt` file.

    Returns:
        Tuple of (is_valid, checkpoint_dict_or_None). If the file is missing,
        unreadable, or missing required keys, returns (False, None) and logs
        the reason instead of raising — callers must handle both branches.
    """
    required_keys = {
        "epoch", "model_state_dict", "optimizer_state_dict",
        "scheduler_state_dict", "scaler_state_dict",
        "best_accuracy", "best_f1", "history", "rng_state", "config_hash",
    }

    if not os.path.isfile(checkpoint_path):
        LOGGER.info(f"No checkpoint found at {checkpoint_path}. Fresh start assumed.")
        return False, None

    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
    except (RuntimeError, EOFError, pickle.UnpicklingError, OSError) as exc:
        LOGGER.error(f"Checkpoint at {checkpoint_path} is corrupted/unreadable: {exc}")
        return False, None

    if not isinstance(checkpoint, dict):
        LOGGER.error(f"Checkpoint at {checkpoint_path} did not deserialize to a dict.")
        return False, None

    missing = required_keys - set(checkpoint.keys())
    if missing:
        LOGGER.error(f"Checkpoint at {checkpoint_path} is missing required keys: {missing}")
        return False, None

    if checkpoint.get("config_hash") != CONFIG_HASH:
        LOGGER.warning(
            f"Checkpoint config_hash ({checkpoint.get('config_hash', 'NONE')[:12]}...) "
            f"differs from current CONFIG hash ({CONFIG_HASH[:12]}...). "
            f"Hyperparameters may have changed since this checkpoint was saved."
        )

    LOGGER.info(f"Checkpoint verified OK at {checkpoint_path} "
                f"(epoch={checkpoint['epoch']}, best_f1={checkpoint['best_f1']:.4f}).")
    return True, checkpoint


def get_latest_checkpoint_path(checkpoint_dir: str) -> Optional[str]:
    """
    Resolve the checkpoint to resume from, preferring the explicit
    `checkpoint_latest.pt` pointer, then falling back to the most recently
    modified versioned checkpoint file in the directory.

    This is the entry point Cell 5 should call to implement mandatory
    automatic resume (never restart from epoch 0 unless explicitly requested).

    Args:
        checkpoint_dir: Directory containing checkpoint files.

    Returns:
        Absolute path to the resolved checkpoint, or None if no valid
        checkpoint could be found.
    """
    latest_pointer = os.path.join(checkpoint_dir, CONFIG["checkpoint"]["latest_filename"])
    if os.path.isfile(latest_pointer):
        is_valid, _ = verify_checkpoint_integrity(latest_pointer)
        if is_valid:
            return latest_pointer
        LOGGER.warning(f"Latest checkpoint pointer at {latest_pointer} is invalid; "
                        f"searching for versioned fallbacks.")

    try:
        candidates = [
            os.path.join(checkpoint_dir, f) for f in os.listdir(checkpoint_dir)
            if f.startswith("checkpoint_epoch") and f.endswith(".pt")
        ]
    except FileNotFoundError:
        return None

    for candidate in sorted(candidates, key=os.path.getmtime, reverse=True):
        is_valid, _ = verify_checkpoint_integrity(candidate)
        if is_valid:
            LOGGER.info(f"Resuming from versioned fallback checkpoint: {candidate}")
            return candidate

    LOGGER.info("No valid checkpoint (latest or versioned) found. Fresh start assumed.")
    return None


def make_versioned_checkpoint_name(epoch: int) -> str:
    """
    Build a collision-free, timestamped, versioned checkpoint filename.

    Args:
        epoch: The epoch number this checkpoint corresponds to.

    Returns:
        A filename string following CONFIG's versioned naming template.
    """
    timestamp = datetime.utcnow().strftime("%Y%m%d%H%M%S")
    return CONFIG["checkpoint"]["versioned_filename_template"].format(
        epoch=epoch, timestamp=timestamp
    )


def backup_checkpoint(checkpoint_path: str, backup_dir: str,
                       keep_last_n: int = 5) -> Optional[str]:
    """
    Copy the given checkpoint into a rotating backup directory, pruning
    the oldest backups beyond `keep_last_n` to bound disk usage.

    Args:
        checkpoint_path: Path to the checkpoint file to back up.
        backup_dir: Directory in which backups are stored.
        keep_last_n: Maximum number of backup files to retain.

    Returns:
        Absolute path to the newly created backup file, or None if the
        source checkpoint did not exist or the copy failed.
    """
    if not os.path.isfile(checkpoint_path):
        LOGGER.warning(f"Cannot back up nonexistent checkpoint: {checkpoint_path}")
        return None

    os.makedirs(backup_dir, exist_ok=True)
    timestamp = datetime.utcnow().strftime("%Y%m%d%H%M%S")
    backup_name = f"backup_{timestamp}_{os.path.basename(checkpoint_path)}"
    backup_path = os.path.join(backup_dir, backup_name)

    try:
        shutil.copy2(checkpoint_path, backup_path)
    except OSError as exc:
        LOGGER.error(f"Checkpoint backup failed for {checkpoint_path}: {exc}")
        return None

    try:
        backups = sorted(
            (os.path.join(backup_dir, f) for f in os.listdir(backup_dir)
             if f.startswith("backup_")),
            key=os.path.getmtime,
        )
        while len(backups) > keep_last_n:
            oldest = backups.pop(0)
            os.remove(oldest)
            LOGGER.info(f"Pruned old checkpoint backup: {oldest}")
    except OSError as exc:
        LOGGER.warning(f"Backup pruning encountered an issue (non-fatal): {exc}")

    LOGGER.info(f"Checkpoint backed up to {backup_path}")
    return backup_path


# ----------------------------------------------------------------------------
# 11. METRICS CSV LOGGER + HISTORY.JSON (APPEND MODE)
# ----------------------------------------------------------------------------

class MetricsCSVLogger:
    """
    Append-safe CSV metrics logger. Writes a header exactly once and appends
    one row per call thereafter — safe to reinstantiate across notebook
    reruns or resumed sessions without duplicating or clobbering history.
    """

    def __init__(self, csv_path: str, fieldnames: List[str]) -> None:
        """
        Args:
            csv_path: Absolute path to the metrics CSV file.
            fieldnames: Ordered list of column names for the CSV.
        """
        self.csv_path = csv_path
        self.fieldnames = fieldnames
        os.makedirs(os.path.dirname(csv_path), exist_ok=True)
        self._ensure_header()

    def _ensure_header(self) -> None:
        """Write the CSV header row only if the file does not already exist."""
        if not os.path.isfile(self.csv_path) or os.path.getsize(self.csv_path) == 0:
            with open(self.csv_path, "w", encoding="utf-8") as fh:
                fh.write(",".join(self.fieldnames) + "\n")

    def log(self, row: Dict[str, Any]) -> None:
        """
        Append a single row of metrics to the CSV file.

        Args:
            row: Mapping of fieldname -> value. Missing fields are written
                as empty strings; extra keys not in `fieldnames` are ignored.
        """
        try:
            values = [str(row.get(field, "")) for field in self.fieldnames]
            with open(self.csv_path, "a", encoding="utf-8") as fh:
                fh.write(",".join(values) + "\n")
        except OSError as exc:
            LOGGER.error(f"Failed to append metrics row to {self.csv_path}: {exc}")


def append_history(history_path: str, epoch_record: Dict[str, Any]) -> None:
    """
    Append a single epoch's record to history.json, preserving all prior
    entries (append mode) rather than overwriting the file each epoch.

    Args:
        history_path: Absolute path to history.json.
        epoch_record: Dictionary describing this epoch's results
            (e.g. {"epoch": 3, "train_loss": 0.21, "val_f1": 0.94, ...}).
    """
    history: List[Dict[str, Any]] = []
    if os.path.isfile(history_path):
        try:
            with open(history_path, "r", encoding="utf-8") as fh:
                history = json.load(fh)
            if not isinstance(history, list):
                LOGGER.warning(f"history.json at {history_path} was not a list; "
                                f"reinitializing as empty history.")
                history = []
        except (json.JSONDecodeError, OSError) as exc:
            LOGGER.warning(f"Could not read existing history.json ({exc}); "
                            f"starting a new history list.")
            history = []

    history.append(epoch_record)

    try:
        with open(history_path, "w", encoding="utf-8") as fh:
            json.dump(history, fh, indent=2, default=str)
    except OSError as exc:
        LOGGER.error(f"Failed to write updated history.json: {exc}")


# ----------------------------------------------------------------------------
# 12. LAST_STATUS.JSON — EXPERIMENT STATE TRACKER
# ----------------------------------------------------------------------------

def write_status(stage: str, status_path: str, extra: Optional[Dict[str, Any]] = None,
                  is_error: bool = False) -> None:
    """
    Persist the current experiment stage/status to last_status.json so an
    interrupted Colab session (timeout, disconnect, crash) can be diagnosed
    and safely resumed without ambiguity about what last happened.

    Args:
        stage: Short machine-readable stage label (e.g. "env_ready",
            "training_epoch_12", "error").
        status_path: Absolute path to last_status.json.
        extra: Additional fields to merge into the status payload.
        is_error: If True, marks this status entry as an error state.
    """
    payload: Dict[str, Any] = {
        "stage": stage,
        "is_error": is_error,
        "experiment_uuid": EXPERIMENT_UUID,
        "experiment_name": CONFIG["experiment_name"],
        "config_hash": CONFIG_HASH,
        "device": DEVICE_INFO.device_name,
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
    }
    if extra:
        payload.update(extra)

    try:
        os.makedirs(os.path.dirname(status_path), exist_ok=True)
        with open(status_path, "w", encoding="utf-8") as fh:
            json.dump(payload, fh, indent=2, default=str)
    except OSError as exc:
        LOGGER.error(f"Failed to write last_status.json at {status_path}: {exc}")


STATUS_PATH: str = os.path.join(PATHS["logs"], CONFIG["logging"]["status_filename"])


# ----------------------------------------------------------------------------
# 13. ENVIRONMENT HEALTH CHECKS (internet, filesystem, package versions)
# ----------------------------------------------------------------------------

def check_internet_connectivity(host: str = "8.8.8.8", port: int = 53,
                                 timeout: float = 3.0) -> bool:
    """
    Verify internet connectivity via a low-level socket probe to a DNS server.

    Args:
        host: IP address to probe (default: Google public DNS).
        port: Port to attempt connection on.
        timeout: Socket timeout in seconds.

    Returns:
        True if a TCP connection succeeds, False otherwise.
    """
    try:
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        return True
    except OSError:
        return False


def verify_directory_integrity(paths: Dict[str, str]) -> List[str]:
    """
    Confirm every expected path in `paths` exists and is writable.

    Args:
        paths: Mapping of logical names to directory paths.

    Returns:
        A list of problem descriptions (empty list means fully healthy).
    """
    problems: List[str] = []
    for key, path in paths.items():
        if not os.path.isdir(path):
            problems.append(f"Missing directory: '{key}' -> {path}")
            continue
        if not os.access(path, os.W_OK):
            problems.append(f"Directory not writable: '{key}' -> {path}")
    return problems


def get_package_versions(packages: List[str]) -> Dict[str, str]:
    """
    Collect installed version strings for a list of import names, for
    provenance logging alongside the config export.

    Args:
        packages: List of importable module names to inspect.

    Returns:
        Mapping of {package_name: version_string_or_'unknown'}.
    """
    versions: Dict[str, str] = {}
    for pkg in packages:
        try:
            module = importlib.import_module(pkg)
            versions[pkg] = getattr(module, "__version__", "unknown")
        except ImportError:
            versions[pkg] = "not_installed"
    return versions


def run_environment_diagnostics() -> Dict[str, Any]:
    """
    Execute the full suite of environment health checks and log a summary.

    Returns:
        A dictionary summarizing GPU, internet, filesystem, disk, RAM, and
        package-version health, useful for embedding into experiment metadata.
    """
    LOGGER.info("Running environment diagnostics...")

    internet_ok = check_internet_connectivity()
    LOGGER.info(f"Internet connectivity: {'OK' if internet_ok else 'UNAVAILABLE'}")
    if not internet_ok:
        LOGGER.warning("No internet detected. Dataset download (Cell 2) will fail "
                        "unless data already exists in Drive cache.")

    dir_problems = verify_directory_integrity(PATHS)
    if dir_problems:
        for problem in dir_problems:
            LOGGER.error(problem)
    else:
        LOGGER.info("All project directories present and writable.")

    package_versions = get_package_versions(
        ["torch", "numpy", "pandas", "sklearn", "psutil", "PIL"]
    )
    LOGGER.info(f"Package versions: {package_versions}")

    diagnostics = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "notebook_run_id": NOTEBOOK_RUN_ID,
        "experiment_uuid": EXPERIMENT_UUID,
        "config_hash": CONFIG_HASH,
        "colab": IS_COLAB,
        "drive_mounted": DRIVE_MOUNTED,
        "device": DEVICE_INFO.device_name,
        "cuda_available": DEVICE_INFO.is_cuda,
        "amp_enabled": CONFIG["train"]["amp_enabled"],
        "compute_capability": DEVICE_INFO.compute_capability,
        "vram_gb": DEVICE_INFO.total_memory_gb,
        "internet_available": internet_ok,
        "directory_problems": dir_problems,
        "disk_free_gb": DISK_FREE_GB,
        "disk_ok": DISK_OK,
        "resource_snapshot": RESOURCE_SNAPSHOT,
        "package_versions": package_versions,
        "package_install_results": _PACKAGE_INSTALL_RESULTS,
        "pytorch_version": torch.__version__,
        "python_version": platform.python_version(),
    }

    diagnostics_path = os.path.join(PATHS["logs"], "environment_diagnostics.json")
    try:
        with open(diagnostics_path, "w", encoding="utf-8") as fh:
            json.dump(diagnostics, fh, indent=2, default=str)
        LOGGER.info(f"Environment diagnostics written to {diagnostics_path}")
    except OSError as exc:
        LOGGER.warning(f"Could not persist diagnostics file: {exc}")

    return diagnostics


try:
    ENV_DIAGNOSTICS: Dict[str, Any] = run_environment_diagnostics()
    write_status("env_ready", STATUS_PATH, extra={"diagnostics_path":
                 os.path.join(PATHS["logs"], "environment_diagnostics.json")})
except Exception as exc:  # noqa: broad-except — must not silently crash Cell 1
    LOGGER.error(f"Environment diagnostics failed: {exc}\n{traceback.format_exc()}")
    write_status("env_error", STATUS_PATH, extra={"error": str(exc)}, is_error=True)
    raise


# ----------------------------------------------------------------------------
# 14. FINAL SUMMARY
# ----------------------------------------------------------------------------

def print_environment_summary(config: Dict[str, Any], device_info: DeviceInfo,
                               diagnostics: Dict[str, Any]) -> None:
    """
    Print a clean, human-readable summary banner of the initialized environment.

    Args:
        config: The global CONFIG dictionary.
        device_info: Populated `DeviceInfo` instance.
        diagnostics: Output of `run_environment_diagnostics()`.
    """
    banner_width = 78
    LOGGER.info("=" * banner_width)
    LOGGER.info(f"{config['project_name']} — ENVIRONMENT READY".center(banner_width))
    LOGGER.info("=" * banner_width)
    LOGGER.info(f"Experiment:        {config['experiment_name']}")
    LOGGER.info(f"Experiment UUID:   {config['experiment_uuid']}")
    LOGGER.info(f"Config SHA256:     {config['config_hash']}")
    LOGGER.info(f"Device:            {device_info.device_name} "
                f"({'CUDA' if device_info.is_cuda else 'CPU'})")
    LOGGER.info(f"AMP enabled:       {config['train']['amp_enabled']}")
    LOGGER.info(f"cuDNN benchmark:   {cudnn.benchmark}")
    LOGGER.info(f"Seed:              {config['seed']}")
    LOGGER.info(f"Drive mounted:     {DRIVE_MOUNTED}")
    LOGGER.info(f"Project root:      {config['drive_root']}")
    LOGGER.info(f"Disk free:         {diagnostics['disk_free_gb']} GB")
    LOGGER.info(f"RAM available:     {diagnostics['resource_snapshot']['ram_available_gb']} GB")
    LOGGER.info(f"Internet:          {diagnostics['internet_available']}")
    LOGGER.info(f"Resume training:   {config['train']['resume']}")
    LOGGER.info("=" * banner_width)


print_environment_summary(CONFIG, DEVICE_INFO, ENV_DIAGNOSTICS)

LOGGER.info(f"Cell 1 completed in {time.time() - NOTEBOOK_START_TIME:.2f} seconds.")

[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] MAL-PhishNet — Cell 1: Environment Setup starting.
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] Python version: 3.12.13 | PyTorch version: 2.11.0+cu128 | Colab: True
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] Package check — psutil: already_installed
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] Package check — gdown: already_installed
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] Package check — numpy: already_installed
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] Package check — pandas: already_installed
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] Package check — sklearn: already_installed
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] Package check — PIL: already_installed


/tmp/ipykernel_863/3319101159.py:121: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  NOTEBOOK_RUN_ID: str = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] GPU detected: Tesla T4 | Compute Capability: 7.5 | VRAM: 14.56 GB | AMP supported: True
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] Automatic Mixed Precision (AMP) ENABLED for this session.
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] cuDNN benchmark mode: True
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] Global seed set to 42 (deterministic=False).
[2026-08-15 15:01:39] [INFO    ] [MAL-PhishNet] Google Drive already mounted.
[2026-08-15 15:01:40] [INFO    ] [MAL-PhishNet] Experiment UUID: 97b7451f-0315-425a-9d5f-479062e7a2cd
[2026-08-15 15:01:41] [INFO    ] [MAL-PhishNet] Directory structure verified/created under: /content/drive/MyDrive/MAL-PhishNet
[2026-08-15 15:01:42] [INFO    ] [MAL-PhishNet] File-based logging attached (training.log + error.log). All subsequent messages persist to disk.
[2026-08-15 15:01:42] [INFO    ] [MAL-PhishNet] Disk space OK at /content/drive/MyDrive/MAL-PhishNet: 4.52 GB free.
[2026-08-15 15:01:42]

/tmp/ipykernel_863/3319101159.py:757: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  payload["_exported_at"] = datetime.utcnow().isoformat() + "Z"


[2026-08-15 15:01:42] [INFO    ] [MAL-PhishNet] Config exported to /content/drive/MyDrive/MAL-PhishNet/logs/config_d963c120c828.json (hash=d963c120c828...).
[2026-08-15 15:01:42] [INFO    ] [MAL-PhishNet] CONFIG SHA256: d963c120c8283aaf36eba114a392d9e66110f89966181520d8c14bbaf3f1f11c
[2026-08-15 15:01:42] [INFO    ] [MAL-PhishNet] Running environment diagnostics...
[2026-08-15 15:01:42] [INFO    ] [MAL-PhishNet] Internet connectivity: OK
[2026-08-15 15:01:42] [INFO    ] [MAL-PhishNet] All project directories present and writable.
[2026-08-15 15:01:42] [INFO    ] [MAL-PhishNet] Package versions: {'torch': '2.11.0+cu128', 'numpy': '2.0.2', 'pandas': '2.2.2', 'sklearn': '1.6.1', 'psutil': '5.9.5', 'PIL': '11.3.0'}
[2026-08-15 15:01:42] [INFO    ] [MAL-PhishNet] Environment diagnostics written to /content/drive/MyDrive/MAL-PhishNet/logs/environment_diagnostics.json


/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


[2026-08-15 15:01:43] [INFO    ] [MAL-PhishNet] ==============================================================================
[2026-08-15 15:01:43] [INFO    ] [MAL-PhishNet]                        MAL-PhishNet — ENVIRONMENT READY                       
[2026-08-15 15:01:43] [INFO    ] [MAL-PhishNet] ==============================================================================
[2026-08-15 15:01:43] [INFO    ] [MAL-PhishNet] Experiment:        malphishnet_v1
[2026-08-15 15:01:43] [INFO    ] [MAL-PhishNet] Experiment UUID:   97b7451f-0315-425a-9d5f-479062e7a2cd
[2026-08-15 15:01:43] [INFO    ] [MAL-PhishNet] Config SHA256:     d963c120c8283aaf36eba114a392d9e66110f89966181520d8c14bbaf3f1f11c
[2026-08-15 15:01:43] [INFO    ] [MAL-PhishNet] Device:            Tesla T4 (CUDA)
[2026-08-15 15:01:43] [INFO    ] [MAL-PhishNet] AMP enabled:       True
[2026-08-15 15:01:43] [INFO    ] [MAL-PhishNet] cuDNN benchmark:   True
[2026-08-15 15:01:43] [INFO    ] [MAL-PhishNet] Seed:              42
[202

In [ ]:
# ============================================================================
# MAL-PhishNet — Cell 2: Multi-Model Dataset Acquisition
# (Text / Behavioral / URL / Web Content — four independent datasets)
# ============================================================================
# Run Cell 1 before this cell.
#
# ARCHITECTURE CHANGE: this project now trains FOUR independent models
# (Text, URL, Behavioral, Web Content), later fused by a Cloud Fusion
# Layer (Cell 6). Each model requires its OWN independent dataset:
#
#   - Text Model        : Nazario (phishing) + SpamAssassin (legitimate)
#   - Behavioral Model   : SAME email corpus as Text (lexical features are
#                          folded into this model per supervisor decision;
#                          no separate acquisition needed)
#   - URL Model          : PhishTank (phishing URLs) + Cisco Umbrella
#                          Top 1M (benign URLs)
#   - Web Content Model  : Mendeley CompPhish v2 (Mapping_File.xlsx +
#                          All_HTML.zip) — paired URL+HTML+label corpus
#                          covering both phishing and legitimate pages
#
# Google Drive caching, resumable downloads, atomic saving, checksums, and
# manifests are preserved exactly as in prior cells — only the acquisition
# SCOPE has expanded from one dataset to four.
#
# Main output:
#   DATASET_INFO
# ============================================================================


# ----------------------------------------------------------------------------
# 0. IMPORTS AND PACKAGE INSTALLATION
# ----------------------------------------------------------------------------

import os
import re
import sys
import gc
import json
import time
import uuid
import email
import shutil
import asyncio
import tarfile
import zipfile
import hashlib
import importlib
import subprocess
import traceback

from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple
from urllib.parse import urlparse


def ensure_package(import_name: str, pip_name: Optional[str] = None) -> None:
    """Install a package only when it is missing."""
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name or import_name]
        )
        importlib.invalidate_caches()


for package_import, package_name in [
    ("requests", "requests"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("openpyxl", "openpyxl"),
    ("tqdm", "tqdm"),
    ("playwright", "playwright"),
]:
    ensure_package(package_import, package_name)


import numpy as np
import pandas as pd
import requests

from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

PLAYWRIGHT_IMPORT_ERROR: Optional[str] = None
try:
    from playwright.async_api import (
        async_playwright,
        TimeoutError as PlaywrightTimeoutError,
    )
except ImportError as exc:
    async_playwright = None  # type: ignore[assignment]
    PlaywrightTimeoutError = TimeoutError  # type: ignore[assignment]
    PLAYWRIGHT_IMPORT_ERROR = str(exc)


# ----------------------------------------------------------------------------
# 1. VERIFY CELL 1
# ----------------------------------------------------------------------------

REQUIRED_CELL1_GLOBALS = [
    "CONFIG", "CONFIG_HASH", "EXPERIMENT_UUID", "PATHS", "LOGGER",
    "STATUS_PATH", "write_status",
]
missing_cell1_globals = [name for name in REQUIRED_CELL1_GLOBALS if name not in globals()]
if missing_cell1_globals:
    raise RuntimeError(f"Cell 2 requires Cell 1 first. Missing objects: {missing_cell1_globals}")


def utc_now_iso() -> str:
    """Return timezone-aware UTC timestamp."""
    return datetime.now(timezone.utc).isoformat()


def write_status_safe(
    status_name: str, extra: Optional[Mapping[str, Any]] = None, is_error: bool = False,
) -> None:
    """Support both Cell-1 write_status function signatures."""
    payload = dict(extra or {})
    try:
        write_status(status_name, STATUS_PATH, extra=payload, is_error=is_error)
    except TypeError:
        write_status(status_name, STATUS_PATH, extra=payload)


LOGGER.info("=" * 78)
LOGGER.info("MAL-PhishNet — Cell 2: Multi-Model Dataset Acquisition".center(78))
LOGGER.info("=" * 78)
LOGGER.info(
    "ARCHITECTURE CHANGE: acquiring FOUR independent datasets — Text/Behavioral "
    "(shared email corpus), URL (PhishTank + Cisco Umbrella), Web Content "
    "(Mendeley CompPhish v2)."
)

write_status_safe("dataset_management_start", {"cell": 2, "scope": "multi_model"})


# ----------------------------------------------------------------------------
# 2. CELL 2 CONFIGURATION
# ----------------------------------------------------------------------------

DEFAULT_CELL2_CONFIG: Dict[str, Any] = {
    "version": "2.11.0-multi-model",

    "request_timeout_seconds": 180,
    "download_retries": 5,
    "download_chunk_bytes": 8 * 1024 * 1024,

    # --- Text / Behavioral (shared email corpus) ---
    "zenodo_record_id": "8339691",
    "nazario_filename": "Nazario.csv",
    "spamassassin_archives": {
        "easy_ham": "https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2",
        "hard_ham": "https://spamassassin.apache.org/old/publiccorpus/20030228_hard_ham.tar.bz2",
    },

    # --- URL model ---
    "phishtank_url": "https://data.phishtank.com/data/online-valid.csv.bz2",
    "cisco_umbrella_top1m_url": "http://s3-us-west-1.amazonaws.com/umbrella-static/top-1m.csv.zip",
    "url_model_max_benign_urls": 20_000,
    "url_model_max_phishing_urls": 20_000,

    # --- Web Content model ---
    "comp_phish_id": "fmbs4kp9wz",
    "comp_phish_version": 2,
    "comp_phish_page": "https://data.mendeley.com/datasets/fmbs4kp9wz/2",
    "comp_phish_required_files": ["Mapping_File.xlsx", "All_HTML.zip"],
    "comp_phish_min_mapping_bytes": 100_000,
    "comp_phish_min_html_zip_bytes": 500_000_000,
    "comp_phish_min_mapping_rows": 10_000,
    "comp_phish_min_html_files": 10_000,
    "playwright_page_timeout_ms": 180_000,
    "playwright_download_timeout_ms": 300_000,
    "playwright_cloudflare_wait_ms": 30_000,
    "playwright_context_download_timeout_ms": 300_000,
    "delete_local_download_all_after_extract": True,
}

CELL2_CONFIG: Dict[str, Any] = {
    **DEFAULT_CELL2_CONFIG,
    **dict(CONFIG.get("dataset_management", {})),
}


# ----------------------------------------------------------------------------
# 3. DATASET PATHS
# ----------------------------------------------------------------------------

ROOT = Path(PATHS.get("root", "/content/drive/MyDrive/MAL-PhishNet"))
DATA_ROOT = Path(PATHS.get("data", ROOT / "data"))
RAW_ROOT = Path(PATHS.get("raw_data", DATA_ROOT / "raw"))
EXTRACTED_ROOT = Path(PATHS.get("extracted_data", DATA_ROOT / "extracted"))
PREPARED_ROOT = Path(PATHS.get("prepared_data", DATA_ROOT / "prepared"))
MANIFEST_ROOT = Path(PATHS.get("manifests", ROOT / "manifests"))

EMAIL_RAW_ROOT = RAW_ROOT / "email"
URL_RAW_ROOT = RAW_ROOT / "url"
WEB_RAW_ROOT = RAW_ROOT / "web_content"
WEB_EXTRACTED_ROOT = EXTRACTED_ROOT / "web_content"
LOCAL_BROWSER_DOWNLOAD_ROOT = Path("/content/malphishnet_mendeley_downloads")

for directory in [
    ROOT, DATA_ROOT, RAW_ROOT, EXTRACTED_ROOT, PREPARED_ROOT, MANIFEST_ROOT,
    EMAIL_RAW_ROOT, URL_RAW_ROOT, WEB_RAW_ROOT, WEB_EXTRACTED_ROOT,
    LOCAL_BROWSER_DOWNLOAD_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------------
# 4. GENERAL FILE HELPERS
# ----------------------------------------------------------------------------

def atomic_json_save(payload: Any, destination: Path) -> None:
    """Save JSON atomically."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(destination.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        file_handle.flush()
        os.fsync(file_handle.fileno())
    os.replace(temporary_path, destination)


def atomic_csv_save(dataframe: pd.DataFrame, destination: Path) -> None:
    """Save CSV atomically."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(destination.suffix + ".tmp")
    dataframe.to_csv(temporary_path, index=False, encoding="utf-8")
    os.replace(temporary_path, destination)


def hash_file(path: Path, algorithm: str = "sha256") -> str:
    """Calculate a file hash without loading it into memory."""
    digest = hashlib.new(algorithm)
    with path.open("rb") as file_handle:
        while True:
            block = file_handle.read(8 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def safe_archive_destination(root: Path, member_name: str) -> Path:
    """Prevent archive path traversal."""
    normalized_name = member_name.replace("\\", "/").lstrip("/")
    destination = (root / normalized_name).resolve()
    root_resolved = root.resolve()
    if destination != root_resolved and root_resolved not in destination.parents:
        raise RuntimeError(f"Unsafe archive member: {member_name}")
    return destination


# ----------------------------------------------------------------------------
# 5. HTTP SESSION AND RESUMABLE DOWNLOAD
# ----------------------------------------------------------------------------

def create_http_session() -> requests.Session:
    """Create a retry-enabled HTTP session."""
    retries = int(CELL2_CONFIG["download_retries"])
    retry_strategy = Retry(
        total=retries, connect=retries, read=retries, status=retries, backoff_factor=1.5,
        status_forcelist=(408, 425, 429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET", "HEAD"]), raise_on_status=False,
    )
    session = requests.Session()
    adapter = HTTPAdapter(max_retries=retry_strategy, pool_connections=10, pool_maxsize=10)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
        ),
        "Accept": "*/*", "Accept-Language": "en-US,en;q=0.9",
    })
    return session


HTTP_SESSION = create_http_session()


def validate_downloaded_payload(path: Path, minimum_bytes: int = 1) -> None:
    """Reject missing, tiny or HTML error files."""
    if not path.is_file():
        raise FileNotFoundError(str(path))
    file_size = path.stat().st_size
    if file_size < minimum_bytes:
        raise RuntimeError(f"File is too small: {path} ({file_size} bytes)")
    with path.open("rb") as file_handle:
        beginning = file_handle.read(1024).lstrip().lower()
    if beginning.startswith(b"<!doctype html") or beginning.startswith(b"<html"):
        raise RuntimeError(f"Server returned an HTML error page instead of a dataset file: {path}")


def download_resumable(
    url: str, destination: Path, expected_size: Optional[int] = None,
    expected_sha256: Optional[str] = None, expected_md5: Optional[str] = None,
    minimum_bytes: int = 1, headers: Optional[Mapping[str, str]] = None,
    description: Optional[str] = None,
) -> Path:
    """Download a file with .part resume support."""
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.is_file():
        existing_is_valid = destination.stat().st_size >= minimum_bytes
        if expected_size is not None:
            existing_is_valid = existing_is_valid and destination.stat().st_size == int(expected_size)
        if existing_is_valid:
            if expected_sha256 and hash_file(destination, "sha256").lower() != expected_sha256.lower():
                destination.unlink(missing_ok=True)
            elif expected_md5 and hash_file(destination, "md5").lower() != expected_md5.lower():
                destination.unlink(missing_ok=True)
            else:
                LOGGER.info("Using cached file: %s", destination)
                return destination

    partial_path = destination.with_suffix(destination.suffix + ".part")
    existing_size = partial_path.stat().st_size if partial_path.exists() else 0
    request_headers = dict(headers or {})
    if existing_size > 0:
        request_headers["Range"] = f"bytes={existing_size}-"

    response = HTTP_SESSION.get(
        url, stream=True, allow_redirects=True,
        timeout=(30, int(CELL2_CONFIG["request_timeout_seconds"])), headers=request_headers,
    )

    if response.status_code not in {200, 206}:
        response_preview = ""
        try:
            response_preview = response.text[:400]
        except Exception:
            pass
        raise RuntimeError(
            f"HTTP {response.status_code} while downloading {url}. Response={response_preview!r}"
        )

    if existing_size > 0 and response.status_code == 200:
        existing_size = 0
        partial_path.unlink(missing_ok=True)

    content_length = response.headers.get("content-length")
    remaining_size = int(content_length) if content_length and content_length.isdigit() else None
    total_size = existing_size + remaining_size if remaining_size is not None else expected_size
    file_mode = "ab" if existing_size > 0 else "wb"

    with partial_path.open(file_mode) as file_handle, tqdm(
        total=total_size, initial=existing_size, unit="B", unit_scale=True,
        unit_divisor=1024, desc=description or destination.name,
    ) as progress_bar:
        for block in response.iter_content(chunk_size=int(CELL2_CONFIG["download_chunk_bytes"])):
            if not block:
                continue
            file_handle.write(block)
            progress_bar.update(len(block))
        file_handle.flush()
        os.fsync(file_handle.fileno())

    os.replace(partial_path, destination)
    validate_downloaded_payload(destination, minimum_bytes)

    if expected_size is not None and destination.stat().st_size != int(expected_size):
        raise RuntimeError(
            f"Size mismatch for {destination.name}. Expected={expected_size}, "
            f"received={destination.stat().st_size}"
        )
    if expected_sha256 and hash_file(destination, "sha256").lower() != expected_sha256.lower():
        raise RuntimeError(f"SHA-256 mismatch: {destination}")
    if expected_md5 and hash_file(destination, "md5").lower() != expected_md5.lower():
        raise RuntimeError(f"MD5 mismatch: {destination}")

    return destination


# ----------------------------------------------------------------------------
# 6. ROBUST CSV READER AND COLUMN HELPERS
# ----------------------------------------------------------------------------

def read_csv_robust(path: Path, compression: Optional[str] = "infer") -> pd.DataFrame:
    """Read CSV using encoding and parser fallbacks."""
    parsing_errors: List[str] = []
    for encoding_name in ["utf-8-sig", "utf-8", "latin-1", "cp1252"]:
        try:
            return pd.read_csv(
                path, compression=compression, encoding=encoding_name,
                encoding_errors="replace", low_memory=False, on_bad_lines="skip",
            )
        except Exception as exc:
            parsing_errors.append(f"{encoding_name}/C: {exc}")
        try:
            return pd.read_csv(
                path, compression=compression, encoding=encoding_name,
                encoding_errors="replace", engine="python", on_bad_lines="skip",
            )
        except Exception as exc:
            parsing_errors.append(f"{encoding_name}/Python: {exc}")
    raise RuntimeError(f"Could not parse CSV {path}. Errors: {parsing_errors[-4:]}")


def normalized_column_name(value: Any) -> str:
    """Normalize dataframe column names."""
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")


def choose_column(
    dataframe: pd.DataFrame, exact_candidates: Sequence[str], contains_candidates: Sequence[str],
) -> Optional[str]:
    """Find a dataframe column safely."""
    normalized_columns = {normalized_column_name(c): str(c) for c in dataframe.columns}
    for candidate in exact_candidates:
        normalized_candidate = normalized_column_name(candidate)
        if normalized_candidate in normalized_columns:
            return normalized_columns[normalized_candidate]
    for normalized_name, original_name in normalized_columns.items():
        if any(token in normalized_name for token in contains_candidates):
            return original_name
    return None


def request_json(url: str, headers: Optional[Mapping[str, str]] = None) -> Any:
    """Request JSON with readable errors."""
    response = HTTP_SESSION.get(
        url, headers=dict(headers or {}),
        timeout=(30, int(CELL2_CONFIG["request_timeout_seconds"])), allow_redirects=True,
    )
    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code} from {url}: {response.text[:300]!r}")
    return response.json()


# ============================================================================
# TEXT / BEHAVIORAL MODEL — SHARED EMAIL CORPUS ACQUISITION
# (unchanged from the prior email-only Cell 2)
# ============================================================================

def zenodo_file_metadata(record_id: str, filename: str) -> Dict[str, Any]:
    """Resolve a Zenodo direct file URL."""
    record_payload = request_json(f"https://zenodo.org/api/records/{record_id}")
    for file_record in record_payload.get("files", []):
        current_filename = str(file_record.get("key") or file_record.get("filename") or "")
        if current_filename.lower() != filename.lower():
            continue
        links = file_record.get("links") or {}
        checksum = str(file_record.get("checksum") or "")
        checksum_algorithm, _, checksum_digest = checksum.partition(":")
        return {
            "filename": current_filename,
            "url": links.get("content") or links.get("self"),
            "size": int(file_record["size"]) if file_record.get("size") else None,
            "md5": checksum_digest if checksum_algorithm.lower() == "md5" else None,
        }
    raise FileNotFoundError(f"{filename} was not found in Zenodo record {record_id}.")


def detect_email_columns(dataframe: pd.DataFrame) -> Dict[str, Optional[str]]:
    """Detect common email CSV columns."""
    return {
        "subject": choose_column(dataframe, ["subject", "email_subject"], ["subject"]),
        "sender": choose_column(dataframe, ["sender", "from", "from_address", "sender_email"], ["sender", "from"]),
        "receiver": choose_column(dataframe, ["receiver", "recipient", "to", "to_address"], ["receiver", "recipient"]),
        "date": choose_column(dataframe, ["date", "sent_date", "timestamp"], ["date", "time"]),
        "body": choose_column(
            dataframe, ["body", "message", "text", "email_body", "content", "email_text"],
            ["body", "message", "content", "text"],
        ),
    }


def canonicalize_nazario(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Create canonical phishing email rows."""
    detected_columns = detect_email_columns(dataframe)
    body_column = detected_columns["body"]
    if body_column is None:
        object_columns = [c for c in dataframe.columns if dataframe[c].dtype == object]
        if not object_columns:
            raise RuntimeError("Nazario dataset has no usable email text column.")
        body_column = max(
            object_columns,
            key=lambda c: dataframe[c].fillna("").astype(str).str.len().mean(),
        )

    def get_string_series(column: Optional[str]) -> pd.Series:
        if column and column in dataframe.columns:
            return dataframe[column].fillna("").astype(str)
        return pd.Series([""] * len(dataframe), index=dataframe.index, dtype=str)

    canonical_dataframe = pd.DataFrame({
        "subject": get_string_series(detected_columns["subject"]),
        "sender": get_string_series(detected_columns["sender"]),
        "receiver": get_string_series(detected_columns["receiver"]),
        "date": get_string_series(detected_columns["date"]),
        "body": dataframe[body_column].fillna("").astype(str),
    })
    canonical_dataframe["label"] = 1
    canonical_dataframe["source"] = "Nazario"
    canonical_dataframe["combined_text"] = (
        canonical_dataframe["subject"].str.strip() + "\n" + canonical_dataframe["body"].str.strip()
    ).str.strip()
    canonical_dataframe = canonical_dataframe[canonical_dataframe["combined_text"].str.len() >= 20].copy()
    canonical_dataframe["content_sha256"] = canonical_dataframe["combined_text"].map(
        lambda v: hashlib.sha256(v.encode("utf-8", errors="ignore")).hexdigest()
    )
    canonical_dataframe = canonical_dataframe.drop_duplicates(
        subset=["content_sha256"], keep="first"
    ).reset_index(drop=True)
    canonical_dataframe.insert(0, "sample_id", [f"nazario_{i:06d}" for i in range(len(canonical_dataframe))])
    return canonical_dataframe


def acquire_nazario() -> Dict[str, Any]:
    """Download and prepare Nazario phishing emails."""
    metadata = zenodo_file_metadata(
        str(CELL2_CONFIG["zenodo_record_id"]), str(CELL2_CONFIG["nazario_filename"])
    )
    raw_path = EMAIL_RAW_ROOT / str(CELL2_CONFIG["nazario_filename"])
    prepared_path = PREPARED_ROOT / "nazario_phishing_email.csv"

    download_resumable(
        str(metadata["url"]), raw_path, expected_size=metadata["size"], expected_md5=metadata["md5"],
        minimum_bytes=1_000_000, description="Nazario.csv",
    )
    raw_dataframe = read_csv_robust(raw_path, compression="infer")
    canonical_dataframe = canonicalize_nazario(raw_dataframe)
    if canonical_dataframe.empty:
        raise RuntimeError("Nazario canonical dataset is empty.")
    atomic_csv_save(canonical_dataframe, prepared_path)

    return {
        "available": True, "prepared_path": str(prepared_path), "canonical_path": str(prepared_path),
        "dataset_path": str(prepared_path), "raw_path": str(raw_path),
        "row_count": int(len(canonical_dataframe)),
        "class_counts": {"1": int(len(canonical_dataframe))},
    }


def decode_email_bytes(raw_email_bytes: bytes) -> Dict[str, str]:
    """Decode one RFC email message."""
    message = email.message_from_bytes(raw_email_bytes)
    subject = str(message.get("Subject", ""))
    sender = str(message.get("From", ""))
    receiver = str(message.get("To", ""))
    date = str(message.get("Date", ""))
    body_parts: List[str] = []

    if message.is_multipart():
        for message_part in message.walk():
            content_type = message_part.get_content_type()
            content_disposition = str(message_part.get("Content-Disposition", "")).lower()
            if content_type == "text/plain" and "attachment" not in content_disposition:
                payload = message_part.get_payload(decode=True)
                if payload:
                    charset = message_part.get_content_charset() or "utf-8"
                    try:
                        body_parts.append(payload.decode(charset, errors="replace"))
                    except LookupError:
                        body_parts.append(payload.decode("utf-8", errors="replace"))
    else:
        payload = message.get_payload(decode=True)
        if payload:
            charset = message.get_content_charset() or "utf-8"
            try:
                body_parts.append(payload.decode(charset, errors="replace"))
            except LookupError:
                body_parts.append(payload.decode("utf-8", errors="replace"))
        else:
            body_parts.append(str(message.get_payload() or ""))

    return {"subject": subject, "sender": sender, "receiver": receiver, "date": date, "body": "\n".join(body_parts)}


def acquire_spamassassin_ham() -> Dict[str, Any]:
    """Download and prepare legitimate SpamAssassin emails."""
    prepared_path = PREPARED_ROOT / "spamassassin_legitimate_email.csv"
    email_rows: List[Dict[str, Any]] = []
    archive_paths: List[str] = []

    for corpus_name, archive_url in dict(CELL2_CONFIG["spamassassin_archives"]).items():
        archive_filename = Path(urlparse(str(archive_url)).path).name
        archive_path = EMAIL_RAW_ROOT / archive_filename
        download_resumable(
            str(archive_url), archive_path, minimum_bytes=100_000,
            description=f"SpamAssassin {corpus_name}",
        )
        archive_paths.append(str(archive_path))

        with tarfile.open(archive_path, mode="r:bz2") as archive:
            email_members = [m for m in archive.getmembers() if m.isfile()]
            for member in tqdm(email_members, desc=f"Parsing {corpus_name}", unit="email"):
                basename = Path(member.name).name
                if basename.startswith("cmds") or basename.startswith("."):
                    continue
                extracted_file = archive.extractfile(member)
                if extracted_file is None:
                    continue
                raw_email_bytes = extracted_file.read()
                if not raw_email_bytes:
                    continue
                decoded_email = decode_email_bytes(raw_email_bytes)
                combined_text = (decoded_email["subject"].strip() + "\n" + decoded_email["body"].strip()).strip()
                if len(combined_text) < 20:
                    continue
                content_sha256 = hashlib.sha256(combined_text.encode("utf-8", errors="ignore")).hexdigest()
                email_rows.append({
                    "subject": decoded_email["subject"], "sender": decoded_email["sender"],
                    "receiver": decoded_email["receiver"], "date": decoded_email["date"],
                    "body": decoded_email["body"], "combined_text": combined_text, "label": 0,
                    "source": f"SpamAssassin_{corpus_name}", "content_sha256": content_sha256,
                })

    canonical_dataframe = pd.DataFrame(email_rows).drop_duplicates(
        subset=["content_sha256"], keep="first"
    ).reset_index(drop=True)
    if canonical_dataframe.empty:
        raise RuntimeError("SpamAssassin legitimate email corpus is empty.")
    canonical_dataframe.insert(
        0, "sample_id", [f"spamassassin_{i:06d}" for i in range(len(canonical_dataframe))]
    )
    atomic_csv_save(canonical_dataframe, prepared_path)

    return {
        "available": True, "prepared_path": str(prepared_path), "canonical_path": str(prepared_path),
        "dataset_path": str(prepared_path), "raw_paths": archive_paths,
        "row_count": int(len(canonical_dataframe)), "class_counts": {"0": int(len(canonical_dataframe))},
    }


def merge_binary_email_corpus(
    nazario_information: Mapping[str, Any], spamassassin_information: Mapping[str, Any],
) -> Dict[str, Any]:
    """Merge validated phishing and legitimate emails (feeds BOTH Text and Behavioral models)."""
    output_path = PREPARED_ROOT / "binary_email_corpus.csv"
    nazario_dataframe = pd.read_csv(str(nazario_information["prepared_path"]), low_memory=False)
    spamassassin_dataframe = pd.read_csv(str(spamassassin_information["prepared_path"]), low_memory=False)

    canonical_columns = [
        "sample_id", "subject", "sender", "receiver", "date", "body",
        "combined_text", "label", "source", "content_sha256",
    ]
    for dataframe in [nazario_dataframe, spamassassin_dataframe]:
        for column in canonical_columns:
            if column not in dataframe.columns:
                dataframe[column] = np.nan if column == "label" else ""

    merged_dataframe = pd.concat(
        [nazario_dataframe[canonical_columns], spamassassin_dataframe[canonical_columns]], ignore_index=True,
    )
    merged_dataframe["label"] = pd.to_numeric(merged_dataframe["label"], errors="coerce").astype("Int64")
    merged_dataframe = merged_dataframe[merged_dataframe["label"].isin([0, 1])].copy()
    merged_dataframe["label"] = merged_dataframe["label"].astype(np.int8)
    merged_dataframe["combined_text"] = merged_dataframe["combined_text"].fillna("").astype(str)
    merged_dataframe = merged_dataframe[merged_dataframe["combined_text"].str.len() >= 20]
    merged_dataframe = merged_dataframe.drop_duplicates(
        subset=["content_sha256"], keep="first"
    ).reset_index(drop=True)

    classes = sorted(merged_dataframe["label"].unique().tolist())
    class_counts = {
        str(label): int(count) for label, count in merged_dataframe["label"].value_counts().sort_index().items()
    }
    if classes != [0, 1] or min(class_counts.values()) < 100:
        raise RuntimeError(f"Binary email corpus validation failed. Classes={classes}, counts={class_counts}")

    atomic_csv_save(merged_dataframe, output_path)

    return {
        "available": True, "prepared_path": str(output_path), "canonical_path": str(output_path),
        "dataset_path": str(output_path), "row_count": int(len(merged_dataframe)),
        "class_counts": class_counts,
        "sources": sorted(merged_dataframe["source"].astype(str).unique().tolist()),
    }


# ============================================================================
# URL MODEL — PHISHTANK (PHISHING) + CISCO UMBRELLA TOP 1M (BENIGN)
# ============================================================================

def acquire_phishtank_urls() -> Dict[str, Any]:
    """Download and prepare PhishTank phishing URLs for the URL model."""
    raw_path = URL_RAW_ROOT / "online-valid.csv.bz2"
    prepared_path = PREPARED_ROOT / "phishtank_urls.csv"

    try:
        download_resumable(
            str(CELL2_CONFIG["phishtank_url"]), raw_path, minimum_bytes=500_000, description="PhishTank",
        )
        with raw_path.open("rb") as file_handle:
            is_bzip2 = file_handle.read(3) == b"BZh"
        dataframe = read_csv_robust(raw_path, compression="bz2" if is_bzip2 else None)

        url_column = choose_column(dataframe, ["url"], ["url"])
        if url_column is None:
            raise RuntimeError(f"PhishTank URL column is missing. Columns={list(dataframe.columns)}")

        max_rows = int(CELL2_CONFIG["url_model_max_phishing_urls"])
        canonical_dataframe = pd.DataFrame({
            "url": dataframe[url_column].fillna("").astype(str),
            "label": 1,
            "source": "PhishTank",
        })
        canonical_dataframe = (
            canonical_dataframe[canonical_dataframe["url"].str.len() > 3]
            .drop_duplicates(subset=["url"])
            .reset_index(drop=True)
        )
        if len(canonical_dataframe) > max_rows:
            canonical_dataframe = canonical_dataframe.sample(
                n=max_rows, random_state=int(CONFIG.get("seed", 42))
            ).reset_index(drop=True)

        canonical_dataframe.insert(0, "sample_id", [f"phishtank_{i:08d}" for i in range(len(canonical_dataframe))])
        atomic_csv_save(canonical_dataframe, prepared_path)

        return {
            "available": True, "prepared_path": str(prepared_path), "raw_path": str(raw_path),
            "row_count": int(len(canonical_dataframe)), "class_counts": {"1": int(len(canonical_dataframe))},
        }
    except Exception as exc:
        LOGGER.warning("PhishTank acquisition unavailable: %s", exc)
        return {"available": False, "error": str(exc), "raw_path": str(raw_path)}


def acquire_benign_urls() -> Dict[str, Any]:
    """Download and prepare Cisco Umbrella Top 1M as benign URLs for the URL model."""
    raw_path = URL_RAW_ROOT / "top-1m.csv.zip"
    prepared_path = PREPARED_ROOT / "benign_urls.csv"

    try:
        download_resumable(
            str(CELL2_CONFIG["cisco_umbrella_top1m_url"]), raw_path, minimum_bytes=1_000_000,
            description="Cisco Umbrella Top 1M",
        )

        max_rows = int(CELL2_CONFIG["url_model_max_benign_urls"])
        with zipfile.ZipFile(raw_path, "r") as archive:
            inner_name = next(n for n in archive.namelist() if n.lower().endswith(".csv"))
            with archive.open(inner_name, "r") as inner_file:
                dataframe = pd.read_csv(
                    inner_file, header=None, names=["rank", "domain"], nrows=max_rows,
                )

        canonical_dataframe = pd.DataFrame({
            "url": "http://" + dataframe["domain"].astype(str).str.strip(),
            "label": 0,
            "source": "CiscoUmbrellaTop1M",
        })
        canonical_dataframe = canonical_dataframe.drop_duplicates(subset=["url"]).reset_index(drop=True)
        canonical_dataframe.insert(0, "sample_id", [f"benignurl_{i:08d}" for i in range(len(canonical_dataframe))])
        atomic_csv_save(canonical_dataframe, prepared_path)

        return {
            "available": True, "prepared_path": str(prepared_path), "raw_path": str(raw_path),
            "row_count": int(len(canonical_dataframe)), "class_counts": {"0": int(len(canonical_dataframe))},
        }
    except Exception as exc:
        LOGGER.warning("Benign URL list acquisition unavailable: %s", exc)
        return {"available": False, "error": str(exc), "raw_path": str(raw_path)}


def merge_url_corpus(phishtank_information: Mapping[str, Any], benign_information: Mapping[str, Any]) -> Dict[str, Any]:
    """Merge phishing and benign URLs into the URL model's binary corpus."""
    output_path = PREPARED_ROOT / "binary_url_corpus.csv"

    if not phishtank_information.get("available") or not benign_information.get("available"):
        LOGGER.warning(
            "URL model corpus incomplete (phishtank_available=%s, benign_available=%s); "
            "URL model will be skipped downstream.",
            phishtank_information.get("available"), benign_information.get("available"),
        )
        return {"available": False, "row_count": 0, "class_counts": {}}

    phishing_dataframe = pd.read_csv(phishtank_information["prepared_path"], low_memory=False)
    benign_dataframe = pd.read_csv(benign_information["prepared_path"], low_memory=False)

    canonical_columns = ["sample_id", "url", "label", "source"]
    merged_dataframe = pd.concat(
        [phishing_dataframe[canonical_columns], benign_dataframe[canonical_columns]], ignore_index=True,
    )
    merged_dataframe["label"] = merged_dataframe["label"].astype(np.int8)
    merged_dataframe = merged_dataframe.drop_duplicates(subset=["url"], keep="first").reset_index(drop=True)

    classes = sorted(merged_dataframe["label"].unique().tolist())
    class_counts = {
        str(label): int(count) for label, count in merged_dataframe["label"].value_counts().sort_index().items()
    }
    if classes != [0, 1] or min(class_counts.values()) < 100:
        raise RuntimeError(f"URL corpus validation failed. Classes={classes}, counts={class_counts}")

    atomic_csv_save(merged_dataframe, output_path)

    return {
        "available": True, "prepared_path": str(output_path), "dataset_path": str(output_path),
        "row_count": int(len(merged_dataframe)), "class_counts": class_counts,
        "sources": sorted(merged_dataframe["source"].astype(str).unique().tolist()),
    }


# ============================================================================
# WEB CONTENT MODEL — MENDELEY COMPPHISH V2 (URL + HTML + LABEL, PAIRED)
# (resurrects the acquisition machinery already debugged earlier in this
# project: Mendeley API resolution -> Playwright browser fallback with
# Cloudflare-wait -> in-browser-context download -> resumable HTML extraction)
# ============================================================================

_CHROMIUM_APT_FALLBACK_PACKAGES: List[str] = [
    "libnss3", "libnspr4", "libatk1.0-0", "libatk-bridge2.0-0", "libcups2",
    "libdrm2", "libdbus-1-3", "libxcb1", "libxkbcommon0", "libx11-6",
    "libxcomposite1", "libxdamage1", "libxext6", "libxfixes3", "libxrandr2",
    "libgbm1", "libpango-1.0-0", "libcairo2", "libasound2", "libatspi2.0-0",
    "fonts-liberation", "xdg-utils",
]


def ensure_playwright_chromium() -> None:
    """Ensure Playwright's Chromium binary AND its OS shared libraries are installed."""
    if PLAYWRIGHT_IMPORT_ERROR:
        raise RuntimeError(
            f"Playwright package is not importable ({PLAYWRIGHT_IMPORT_ERROR}). "
            "Restart the Colab runtime and re-run Cell 1 then Cell 2."
        )
    try:
        subprocess.check_call([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"])
        return
    except subprocess.CalledProcessError as exc:
        LOGGER.warning("`playwright install --with-deps chromium` failed (%s); trying manual apt fallback.", exc)
    try:
        subprocess.check_call(["apt-get", "update", "-qq"])
        subprocess.check_call(["apt-get", "install", "-y", "-qq"] + _CHROMIUM_APT_FALLBACK_PACKAGES)
    except (subprocess.CalledProcessError, FileNotFoundError) as apt_exc:
        LOGGER.warning("Manual apt dependency install failed (%s).", apt_exc)
    try:
        subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
    except subprocess.CalledProcessError as exc2:
        raise RuntimeError(f"Failed to install Playwright Chromium binary: {exc2}") from exc2


async def wait_out_cloudflare_challenge(page, max_wait_ms: int = 30_000, poll_interval_ms: int = 1_000) -> bool:
    """Poll the page title for a Cloudflare 'Just a moment...' interstitial and wait for it to clear."""
    elapsed_ms = 0
    while elapsed_ms < max_wait_ms:
        try:
            title = await page.title()
        except Exception:
            return True
        if "just a moment" not in title.lower():
            return True
        await page.wait_for_timeout(poll_interval_ms)
        elapsed_ms += poll_interval_ms
    LOGGER.warning("Cloudflare interstitial did not clear within %sms.", max_wait_ms)
    return False


async def download_via_browser_context(
    context, url: str, destination: Path, minimum_bytes: int,
    expected_size: Optional[int] = None, expected_sha256: Optional[str] = None,
    timeout_ms: int = 300_000,
) -> bool:
    """Download a file using the Playwright browser context's network stack (bypasses Cloudflare)."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        response = await context.request.get(url, timeout=timeout_ms)
        if not response.ok:
            LOGGER.warning("Browser-context download HTTP %s for %s", response.status, url)
            return False
        body = await response.body()
    except Exception as exc:
        LOGGER.warning("Browser-context download failed for %s: %s", url, exc)
        return False

    if len(body) < minimum_bytes:
        LOGGER.warning("Browser-context download too small for %s: %d bytes", url, len(body))
        return False

    temporary_path = destination.with_suffix(destination.suffix + ".tmp")
    with temporary_path.open("wb") as file_handle:
        file_handle.write(body)
    os.replace(temporary_path, destination)

    try:
        validate_downloaded_payload(destination, minimum_bytes)
    except Exception as exc:
        LOGGER.warning("Downloaded file failed validation for %s: %s", url, exc)
        destination.unlink(missing_ok=True)
        return False

    if expected_size is not None and destination.stat().st_size != int(expected_size):
        destination.unlink(missing_ok=True)
        return False
    if expected_sha256 and hash_file(destination, "sha256").lower() != expected_sha256.lower():
        destination.unlink(missing_ok=True)
        return False

    LOGGER.info("Browser-context download succeeded: %s (%d bytes)", destination, len(body))
    return True


def run_coroutine_in_separate_thread(coroutine_factory):
    """Run async Playwright outside Colab's existing asyncio loop."""
    def runner():
        return asyncio.run(coroutine_factory())
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(runner).result()


def recursively_find_file_records(payload: Any) -> List[Dict[str, Any]]:
    """Locate Mendeley file objects inside nested JSON."""
    collected_records: List[Dict[str, Any]] = []

    def walk(current_value: Any) -> None:
        if isinstance(current_value, dict):
            filename = (
                current_value.get("filename") or current_value.get("file_name") or current_value.get("key")
            )
            content_details = current_value.get("content_details")
            has_file_information = bool(
                filename and (
                    isinstance(content_details, dict) or current_value.get("download_url")
                    or current_value.get("links") or current_value.get("size")
                )
            )
            if has_file_information:
                collected_records.append(dict(current_value))
            for nested_value in current_value.values():
                walk(nested_value)
        elif isinstance(current_value, list):
            for nested_value in current_value:
                walk(nested_value)

    walk(payload)

    unique_records: Dict[Tuple[str, str], Dict[str, Any]] = {}
    for record in collected_records:
        filename = str(record.get("filename") or record.get("file_name") or record.get("key") or "")
        content_details = record.get("content_details") if isinstance(record.get("content_details"), dict) else {}
        file_id = str(
            record.get("id") or record.get("file_id") or record.get("uuid") or content_details.get("id") or ""
        )
        unique_records[(Path(filename).name.lower(), file_id)] = record
    return list(unique_records.values())


def normalize_mendeley_file_record(record: Mapping[str, Any]) -> Dict[str, Any]:
    """Normalize different Mendeley file object formats."""
    content_details = record.get("content_details") if isinstance(record.get("content_details"), dict) else {}
    links = record.get("links") if isinstance(record.get("links"), dict) else {}
    filename = str(record.get("filename") or record.get("file_name") or record.get("key") or "")
    size_value = content_details.get("size") or record.get("size") or record.get("file_size")
    sha256_value = content_details.get("sha256_hash") or record.get("sha256_hash")

    download_urls: List[str] = []
    for candidate_url in [
        content_details.get("download_url"), record.get("download_url"),
        links.get("content"), links.get("download"), record.get("url"),
    ]:
        if candidate_url and str(candidate_url).startswith("http"):
            download_urls.append(str(candidate_url))

    return {
        "filename": Path(filename).name,
        "size": int(size_value) if size_value not in {None, ""} else None,
        "sha256": str(sha256_value) if sha256_value else None,
        "download_urls": list(dict.fromkeys(download_urls)),
    }


def mendeley_headers() -> Dict[str, str]:
    """Create Mendeley request headers."""
    return {
        "Accept": "application/vnd.mendeley-public-dataset.1+json, application/json",
        "Referer": str(CELL2_CONFIG["comp_phish_page"]),
        "Origin": "https://data.mendeley.com",
        "Session-Id": str(uuid.uuid4()),
    }


def collect_mendeley_api_records() -> Tuple[List[Dict[str, Any]], List[str]]:
    """Try official Mendeley API variants."""
    dataset_id = str(CELL2_CONFIG["comp_phish_id"])
    version = int(CELL2_CONFIG["comp_phish_version"])
    headers = mendeley_headers()
    endpoints = [
        f"https://api.mendeley.com/datasets/{dataset_id}?version={version}&fields=*",
        f"https://api.mendeley.com/datasets/{dataset_id}?version={version}",
        f"https://api.data.mendeley.com/datasets/{dataset_id}?version={version}&fields=*",
        f"https://api.data.mendeley.com/datasets/{dataset_id}?version={version}",
    ]
    records: List[Dict[str, Any]] = []
    errors: List[str] = []
    for endpoint in endpoints:
        try:
            payload = request_json(endpoint, headers=headers)
            records.extend(recursively_find_file_records(payload))
        except Exception as exc:
            errors.append(f"{endpoint}: {exc}")
    return recursively_find_file_records(records), errors


def download_required_from_records(records: Sequence[Mapping[str, Any]]) -> Dict[str, Path]:
    """Download required CompPhish files from metadata URLs (best-effort, plain requests)."""
    required_files = {
        str(f).lower(): str(f) for f in CELL2_CONFIG["comp_phish_required_files"]
    }
    records_by_name: Dict[str, Dict[str, Any]] = {}
    for record in records:
        normalized_record = normalize_mendeley_file_record(record)
        filename = normalized_record["filename"]
        if filename:
            records_by_name[filename.lower()] = normalized_record

    downloaded_paths: Dict[str, Path] = {}
    for lowercase_filename, exact_filename in required_files.items():
        destination = WEB_RAW_ROOT / exact_filename
        minimum_bytes = (
            int(CELL2_CONFIG["comp_phish_min_html_zip_bytes"]) if exact_filename.lower().endswith(".zip")
            else int(CELL2_CONFIG["comp_phish_min_mapping_bytes"])
        )
        if destination.is_file() and destination.stat().st_size >= minimum_bytes:
            downloaded_paths[exact_filename] = destination
            continue
        metadata = records_by_name.get(lowercase_filename)
        if metadata is None:
            continue
        for download_url in metadata["download_urls"]:
            try:
                download_resumable(
                    download_url, destination, expected_size=metadata["size"],
                    expected_sha256=metadata["sha256"], minimum_bytes=minimum_bytes,
                    headers=mendeley_headers(), description=exact_filename,
                )
                downloaded_paths[exact_filename] = destination
                break
            except Exception as exc:
                LOGGER.warning("Direct download failed for %s: %s", exact_filename, exc)
    return downloaded_paths


async def playwright_comp_phish_async() -> Tuple[List[Dict[str, Any]], Optional[Path], List[str]]:
    """Async Playwright download fallback for the Web Content model corpus."""
    captured_records: List[Dict[str, Any]] = []
    errors: List[str] = []
    response_tasks: List[asyncio.Task] = []
    downloaded_archive: Optional[Path] = None
    browser = None

    async def inspect_response(response) -> None:
        try:
            if "mendeley" not in response.url.lower():
                return
            response_headers = await response.all_headers()
            if "json" not in response_headers.get("content-type", "").lower():
                return
            payload = await response.json()
            captured_records.extend(recursively_find_file_records(payload))
        except Exception:
            pass

    try:
        ensure_playwright_chromium()
        async with async_playwright() as playwright:
            browser = await playwright.chromium.launch(
                headless=True,
                args=["--no-sandbox", "--disable-dev-shm-usage", "--disable-gpu",
                      "--disable-blink-features=AutomationControlled"],
            )
            context = await browser.new_context(
                accept_downloads=True, locale="en-US",
                user_agent=(
                    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
                ),
                viewport={"width": 1440, "height": 1000},
            )
            page = await context.new_page()
            page.set_default_timeout(int(CELL2_CONFIG["playwright_page_timeout_ms"]))
            page.set_default_navigation_timeout(int(CELL2_CONFIG["playwright_page_timeout_ms"]))

            def response_handler(response) -> None:
                response_tasks.append(asyncio.create_task(inspect_response(response)))
            page.on("response", response_handler)

            dataset_page = str(CELL2_CONFIG["comp_phish_page"])
            LOGGER.info("Opening official CompPhish page: %s", dataset_page)
            await page.goto(dataset_page, wait_until="domcontentloaded")
            try:
                await page.wait_for_load_state("networkidle", timeout=90_000)
            except Exception:
                await page.wait_for_timeout(15_000)

            cloudflare_cleared = await wait_out_cloudflare_challenge(
                page, max_wait_ms=int(CELL2_CONFIG["playwright_cloudflare_wait_ms"])
            )
            if not cloudflare_cleared:
                errors.append("Cloudflare interstitial did not clear within the configured wait.")

            for selector in [
                "button:has-text('Accept All')", "button:has-text('Accept all')",
                "button:has-text('I agree')", "button:has-text('Agree')",
            ]:
                try:
                    cookie_button = page.locator(selector).first
                    if await cookie_button.count() > 0 and await cookie_button.is_visible():
                        await cookie_button.click(force=True, timeout=10_000)
                        await page.wait_for_timeout(2_000)
                        break
                except Exception:
                    continue

            dataset_id = str(CELL2_CONFIG["comp_phish_id"])
            version = int(CELL2_CONFIG["comp_phish_version"])
            for endpoint in [
                f"https://api.mendeley.com/datasets/{dataset_id}?version={version}&fields=*",
                f"https://api.data.mendeley.com/datasets/{dataset_id}?version={version}&fields=*",
                f"https://api.mendeley.com/datasets/{dataset_id}?version={version}",
            ]:
                try:
                    api_response = await context.request.get(
                        endpoint,
                        headers={
                            "Accept": "application/vnd.mendeley-public-dataset.1+json, application/json",
                            "Referer": dataset_page, "Origin": "https://data.mendeley.com",
                            "Session-Id": str(uuid.uuid4()),
                        },
                        timeout=120_000,
                    )
                    if api_response.ok:
                        captured_records.extend(recursively_find_file_records(await api_response.json()))
                    else:
                        errors.append(f"Browser API HTTP {api_response.status}: {endpoint}")
                except Exception as exc:
                    errors.append(f"Browser API {endpoint}: {exc}")

            if response_tasks:
                await asyncio.gather(*response_tasks, return_exceptions=True)

            required_files_map = {str(n).lower(): str(n) for n in CELL2_CONFIG["comp_phish_required_files"]}
            records_by_name: Dict[str, Dict[str, Any]] = {}
            for record in captured_records:
                normalized = normalize_mendeley_file_record(record)
                if normalized["filename"]:
                    records_by_name[normalized["filename"].lower()] = normalized

            context_download_success: Dict[str, bool] = {}
            for lowercase_filename, exact_filename in required_files_map.items():
                destination = WEB_RAW_ROOT / exact_filename
                minimum_bytes = (
                    int(CELL2_CONFIG["comp_phish_min_html_zip_bytes"]) if exact_filename.lower().endswith(".zip")
                    else int(CELL2_CONFIG["comp_phish_min_mapping_bytes"])
                )
                if destination.is_file() and destination.stat().st_size >= minimum_bytes:
                    context_download_success[exact_filename] = True
                    continue
                metadata = records_by_name.get(lowercase_filename)
                if metadata is None:
                    context_download_success[exact_filename] = False
                    continue
                succeeded = False
                for download_url in metadata["download_urls"]:
                    succeeded = await download_via_browser_context(
                        context, download_url, destination, minimum_bytes,
                        expected_size=metadata["size"], expected_sha256=metadata["sha256"],
                        timeout_ms=int(CELL2_CONFIG["playwright_context_download_timeout_ms"]),
                    )
                    if succeeded:
                        break
                    errors.append(f"Browser-context download failed for {exact_filename} via {download_url}")
                context_download_success[exact_filename] = succeeded

            required_names_downloaded = all(
                context_download_success.get(name, False) for name in required_files_map.values()
            )

            if not required_names_downloaded:
                LOGGER.info("Trying automatic Mendeley Download All...")
                await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                await page.wait_for_timeout(3_000)

                async def save_browser_download(download) -> Path:
                    suggested_filename = download.suggested_filename or "comp_phish_download_all.zip"
                    destination = LOCAL_BROWSER_DOWNLOAD_ROOT / suggested_filename
                    await download.save_as(str(destination))
                    validate_downloaded_payload(destination, minimum_bytes=100_000)
                    return destination

                for selector in [
                    "button:has-text('Download All')", "button:has-text('Download all')",
                    "a:has-text('Download All')", "a:has-text('Download all')",
                ]:
                    locator = page.locator(selector).first
                    try:
                        if await locator.count() == 0 or not await locator.is_visible():
                            continue
                        try:
                            async with page.expect_download(timeout=30_000) as download_information:
                                await locator.click(force=True, timeout=30_000)
                            downloaded_archive = await save_browser_download(await download_information.value)
                        except PlaywrightTimeoutError:
                            for confirmation_selector in [
                                "[role='dialog'] button:has-text('Download')",
                                "button:has-text('Accept and Download')", "button:has-text('Download')",
                            ]:
                                confirmation_button = page.locator(confirmation_selector).first
                                try:
                                    if await confirmation_button.count() == 0 or not await confirmation_button.is_visible():
                                        continue
                                    async with page.expect_download(
                                        timeout=int(CELL2_CONFIG["playwright_download_timeout_ms"])
                                    ) as download_information:
                                        await confirmation_button.click(force=True, timeout=30_000)
                                    downloaded_archive = await save_browser_download(await download_information.value)
                                    break
                                except Exception as exc:
                                    errors.append(f"Confirmation selector {confirmation_selector}: {exc}")
                        if downloaded_archive is not None:
                            break
                    except Exception as exc:
                        errors.append(f"Download selector {selector}: {exc}")

            await context.close()
            await browser.close()
            browser = None
    except Exception as exc:
        errors.append(f"Async Playwright fallback failed: {type(exc).__name__}: {exc}")
        if browser is not None:
            try:
                await browser.close()
            except Exception:
                pass

    unique_records: Dict[Tuple[str, str], Dict[str, Any]] = {}
    for record in captured_records:
        normalized_record = normalize_mendeley_file_record(record)
        record_key = (normalized_record["filename"].lower(), json.dumps(record, sort_keys=True, default=str)[:200])
        unique_records[record_key] = dict(record)

    return list(unique_records.values()), downloaded_archive, errors


def playwright_comp_phish_fallback() -> Tuple[List[Dict[str, Any]], Optional[Path], List[str]]:
    """Synchronous wrapper for the async Playwright fallback."""
    return run_coroutine_in_separate_thread(playwright_comp_phish_async)


def extract_required_from_download_all(archive_path: Path) -> Dict[str, Path]:
    """Extract Mapping_File.xlsx and All_HTML.zip from a Download-All archive."""
    if archive_path is None or not archive_path.is_file():
        return {}
    if not zipfile.is_zipfile(archive_path):
        raise RuntimeError(f"Mendeley Download All file is not a valid ZIP: {archive_path}")
    required_files = {str(f).lower(): str(f) for f in CELL2_CONFIG["comp_phish_required_files"]}
    extracted_paths: Dict[str, Path] = {}
    with zipfile.ZipFile(archive_path, "r") as archive:
        members_by_name = {Path(m.filename).name.lower(): m for m in archive.infolist()}
        for lowercase_filename, exact_filename in required_files.items():
            member = members_by_name.get(lowercase_filename)
            if member is None:
                continue
            destination = WEB_RAW_ROOT / exact_filename
            temporary_path = destination.with_suffix(destination.suffix + ".tmp")
            with archive.open(member, "r") as source_handle, temporary_path.open("wb") as destination_handle:
                shutil.copyfileobj(source_handle, destination_handle, length=8 * 1024 * 1024)
            os.replace(temporary_path, destination)
            extracted_paths[exact_filename] = destination
    return extracted_paths


def validate_mapping_excel(path: Path) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """Validate and inspect Mapping_File.xlsx."""
    validate_downloaded_payload(path, int(CELL2_CONFIG["comp_phish_min_mapping_bytes"]))
    dataframe = pd.read_excel(path, engine="openpyxl")
    if len(dataframe) < int(CELL2_CONFIG["comp_phish_min_mapping_rows"]):
        raise RuntimeError(f"CompPhish mapping contains only {len(dataframe)} rows.")

    url_column = choose_column(dataframe, ["url", "urls", "website_url", "raw_url"], ["url"])
    label_column = choose_column(dataframe, ["label", "class", "target", "result"], ["label", "class", "target"])
    serial_column = choose_column(
        dataframe,
        ["serial_number", "serial_no", "serial", "sr_no", "s_no", "id", "index", "html_file", "file_name"],
        ["serial", "html", "file", "index"],
    )
    if url_column is None or label_column is None or serial_column is None:
        raise RuntimeError(
            f"Could not detect URL/label/serial columns in Mapping_File.xlsx. Columns={list(dataframe.columns)}"
        )
    return dataframe, {"url": url_column, "label": label_column, "serial": serial_column}


def validate_html_zip(path: Path) -> Dict[str, Any]:
    """Validate All_HTML.zip."""
    validate_downloaded_payload(path, int(CELL2_CONFIG["comp_phish_min_html_zip_bytes"]))
    if not zipfile.is_zipfile(path):
        raise RuntimeError(f"Invalid ZIP file: {path}")
    with zipfile.ZipFile(path, "r") as archive:
        file_members = [m for m in archive.infolist() if not m.is_dir()]
        html_members = [
            m for m in file_members if Path(m.filename).suffix.lower() in {".txt", ".html", ".htm"}
        ]
        if len(html_members) < int(CELL2_CONFIG["comp_phish_min_html_files"]):
            raise RuntimeError(f"All_HTML.zip contains only {len(html_members)} HTML/text files.")
        for member in [html_members[0], html_members[len(html_members) // 2], html_members[-1]]:
            with archive.open(member, "r") as file_handle:
                file_handle.read(min(member.file_size, 4096))
    return {
        "member_count": int(len(file_members)), "html_member_count": int(len(html_members)),
        "zip_size_bytes": int(path.stat().st_size),
    }


def extract_html_zip_resumable(zip_path: Path, destination_root: Path) -> Dict[str, Any]:
    """Extract HTML files while skipping already-complete files (resumable)."""
    destination_root.mkdir(parents=True, exist_ok=True)
    completion_marker_path = destination_root / ".extraction_complete.json"
    source_fingerprint = f"{zip_path.stat().st_size}:{int(zip_path.stat().st_mtime)}"

    if completion_marker_path.is_file():
        try:
            marker_information = json.loads(completion_marker_path.read_text(encoding="utf-8"))
            if marker_information.get("source_fingerprint") == source_fingerprint:
                existing_count = sum(
                    1 for p in destination_root.rglob("*")
                    if p.is_file() and p.suffix.lower() in {".txt", ".html", ".htm"}
                )
                if existing_count >= int(CELL2_CONFIG["comp_phish_min_html_files"]):
                    LOGGER.info("Using extracted CompPhish HTML cache: %s", destination_root)
                    return marker_information
        except Exception:
            pass

    extracted_this_run = 0
    skipped_existing = 0
    with zipfile.ZipFile(zip_path, "r") as archive:
        html_members = [
            m for m in archive.infolist()
            if not m.is_dir() and Path(m.filename).suffix.lower() in {".txt", ".html", ".htm"}
        ]
        for member in tqdm(html_members, desc="Extracting CompPhish HTML", unit="file"):
            destination = safe_archive_destination(destination_root, member.filename)
            destination.parent.mkdir(parents=True, exist_ok=True)
            if destination.is_file() and destination.stat().st_size == member.file_size:
                skipped_existing += 1
                continue
            temporary_path = destination.with_suffix(destination.suffix + ".tmp")
            with archive.open(member, "r") as source_handle, temporary_path.open("wb") as destination_handle:
                shutil.copyfileobj(source_handle, destination_handle, length=1024 * 1024)
            os.replace(temporary_path, destination)
            extracted_this_run += 1

    completion_information = {
        "source_fingerprint": source_fingerprint, "html_file_count": int(len(html_members)),
        "extracted_this_run": int(extracted_this_run), "skipped_existing": int(skipped_existing),
        "completed_at_utc": utc_now_iso(),
    }
    atomic_json_save(completion_information, completion_marker_path)
    return completion_information


def normalize_binary_label(value: Any) -> Optional[int]:
    """Normalize common binary label forms found in the CompPhish mapping."""
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    text = str(value).strip().lower()
    if "phish" in text:
        return 1
    if "legit" in text or "benign" in text or text == "ham":
        return 0
    if text in {"1", "1.0", "malicious", "spam", "fraud", "bad", "true"}:
        return 1
    if text in {"0", "0.0", "good", "false"}:
        return 0
    try:
        numeric_value = int(float(text))
        if numeric_value in {0, 1}:
            return numeric_value
    except Exception:
        pass
    return None


def normalize_serial_number(value: Any) -> str:
    """Normalize serial number or HTML filename."""
    text = str(value).strip()
    text = re.sub(r"\.(txt|html|htm)$", "", text, flags=re.IGNORECASE)
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    return text


def prepare_web_content_dataset(mapping_path: Path, html_root: Path) -> Dict[str, Any]:
    """Build the aligned URL + HTML + label corpus for the Web Content model."""
    canonical_output_path = PREPARED_ROOT / "web_content_corpus.csv"
    mapping_dataframe, detected_columns = validate_mapping_excel(mapping_path)

    html_paths = [
        p for p in html_root.rglob("*") if p.is_file() and p.suffix.lower() in {".txt", ".html", ".htm"}
    ]
    if len(html_paths) < int(CELL2_CONFIG["comp_phish_min_html_files"]):
        raise RuntimeError(f"Extracted Web Content HTML count is too low: {len(html_paths)}")

    html_lookup: Dict[str, str] = {}
    for html_path in html_paths:
        serial_number = normalize_serial_number(html_path.stem)
        html_lookup.setdefault(serial_number, str(html_path))
        html_lookup.setdefault(html_path.name.lower(), str(html_path))

    canonical_rows: List[Dict[str, Any]] = []
    for _, mapping_row in mapping_dataframe.iterrows():
        serial_number = normalize_serial_number(mapping_row[detected_columns["serial"]])
        raw_url = mapping_row[detected_columns["url"]]
        url = "" if pd.isna(raw_url) else str(raw_url).strip()
        label = normalize_binary_label(mapping_row[detected_columns["label"]])

        if not serial_number or not url or url.lower() == "nan" or label is None:
            continue

        html_path = html_lookup.get(serial_number, "")
        if not html_path:
            for extension in [".txt", ".html", ".htm"]:
                html_path = html_lookup.get((serial_number + extension).lower(), "")
                if html_path:
                    break

        html_available = int(bool(html_path and Path(html_path).is_file()))
        canonical_rows.append({
            "sample_id": f"webcontent_{serial_number}", "serial_number": serial_number,
            "url": url, "label": int(label), "html_path": html_path,
            "html_available": html_available, "source": "CompPhish_v2",
        })

    canonical_dataframe = pd.DataFrame(canonical_rows).drop_duplicates(
        subset=["sample_id"], keep="first"
    ).reset_index(drop=True)

    classes = sorted(canonical_dataframe["label"].unique().tolist())
    if len(canonical_dataframe) < int(CELL2_CONFIG["comp_phish_min_mapping_rows"]) or classes != [0, 1]:
        raise RuntimeError(
            f"Prepared Web Content corpus validation failed. "
            f"Rows={len(canonical_dataframe)}, classes={classes}"
        )

    html_available_count = int(canonical_dataframe["html_available"].sum())
    if html_available_count < int(CELL2_CONFIG["comp_phish_min_html_files"]):
        raise RuntimeError(f"Too few mapping rows resolved to HTML files: {html_available_count}")

    atomic_csv_save(canonical_dataframe, canonical_output_path)

    class_counts = {
        str(l): int(c) for l, c in canonical_dataframe["label"].value_counts().sort_index().items()
    }
    return {
        "available": True, "prepared_path": str(canonical_output_path),
        "dataset_path": str(canonical_output_path), "mapping_path": str(mapping_path),
        "html_root": str(html_root), "row_count": int(len(canonical_dataframe)),
        "class_counts": class_counts, "html_available_count": html_available_count,
    }


def acquire_web_content_dataset() -> Dict[str, Any]:
    """Fully automated Web Content model corpus acquisition (Mendeley CompPhish v2)."""
    mapping_path = WEB_RAW_ROOT / "Mapping_File.xlsx"
    html_zip_path = WEB_RAW_ROOT / "All_HTML.zip"
    acquisition_errors: List[str] = []

    def cache_is_valid() -> bool:
        try:
            validate_mapping_excel(mapping_path)
            validate_html_zip(html_zip_path)
            return True
        except Exception as exc:
            acquisition_errors.append(str(exc))
            return False

    try:
        if cache_is_valid():
            LOGGER.info("Valid Web Content cache found.")
        else:
            LOGGER.info("Web Content cache incomplete: %s", acquisition_errors[-1])
            api_records, api_errors = collect_mendeley_api_records()
            acquisition_errors.extend(api_errors)
            if api_records:
                download_required_from_records(api_records)

        if not cache_is_valid():
            browser_records, download_all_archive, browser_errors = playwright_comp_phish_fallback()
            acquisition_errors.extend(browser_errors)
            if browser_records:
                download_required_from_records(browser_records)
            if download_all_archive:
                try:
                    extract_required_from_download_all(download_all_archive)
                finally:
                    if bool(CELL2_CONFIG["delete_local_download_all_after_extract"]):
                        download_all_archive.unlink(missing_ok=True)

        if not cache_is_valid():
            error_report_path = MANIFEST_ROOT / "web_content_acquisition_errors.json"
            atomic_json_save(
                {"dataset_page": CELL2_CONFIG["comp_phish_page"], "errors": acquisition_errors,
                 "created_at_utc": utc_now_iso()},
                error_report_path,
            )
            raise RuntimeError(
                f"Web Content dataset could not be downloaded automatically. "
                f"Error report: {error_report_path}. Latest errors: {acquisition_errors[-5:]}"
            )

        extraction_information = extract_html_zip_resumable(html_zip_path, WEB_EXTRACTED_ROOT)
        prepared_information = prepare_web_content_dataset(mapping_path, WEB_EXTRACTED_ROOT)
        prepared_information.update({
            "html_zip_path": str(html_zip_path), "extraction_information": extraction_information,
            "source_page": CELL2_CONFIG["comp_phish_page"], "doi": "10.17632/fmbs4kp9wz.2",
        })
        return prepared_information

    except Exception as exc:
        LOGGER.warning(
            "Web Content dataset unavailable this run: %s. Web Content model will be skipped downstream.", exc
        )
        return {"available": False, "error": str(exc)}


# ----------------------------------------------------------------------------
# 7. EXECUTE COMPLETE CELL 2 (MULTI-MODEL)
# ----------------------------------------------------------------------------

try:
    cell2_started = time.perf_counter()

    LOGGER.info("--- Text/Behavioral model dataset (shared email corpus) ---")
    nazario_information = acquire_nazario()
    spamassassin_information = acquire_spamassassin_ham()
    binary_email_information = merge_binary_email_corpus(nazario_information, spamassassin_information)

    LOGGER.info("--- URL model dataset ---")
    phishtank_information = acquire_phishtank_urls()
    benign_url_information = acquire_benign_urls()
    binary_url_information = merge_url_corpus(phishtank_information, benign_url_information)

    LOGGER.info("--- Web Content model dataset ---")
    web_content_information = acquire_web_content_dataset()

    datasets_dictionary = {
        "nazario_phishing_email": nazario_information,
        "spamassassin_legitimate_email": spamassassin_information,
        "binary_email_corpus": binary_email_information,
        "phishtank_urls": phishtank_information,
        "benign_urls": benign_url_information,
        "binary_url_corpus": binary_url_information,
        "web_content_corpus": web_content_information,
    }

    dataset_hashes: Dict[str, str] = {}
    hash_candidates = {
        "binary_email_prepared": binary_email_information.get("prepared_path"),
        "binary_url_prepared": binary_url_information.get("prepared_path"),
        "web_content_prepared": web_content_information.get("prepared_path"),
    }
    for name, path_value in hash_candidates.items():
        if path_value and Path(str(path_value)).is_file():
            dataset_hashes[name] = hash_file(Path(str(path_value)), "sha256")

    dataset_manifest_path = MANIFEST_ROOT / "cell2_dataset_info.json"

    DATASET_INFO: Dict[str, Any] = {
        "cell": 2,
        "version": CELL2_CONFIG["version"],
        "scope": "multi_model",
        "created_at_utc": utc_now_iso(),
        "config_hash": CONFIG_HASH,
        "experiment_uuid": EXPERIMENT_UUID,
        "datasets": datasets_dictionary,

        # Per-model dataset access, used directly by Cell 3.
        "model_datasets": {
            "text": binary_email_information,
            "behavioral": binary_email_information,  # same corpus; lexical folded into behavioral
            "url": binary_url_information,
            "web_content": web_content_information,
        },

        "dataset_hashes": dataset_hashes,
        "available_models": [
            model_name for model_name, info in {
                "text": binary_email_information, "behavioral": binary_email_information,
                "url": binary_url_information, "web_content": web_content_information,
            }.items() if bool(info.get("available"))
        ],
        "unavailable_models": [
            model_name for model_name, info in {
                "text": binary_email_information, "behavioral": binary_email_information,
                "url": binary_url_information, "web_content": web_content_information,
            }.items() if not bool(info.get("available"))
        ],
        "processing_seconds": round(time.perf_counter() - cell2_started, 4),
        "manifest_path": str(dataset_manifest_path),
    }

    atomic_json_save(DATASET_INFO, dataset_manifest_path)

    write_status_safe(
        "dataset_management_complete",
        {"cell": 2, "scope": "multi_model", "available_models": DATASET_INFO["available_models"],
         "unavailable_models": DATASET_INFO["unavailable_models"], "manifest_path": str(dataset_manifest_path)},
    )

except Exception as exc:
    error_trace = traceback.format_exc()
    error_report_path = MANIFEST_ROOT / "cell2_failure.json"
    atomic_json_save(
        {"cell": 2, "error": str(exc), "traceback": error_trace, "timestamp_utc": utc_now_iso()},
        error_report_path,
    )
    LOGGER.error("Cell 2 failed: %s\n%s", exc, error_trace)
    write_status_safe("dataset_management_error", {"cell": 2, "error": str(exc)}, is_error=True)
    raise


# ----------------------------------------------------------------------------
# 8. FINAL SUMMARY
# ----------------------------------------------------------------------------

print("=" * 78)
print("MAL-PhishNet — Cell 2 Complete (Multi-Model Dataset Acquisition)".center(78))
print("=" * 78)
print(f"Available models   : {DATASET_INFO['available_models']}")
print(f"Unavailable models  : {DATASET_INFO['unavailable_models']}")
print("-" * 78)
print(f"Text/Behavioral rows: {binary_email_information.get('row_count', 'N/A')} — "
      f"classes {binary_email_information.get('class_counts', {})}")
print(f"URL model rows      : {binary_url_information.get('row_count', 'N/A')} — "
      f"classes {binary_url_information.get('class_counts', {})}")
print(f"Web Content rows    : {web_content_information.get('row_count', 'N/A')} — "
      f"classes {web_content_information.get('class_counts', {})}, "
      f"HTML available: {web_content_information.get('html_available_count', 'N/A')}")
print("-" * 78)
print(f"Manifest            : {DATASET_INFO['manifest_path']}")
print(f"Processing seconds  : {DATASET_INFO['processing_seconds']}")
print("=" * 78)

[2026-08-15 15:01:59] [INFO    ] [MAL-PhishNet] ==============================================================================
[2026-08-15 15:01:59] [INFO    ] [MAL-PhishNet]             MAL-PhishNet — Cell 2: Multi-Model Dataset Acquisition            
[2026-08-15 15:01:59] [INFO    ] [MAL-PhishNet] ==============================================================================
[2026-08-15 15:01:59] [INFO    ] [MAL-PhishNet] ARCHITECTURE CHANGE: acquiring FOUR independent datasets — Text/Behavioral (shared email corpus), URL (PhishTank + Cisco Umbrella), Web Content (Mendeley CompPhish v2).


/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


[2026-08-15 15:02:00] [INFO    ] [MAL-PhishNet] --- Text/Behavioral model dataset (shared email corpus) ---
[2026-08-15 15:02:02] [INFO    ] [MAL-PhishNet] Using cached file: /content/drive/MyDrive/MAL-PhishNet/data/raw/email/Nazario.csv
[2026-08-15 15:02:04] [INFO    ] [MAL-PhishNet] Using cached file: /content/drive/MyDrive/MAL-PhishNet/data/raw/email/20030228_easy_ham.tar.bz2


Parsing easy_ham:   0%|          | 0/2501 [00:00<?, ?email/s]

[2026-08-15 15:02:09] [INFO    ] [MAL-PhishNet] Using cached file: /content/drive/MyDrive/MAL-PhishNet/data/raw/email/20030228_hard_ham.tar.bz2


Parsing hard_ham:   0%|          | 0/251 [00:00<?, ?email/s]

[2026-08-15 15:02:13] [INFO    ] [MAL-PhishNet] --- URL model dataset ---
[2026-08-15 15:02:13] [INFO    ] [MAL-PhishNet] Using cached file: /content/drive/MyDrive/MAL-PhishNet/data/raw/url/online-valid.csv.bz2
[2026-08-15 15:02:14] [INFO    ] [MAL-PhishNet] Using cached file: /content/drive/MyDrive/MAL-PhishNet/data/raw/url/top-1m.csv.zip
[2026-08-15 15:02:15] [INFO    ] [MAL-PhishNet] --- Web Content model dataset ---
[2026-08-15 15:02:22] [INFO    ] [MAL-PhishNet] Valid Web Content cache found.
[2026-08-15 15:03:15] [INFO    ] [MAL-PhishNet] Using extracted CompPhish HTML cache: /content/drive/MyDrive/MAL-PhishNet/data/extracted/web_content
       MAL-PhishNet — Cell 2 Complete (Multi-Model Dataset Acquisition)       
Available models   : ['text', 'behavioral', 'url', 'web_content']
Unavailable models  : []
------------------------------------------------------------------------------
Text/Behavioral rows: 4284 — classes {'0': 2720, '1': 1564}
URL model rows      : 40000 — classes {

/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


In [ ]:
# ============================================================================
# MAL-PhishNet — Cell 3: Multi-Model Feature Engineering
# (Text / Behavioral / URL / Web Content — four independent pipelines)
# ============================================================================
# Run updated Cell 2 (multi-model) before this cell.
#
# ARCHITECTURE CHANGE: builds FOUR completely independent feature matrices,
# one per model, each with its OWN stratified train/validation/test split
# (never shared across models, since each model's dataset is independent):
#
#   1. Text Model        : TF-IDF only (lowercase, HTML/URL removal,
#                           tokenization, stopword removal, lemmatization).
#                           NO lexical/behavioral features mixed in.
#   2. Behavioral Model   : handcrafted lexical + behavioral features only.
#                           NO TF-IDF (per supervisor's explicit instruction
#                           "DO NOT use TF-IDF here"). Lexical features are
#                           folded into this model per your decision.
#   3. URL Model          : handcrafted URL-string features (length, dots,
#                           digits, entropy, HTTPS, shortener, IP, TLD,
#                           suspicious keywords). No text/behavioral mixing.
#   4. Web Content Model  : handcrafted HTML/DOM structural features
#                           (forms, iframes, scripts, external links, tag
#                           counts) extracted from the paired URL+HTML
#                           corpus. Kept as handcrafted features (not a
#                           CNN/Transformer/graph embedding) so it remains
#                           compatible with the classical-ML training/
#                           evaluation cells (5/6) already built — this is
#                           a deliberate, disclosed simplification within
#                           the supervisor's explicitly allowed "Random
#                           Forest" option for this model.
#
# Each model's dataset is read from DATASET_INFO["model_datasets"][name].
# If a model's dataset was unavailable in Cell 2 (available=False), that
# model's pipeline is skipped entirely with a logged warning — the other
# three still build normally.
#
# Final output:
#   PREPROCESSING_INFO   (keys: "text", "behavioral", "url", "web_content",
#                          each either a result dict or None)
# ============================================================================


# ----------------------------------------------------------------------------
# 0. IMPORTS AND DEPENDENCY CHECK
# ----------------------------------------------------------------------------

import os
import re
import gc
import json
import math
import time
import hashlib
import ipaddress
import traceback
import unicodedata

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Set, Tuple
from urllib.parse import urlsplit, urljoin


_REQUIRED_GLOBALS = [
    "CONFIG", "CONFIG_HASH", "EXPERIMENT_UUID", "PATHS", "LOGGER",
    "STATUS_PATH", "write_status", "DATASET_INFO",
]
_MISSING_GLOBALS = [name for name in _REQUIRED_GLOBALS if name not in globals()]
if _MISSING_GLOBALS:
    raise RuntimeError(
        f"Cell 3 requires updated Cells 1 and 2 first. Missing objects: {_MISSING_GLOBALS}"
    )


# ----------------------------------------------------------------------------
# 1. PACKAGES
# ----------------------------------------------------------------------------

_CELL3_PACKAGES = {
    "beautifulsoup4": "bs4", "lxml": "lxml", "scikit-learn": "sklearn",
    "joblib": "joblib", "nltk": "nltk", "tldextract": "tldextract", "tqdm": "tqdm",
}

if "ensure_packages_installed" not in globals():
    raise RuntimeError("Cell 1 function ensure_packages_installed is unavailable.")

_PACKAGE_RESULTS = ensure_packages_installed(_CELL3_PACKAGES)
_FAILED_PACKAGES = {
    package: result for package, result in _PACKAGE_RESULTS.items()
    if str(result).startswith("failed")
}
if _FAILED_PACKAGES:
    raise RuntimeError(f"Cell 3 package installation failed:\n{json.dumps(_FAILED_PACKAGES, indent=2)}")

import joblib
import numpy as np
import pandas as pd
import tldextract

from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import nltk

for _nltk_resource, _nltk_package in [
    ("tokenizers/punkt", "punkt"), ("tokenizers/punkt_tab", "punkt_tab"),
    ("corpora/stopwords", "stopwords"), ("corpora/wordnet", "wordnet"), ("corpora/omw-1.4", "omw-1.4"),
]:
    try:
        nltk.data.find(_nltk_resource)
    except LookupError:
        try:
            nltk.download(_nltk_package, quiet=True)
        except Exception:
            pass

from nltk.corpus import stopwords as nltk_stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


write_status(
    "preprocessing_pipeline_start", STATUS_PATH,
    extra={"cell": 3, "experiment_uuid": EXPERIMENT_UUID, "scope": "multi_model"},
)

LOGGER.info("=" * 78)
LOGGER.info("MAL-PhishNet — Cell 3: Multi-Model Feature Engineering".center(78))
LOGGER.info("=" * 78)


# ----------------------------------------------------------------------------
# 2. LOCAL CONFIGURATION
# ----------------------------------------------------------------------------

_DEFAULT_PREPROCESSING_CONFIG: Dict[str, Any] = {
    "version": "3.5.0-multi-model",
    "seed": int(CONFIG.get("seed", 42)),

    "train_split": 0.70, "validation_split": 0.15, "test_split": 0.15,

    "tfidf_max_features": 500, "tfidf_ngram_range": [1, 2], "tfidf_min_df": 2,
    "minimum_token_length": 2,

    "url_max_features_note": "all URL features are handcrafted, no vectorizer needed",

    "suspicious_url_terms": [
        "login", "signin", "verify", "verification", "secure", "security",
        "update", "account", "confirm", "bank", "wallet", "payment",
        "password", "credential", "recover", "unlock", "authenticate",
    ],

    "maximum_html_bytes": 3 * 1024 * 1024,
    "maximum_html_visible_characters": 15_000,
}

PREP_CONFIG: Dict[str, Any] = {
    **_DEFAULT_PREPROCESSING_CONFIG,
    **dict(CONFIG.get("preprocessing", {})),
}

_SPLIT_TOTAL = (
    float(PREP_CONFIG["train_split"]) + float(PREP_CONFIG["validation_split"]) + float(PREP_CONFIG["test_split"])
)
if not math.isclose(_SPLIT_TOTAL, 1.0, rel_tol=1e-7, abs_tol=1e-7):
    raise ValueError(f"Train/validation/test fractions must sum to 1.0. Current total: {_SPLIT_TOTAL}")


# ----------------------------------------------------------------------------
# 3. GENERAL HELPERS
# ----------------------------------------------------------------------------

def utc_now_iso() -> str:
    """Return timezone-aware UTC timestamp."""
    return datetime.now(timezone.utc).isoformat()


def sha256_text(value: str) -> str:
    """Calculate SHA256 of text."""
    return hashlib.sha256(value.encode("utf-8", errors="replace")).hexdigest()


def atomic_json_save(payload: Any, destination: str) -> None:
    """Save JSON atomically."""
    directory = os.path.dirname(destination)
    if directory:
        os.makedirs(directory, exist_ok=True)
    temporary_path = destination + ".tmp"
    with open(temporary_path, "w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, indent=2, sort_keys=True, default=str)
        file_handle.flush()
        os.fsync(file_handle.fileno())
    os.replace(temporary_path, destination)


def atomic_csv_save(dataframe: pd.DataFrame, destination: str) -> None:
    """Save CSV atomically."""
    directory = os.path.dirname(destination)
    if directory:
        os.makedirs(directory, exist_ok=True)
    temporary_path = destination + ".tmp"
    dataframe.to_csv(temporary_path, index=False, encoding="utf-8")
    os.replace(temporary_path, destination)


def clean_text(value: Any) -> str:
    """Normalize whitespace/control characters without altering semantic content."""
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    text = unicodedata.normalize("NFKC", str(value))
    text = "".join(ch for ch in text if ch in {"\n", "\r", "\t"} or unicodedata.category(ch) != "Cc")
    text = text.replace("\u200b", "").replace("\ufeff", "").replace("\u2060", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\r\n?", "\n", text)
    return text.strip()


def fixed_unicode_array(values: Iterable[Any]) -> np.ndarray:
    """Convert strings to a fixed-width Unicode array (never object dtype)."""
    string_values = [str(v) for v in values]
    maximum_length = max([1] + [len(v) for v in string_values])
    return np.asarray(string_values, dtype=f"<U{maximum_length}")


def verify_safe_npz(npz_path: str) -> None:
    """Verify an NPZ contains no object arrays, loadable with allow_pickle=False."""
    with np.load(npz_path, allow_pickle=False) as archive:
        object_arrays = [k for k in archive.files if archive[k].dtype == object]
        if object_arrays:
            raise RuntimeError(f"Unsafe object arrays found in {npz_path}: {object_arrays}")


def save_safe_npz(destination: str, **arrays: np.ndarray) -> None:
    """Save and verify a non-object-dtype NPZ, atomically."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    for name, array in arrays.items():
        if np.asarray(array).dtype == object:
            raise TypeError(f"Object array rejected before saving: {name}")
    temporary_path = destination + ".tmp.npz"
    with open(temporary_path, "wb") as file_handle:
        np.savez_compressed(file_handle, **arrays)
        file_handle.flush()
        os.fsync(file_handle.fileno())
    os.replace(temporary_path, destination)
    verify_safe_npz(destination)


def text_entropy(value: str) -> float:
    """Calculate Shannon entropy of a string."""
    if not value:
        return 0.0
    frequencies = Counter(value)
    total = len(value)
    return float(-sum((c / total) * math.log2(c / total) for c in frequencies.values()))


def require_binary_labels(dataframe: pd.DataFrame, dataset_name: str) -> Dict[int, int]:
    """Require both legitimate (0) and phishing (1) classes present."""
    if "label" not in dataframe.columns:
        raise ValueError(f"{dataset_name} has no label column.")
    labels = pd.to_numeric(dataframe["label"], errors="coerce")
    if labels.isna().any():
        raise ValueError(f"{dataset_name} contains invalid labels.")
    unique_labels = sorted(labels.astype(int).unique().tolist())
    if unique_labels != [0, 1]:
        raise RuntimeError(f"{dataset_name} must contain both classes. Found: {unique_labels}")
    counts = labels.astype(int).value_counts().sort_index().to_dict()
    return {int(k): int(v) for k, v in counts.items()}


def create_stratified_split(dataframe: pd.DataFrame, seed: int) -> Tuple[pd.DataFrame, Dict[str, Dict[int, int]]]:
    """
    Create an independent stratified train/validation/test split for one
    model's dataset. Each model gets its OWN split — never shared with
    another model's samples.

    Args:
        dataframe: Dataframe with at least 'sample_id' and 'label'.
        seed: Random seed for reproducibility.

    Returns:
        Tuple of (dataframe with new 'split' column, per-split class counts).
    """
    labels = dataframe["label"].astype(int).to_numpy()
    indices = np.arange(len(dataframe))

    temporary_fraction = float(PREP_CONFIG["validation_split"]) + float(PREP_CONFIG["test_split"])
    test_fraction_within_temporary = float(PREP_CONFIG["test_split"]) / temporary_fraction

    train_indices, temporary_indices = train_test_split(
        indices, test_size=temporary_fraction, random_state=seed, stratify=labels
    )
    validation_indices, test_indices = train_test_split(
        temporary_indices, test_size=test_fraction_within_temporary,
        random_state=seed + 1, stratify=labels[temporary_indices],
    )

    result = dataframe.copy()
    result["split"] = ""
    result.loc[train_indices, "split"] = "train"
    result.loc[validation_indices, "split"] = "validation"
    result.loc[test_indices, "split"] = "test"

    split_counts: Dict[str, Dict[int, int]] = {}
    for split_name in ("train", "validation", "test"):
        subset_labels = result.loc[result["split"] == split_name, "label"].astype(int)
        classes = sorted(subset_labels.unique().tolist())
        if classes != [0, 1]:
            raise RuntimeError(f"{split_name} split does not contain both classes: {classes}")
        split_counts[split_name] = {int(k): int(v) for k, v in subset_labels.value_counts().sort_index().items()}

    return result, split_counts


# ----------------------------------------------------------------------------
# 4. PIPELINE SIGNATURE AND PER-MODEL PATHS
# ----------------------------------------------------------------------------

_PIPELINE_SIGNATURE_PAYLOAD = {
    "cell3_version": PREP_CONFIG["version"],
    "config_hash": CONFIG_HASH,
    "dataset_hashes": DATASET_INFO.get("dataset_hashes", {}),
    "preprocessing_config": PREP_CONFIG,
}
PIPELINE_SIGNATURE = sha256_text(json.dumps(_PIPELINE_SIGNATURE_PAYLOAD, sort_keys=True, default=str))
_SIGNATURE_DIRECTORY = PIPELINE_SIGNATURE[:16]

_PROCESSED_ROOT = os.path.join(PATHS["datasets_processed"], "preprocessing", _SIGNATURE_DIRECTORY)
_REPORTS_ROOT = os.path.join(PATHS["outputs_metrics"], "preprocessing", _SIGNATURE_DIRECTORY)

MODEL_NAMES = ["text", "behavioral", "url", "web_content"]

CELL3_PATHS: Dict[str, Dict[str, str]] = {}
for _model_name in MODEL_NAMES:
    CELL3_PATHS[_model_name] = {
        "splits": os.path.join(_PROCESSED_ROOT, _model_name, "splits"),
        "features": os.path.join(_PROCESSED_ROOT, _model_name, "features"),
        "sequences": os.path.join(_PROCESSED_ROOT, _model_name, "sequences"),
        "vocabularies": os.path.join(_PROCESSED_ROOT, _model_name, "vocabularies"),
        "scalers": os.path.join(_PROCESSED_ROOT, _model_name, "scalers"),
        "reports": os.path.join(_REPORTS_ROOT, _model_name),
    }
    for _path in CELL3_PATHS[_model_name].values():
        os.makedirs(_path, exist_ok=True)


def dataset_hash_for(model_name: str) -> str:
    """SHA256 of one model's underlying dataset content, from Cell 2."""
    hash_key = {
        "text": "binary_email_prepared", "behavioral": "binary_email_prepared",
        "url": "binary_url_prepared", "web_content": "web_content_prepared",
    }[model_name]
    return str(DATASET_INFO.get("dataset_hashes", {}).get(hash_key, ""))


# ----------------------------------------------------------------------------
# 5. TEXTUAL CLEANING PIPELINE (Text Model only)
# ----------------------------------------------------------------------------

try:
    _STOPWORDS: Set[str] = set(nltk_stopwords.words("english"))
except Exception:
    _STOPWORDS = set()
    LOGGER.warning("NLTK stopwords corpus unavailable; proceeding without stopword removal.")

_LEMMATIZER = WordNetLemmatizer()
URL_PATTERN = re.compile(r"https?://[^\s<>'\"\]\)]+|www\.[^\s<>'\"\]\)]+", re.IGNORECASE)


def strip_html(text: str) -> str:
    """Remove HTML tags, returning only visible text."""
    if "<" not in text:
        return text
    try:
        return BeautifulSoup(text, "lxml").get_text(" ", strip=True)
    except Exception:
        return re.sub(r"<[^>]+>", " ", text)


def clean_text_for_vectorization(raw_text: str) -> str:
    """
    Full textual cleaning pipeline for TF-IDF input (Text Model only):
    lowercase -> HTML removal -> URL removal -> punctuation removal ->
    number normalization -> tokenization -> stopword removal -> lemmatization.
    """
    text = clean_text(raw_text).lower()
    text = strip_html(text)
    text = URL_PATTERN.sub(" ", text)
    text = re.sub(r"\d+", " NUM ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    try:
        tokens = word_tokenize(text)
    except Exception:
        tokens = text.split()

    minimum_length = int(PREP_CONFIG["minimum_token_length"])
    tokens = [t for t in tokens if len(t) >= minimum_length and t not in _STOPWORDS]

    lemmatized_tokens = []
    for token in tokens:
        try:
            lemmatized_tokens.append(_LEMMATIZER.lemmatize(token))
        except Exception:
            lemmatized_tokens.append(token)
    return " ".join(lemmatized_tokens)


# ----------------------------------------------------------------------------
# 6. LEXICAL + BEHAVIORAL FEATURE EXTRACTORS (Behavioral Model)
# ----------------------------------------------------------------------------

IP_PATTERN = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
HTML_TAG_PATTERN = re.compile(r"<[^>]+>")

URGENCY_KEYWORDS = ["urgent", "immediately", "right away", "act now", "expire", "expires",
                    "expiring", "deadline", "within 24 hours", "time-sensitive", "asap"]
THREAT_KEYWORDS = ["suspended", "disabled", "locked", "terminated", "restricted",
                   "unauthorized", "unusual activity", "security alert", "account closure",
                   "legal action", "penalty"]
FINANCIAL_KEYWORDS = ["bank", "payment", "invoice", "refund", "transfer", "wire transfer",
                      "billing", "credit card", "tax refund", "prize", "winner", "lottery",
                      "bitcoin", "cryptocurrency"]
CREDENTIAL_KEYWORDS = ["password", "credential", "login details", "ssn", "social security",
                       "pin number", "security code", "one-time password", "otp"]
LOGIN_KEYWORDS = ["login", "sign in", "signin", "log in", "log-in"]
VERIFICATION_KEYWORDS = ["verify your account", "account verification", "confirm your account",
                         "confirm your identity", "reactivate", "re-activate"]
CLICK_NOW_KEYWORDS = ["click here", "click below", "click now", "click the link",
                      "follow this link", "download now"]
PASSWORD_KEYWORDS = ["password", "passcode", "pin", "reset your password"]
EXECUTABLE_EXTENSIONS = [".exe", ".scr", ".bat", ".cmd", ".js", ".jar", ".vbs", ".msi", ".ps1"]
SHORTENER_DOMAINS = {"bit.ly", "tinyurl.com", "t.co", "goo.gl", "ow.ly", "is.gd",
                     "cutt.ly", "rb.gy", "rebrand.ly"}
SUSPICIOUS_TLDS = {"ru", "tk", "top", "xyz", "click", "info", "gq", "cf", "ml", "ga",
                   "work", "loan", "win", "download"}
FREE_EMAIL_PROVIDERS = {"gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "aol.com",
                        "protonmail.com", "icloud.com", "mail.com"}


def _is_valid_ipv4(candidate: str) -> bool:
    try:
        ipaddress.IPv4Address(candidate)
        return True
    except Exception:
        return False


def _extract_domain(url: str) -> str:
    normalized_url = url if "://" in url else f"http://{url}"
    try:
        return (urlsplit(normalized_url).hostname or "").lower()
    except ValueError:
        return ""


def extract_lexical_features(subject: str, body: str) -> Dict[str, float]:
    """Extract handcrafted lexical features from raw email text (folded into Behavioral model)."""
    subject = clean_text(subject)
    body = clean_text(body)
    combined = f"{subject}\n{body}"

    urls = URL_PATTERN.findall(combined)
    ip_addresses = [m for m in IP_PATTERN.findall(combined) if _is_valid_ipv4(m)]
    html_tags = HTML_TAG_PATTERN.findall(combined)
    domains = {d for d in (_extract_domain(u) for u in urls) if d}

    total_chars = max(len(combined), 1)
    letters = [c for c in combined if c.isalpha()]
    words = combined.split()

    uppercase_count = sum(1 for c in letters if c.isupper())
    digit_count = sum(1 for c in combined if c.isdigit())
    special_count = sum(1 for c in combined if not c.isalnum() and not c.isspace())
    punctuation_count = sum(1 for c in combined if c in ".,;:!?\"'()[]{}-_/\\|@#$%^&*+=<>~`")
    suspicious_keyword_count = sum(combined.lower().count(t) for t in PREP_CONFIG["suspicious_url_terms"])
    attachment_mentions = len(re.findall(r"\battach(?:ment|ed|ments)?\b", combined, flags=re.IGNORECASE))
    unique_words = {w.lower() for w in words}

    return {
        "lex_email_length": float(len(combined)), "lex_subject_length": float(len(subject)),
        "lex_body_length": float(len(body)), "lex_num_urls": float(len(urls)),
        "lex_num_ip_addresses": float(len(ip_addresses)), "lex_num_domains": float(len(domains)),
        "lex_num_special_characters": float(special_count),
        "lex_uppercase_ratio": float(uppercase_count / max(len(letters), 1)),
        "lex_digit_ratio": float(digit_count / total_chars),
        "lex_punctuation_ratio": float(punctuation_count / total_chars),
        "lex_num_suspicious_keywords": float(suspicious_keyword_count),
        "lex_num_attachments_mentioned": float(attachment_mentions),
        "lex_num_html_tags": float(len(html_tags)), "lex_entropy": float(text_entropy(combined)),
        "lex_average_word_length": float(sum(len(w) for w in words) / max(len(words), 1)),
        "lex_vocabulary_richness": float(len(unique_words) / max(len(words), 1)),
    }


_SENDER_PATTERN = re.compile(r'^\s*"?([^"<]*)"?\s*<?([^<>\s]+@[^<>\s]+)>?\s*$')


def parse_sender_field(sender_raw: str) -> Tuple[str, str]:
    """Split a raw 'From' header value into (display_name, email_address)."""
    sender_raw = clean_text(sender_raw)
    match = _SENDER_PATTERN.match(sender_raw)
    if match:
        return match.group(1).strip(), match.group(2).strip().lower()
    at_match = re.search(r"[^\s<>]+@[^\s<>]+", sender_raw)
    if at_match:
        return "", at_match.group(0).lower()
    return sender_raw, ""


def extract_behavioral_features(subject: str, body: str, sender: str) -> Dict[str, float]:
    """
    Extract behavioral indicators derivable from available email metadata.

    NOTE: Reply-To mismatch and SPF/DKIM/DMARC flags are intentionally NOT
    computed — neither source corpus captures that metadata, and fabricating
    it is disallowed.
    """
    subject = clean_text(subject)
    body = clean_text(body)
    combined = f"{subject}\n{body}".lower()

    display_name, email_address = parse_sender_field(sender)
    sender_domain = email_address.split("@")[-1].lower() if "@" in email_address else ""
    sender_domain_suffix = sender_domain.rsplit(".", 1)[-1] if "." in sender_domain else ""

    display_name_lower = display_name.lower()
    embedded_email_match = re.search(r"[^\s]+@[^\s]+", display_name_lower)
    display_name_mismatch = 0.0
    if embedded_email_match:
        embedded_domain = embedded_email_match.group(0).split("@")[-1]
        if sender_domain and embedded_domain and embedded_domain != sender_domain:
            display_name_mismatch = 1.0

    urls = URL_PATTERN.findall(f"{subject}\n{body}")
    url_domains = [d for d in (_extract_domain(u) for u in urls) if d]
    shortened_url_count = sum(1 for d in url_domains if d in SHORTENER_DOMAINS)
    suspicious_tld_count = sum(1 for d in url_domains if d.rsplit(".", 1)[-1] in SUSPICIOUS_TLDS)
    executable_mentions = sum(combined.count(ext) for ext in EXECUTABLE_EXTENSIONS)

    def _keyword_score(keywords: Sequence[str]) -> float:
        return float(sum(combined.count(k) for k in keywords))

    return {
        "beh_sender_domain_length": float(len(sender_domain)),
        "beh_sender_is_free_provider": float(sender_domain in FREE_EMAIL_PROVIDERS),
        "beh_sender_suspicious_tld": float(sender_domain_suffix in SUSPICIOUS_TLDS),
        "beh_display_name_mismatch": float(display_name_mismatch),
        "beh_urgency_score": _keyword_score(URGENCY_KEYWORDS),
        "beh_threatening_language_score": _keyword_score(THREAT_KEYWORDS),
        "beh_financial_request_score": _keyword_score(FINANCIAL_KEYWORDS),
        "beh_credential_request_score": _keyword_score(CREDENTIAL_KEYWORDS),
        "beh_login_keyword_frequency": _keyword_score(LOGIN_KEYWORDS),
        "beh_account_verification_keywords": _keyword_score(VERIFICATION_KEYWORDS),
        "beh_click_now_keywords": _keyword_score(CLICK_NOW_KEYWORDS),
        "beh_password_keywords": _keyword_score(PASSWORD_KEYWORDS),
        "beh_attachment_indicator": float(1.0 if re.search(r"\battach(?:ment|ed|ments)?\b", combined) else 0.0),
        "beh_executable_attachment_mentions": float(executable_mentions),
        "beh_shortened_url_count": float(shortened_url_count),
        "beh_suspicious_tld_url_count": float(suspicious_tld_count),
    }


_OMITTED_BEHAVIORAL_FEATURES = [
    "reply_to_mismatch (no Reply-To header captured by Cell 2's parser)",
    "spf_flag / dkim_flag / dmarc_flag (no authentication-result metadata available)",
]


# ----------------------------------------------------------------------------
# 7. URL FEATURE EXTRACTOR (URL Model)
# ----------------------------------------------------------------------------

_TLD_EXTRACTOR = tldextract.TLDExtract(
    cache_dir=os.path.join(_PROCESSED_ROOT, "tldextract_cache"), suffix_list_urls=(),
)


def extract_domain_parts(url: str) -> Dict[str, str]:
    """Extract domain components (host/domain/suffix/subdomain/registered_domain)."""
    try:
        hostname = (urlsplit(url).hostname or "").lower()
    except ValueError:
        hostname = ""
    extracted = _TLD_EXTRACTOR(hostname)
    registered_domain = ".".join(p for p in (extracted.domain, extracted.suffix) if p)
    return {
        "host": hostname, "domain": extracted.domain or "", "suffix": extracted.suffix or "",
        "subdomain": extracted.subdomain or "", "registered_domain": registered_domain or hostname,
    }


def extract_url_features(raw_url: str) -> Dict[str, float]:
    """
    Extract handcrafted lexical URL features for the URL Model.

    Args:
        raw_url: Raw URL string.

    Returns:
        Dictionary of url_* feature name -> float value.
    """
    url = clean_text(raw_url)
    try:
        parsed = urlsplit(url)
    except ValueError:
        parsed = urlsplit("http://invalid.local/")

    domain_parts = extract_domain_parts(url)
    host, domain, suffix, subdomain = (
        domain_parts["host"], domain_parts["domain"], domain_parts["suffix"], domain_parts["subdomain"]
    )
    path, query, fragment = parsed.path or "", parsed.query or "", parsed.fragment or ""
    lowered = url.lower()

    digit_count = sum(c.isdigit() for c in url)
    letter_count = sum(c.isalpha() for c in url)
    special_count = len(url) - digit_count - letter_count
    subdomain_count = len([p for p in subdomain.split(".") if p])

    try:
        explicit_port = parsed.port is not None
    except ValueError:
        explicit_port = False

    try:
        ipaddress.ip_address(host)
        host_is_ip = 1.0
    except Exception:
        host_is_ip = 0.0

    suspicious_count = sum(lowered.count(t) for t in PREP_CONFIG["suspicious_url_terms"])

    return {
        "url_total_length": float(len(url)), "url_host_length": float(len(host)),
        "url_domain_length": float(len(domain)), "url_suffix_length": float(len(suffix)),
        "url_subdomain_length": float(len(subdomain)), "url_subdomain_count": float(subdomain_count),
        "url_path_length": float(len(path)), "url_query_length": float(len(query)),
        "url_fragment_length": float(len(fragment)), "url_digit_count": float(digit_count),
        "url_letter_count": float(letter_count), "url_special_count": float(special_count),
        "url_digit_ratio": float(digit_count / max(len(url), 1)),
        "url_special_ratio": float(special_count / max(len(url), 1)),
        "url_dot_count": float(url.count(".")), "url_hyphen_count": float(url.count("-")),
        "url_underscore_count": float(url.count("_")), "url_slash_count": float(url.count("/")),
        "url_question_count": float(url.count("?")), "url_equal_count": float(url.count("=")),
        "url_ampersand_count": float(url.count("&")), "url_at_count": float(url.count("@")),
        "url_percent_count": float(url.count("%")),
        "url_https": float(parsed.scheme.lower() == "https"),
        "url_explicit_port": float(explicit_port), "url_host_is_ip": float(host_is_ip),
        "url_punycode": float("xn--" in host), "url_shortener": float(host in SHORTENER_DOMAINS),
        "url_suspicious_tld": float(suffix.lower() in SUSPICIOUS_TLDS),
        "url_suspicious_term_count": float(suspicious_count),
        "url_has_login_term": float("login" in lowered or "signin" in lowered),
        "url_has_verify_term": float("verify" in lowered),
        "url_has_account_term": float("account" in lowered),
        "url_has_password_term": float("password" in lowered or "credential" in lowered),
        "url_entropy": float(text_entropy(url)), "url_host_entropy": float(text_entropy(host)),
        "url_path_entropy": float(text_entropy(path)),
    }


# ----------------------------------------------------------------------------
# 8. WEB CONTENT FEATURE EXTRACTOR (Web Content Model)
# ----------------------------------------------------------------------------

def read_html_file(file_path: str) -> str:
    """Read bounded HTML content with encoding fallback."""
    if not file_path or not os.path.isfile(file_path):
        return ""
    try:
        with open(file_path, "rb") as file_handle:
            raw_bytes = file_handle.read(int(PREP_CONFIG["maximum_html_bytes"]))
        for encoding in ("utf-8", "cp1252", "latin-1"):
            try:
                return raw_bytes.decode(encoding)
            except UnicodeDecodeError:
                continue
        return raw_bytes.decode("utf-8", errors="replace")
    except OSError:
        return ""


def extract_web_content_features(html_content: str, page_url: str) -> Dict[str, float]:
    """
    Extract handcrafted HTML/DOM structural features for the Web Content Model.

    Args:
        html_content: Raw HTML text of the landing page (may be empty if unavailable).
        page_url: The URL this HTML was fetched from (used for external-resource detection).

    Returns:
        Dictionary of web_* feature name -> float value.
    """
    empty_features = {
        "web_available": 0.0, "web_byte_length": 0.0, "web_visible_text_length": 0.0,
        "web_tag_count": 0.0, "web_unique_tag_count": 0.0, "web_anchor_count": 0.0,
        "web_external_link_ratio": 0.0, "web_empty_link_ratio": 0.0, "web_form_count": 0.0,
        "web_external_form_count": 0.0, "web_input_count": 0.0, "web_password_input_count": 0.0,
        "web_hidden_input_count": 0.0, "web_script_count": 0.0, "web_external_script_count": 0.0,
        "web_iframe_count": 0.0, "web_image_count": 0.0, "web_meta_refresh_count": 0.0,
        "web_event_handler_count": 0.0, "web_javascript_link_count": 0.0, "web_eval_count": 0.0,
        "web_base64_count": 0.0, "web_document_write_count": 0.0, "web_title_missing": 1.0,
        "web_favicon_external": 0.0, "web_button_count": 0.0,
    }
    if not html_content:
        return empty_features

    try:
        soup = BeautifulSoup(html_content, "lxml")
    except Exception:
        soup = BeautifulSoup(html_content, "html.parser")

    all_tags = soup.find_all(True)
    anchors, forms, inputs = soup.find_all("a"), soup.find_all("form"), soup.find_all("input")
    scripts, iframes, images = soup.find_all("script"), soup.find_all("iframe"), soup.find_all("img")
    buttons = soup.find_all("button")
    meta_refresh = soup.find_all("meta", attrs={"http-equiv": re.compile("refresh", re.IGNORECASE)})

    page_domain = extract_domain_parts(page_url)["registered_domain"]

    external_links = empty_links = 0
    for anchor in anchors:
        href = clean_text(anchor.get("href", ""))
        if not href or href in {"#", "/"}:
            empty_links += 1
            continue
        target_domain = extract_domain_parts(urljoin(page_url, href))["registered_domain"]
        if target_domain and page_domain and target_domain != page_domain:
            external_links += 1

    external_forms = 0
    for form in forms:
        action = clean_text(form.get("action", ""))
        if not action:
            continue
        target_domain = extract_domain_parts(urljoin(page_url, action))["registered_domain"]
        if target_domain and page_domain and target_domain != page_domain:
            external_forms += 1

    external_scripts = 0
    for script in scripts:
        source = clean_text(script.get("src", ""))
        if not source:
            continue
        target_domain = extract_domain_parts(urljoin(page_url, source))["registered_domain"]
        if target_domain and page_domain and target_domain != page_domain:
            external_scripts += 1

    event_handler_count = sum(
        1 for tag in all_tags for attr in tag.attrs.keys() if str(attr).lower().startswith("on")
    )
    password_input_count = sum(str(t.get("type", "")).lower() == "password" for t in inputs)
    hidden_input_count = sum(str(t.get("type", "")).lower() == "hidden" for t in inputs)
    javascript_links = sum(
        1 for a in anchors if clean_text(a.get("href", "")).lower().startswith("javascript:")
    )

    title_tag = soup.find("title")
    favicon_external = 0
    favicon = soup.find("link", rel=lambda v: v and "icon" in str(v).lower())
    if favicon:
        favicon_domain = extract_domain_parts(urljoin(page_url, clean_text(favicon.get("href", ""))))["registered_domain"]
        favicon_external = int(bool(favicon_domain and page_domain and favicon_domain != page_domain))

    for removable_tag in soup(["script", "style", "noscript"]):
        removable_tag.decompose()
    visible_text = clean_text(soup.get_text(" ", strip=True))[
        : int(PREP_CONFIG["maximum_html_visible_characters"])
    ]
    lowered_html = html_content.lower()
    anchor_denominator = max(len(anchors), 1)

    return {
        "web_available": 1.0,
        "web_byte_length": float(len(html_content.encode("utf-8", errors="replace"))),
        "web_visible_text_length": float(len(visible_text)),
        "web_tag_count": float(len(all_tags)),
        "web_unique_tag_count": float(len({t.name for t in all_tags if t.name})),
        "web_anchor_count": float(len(anchors)),
        "web_external_link_ratio": float(external_links / anchor_denominator),
        "web_empty_link_ratio": float(empty_links / anchor_denominator),
        "web_form_count": float(len(forms)), "web_external_form_count": float(external_forms),
        "web_input_count": float(len(inputs)), "web_password_input_count": float(password_input_count),
        "web_hidden_input_count": float(hidden_input_count), "web_script_count": float(len(scripts)),
        "web_external_script_count": float(external_scripts), "web_iframe_count": float(len(iframes)),
        "web_image_count": float(len(images)), "web_meta_refresh_count": float(len(meta_refresh)),
        "web_event_handler_count": float(event_handler_count),
        "web_javascript_link_count": float(javascript_links),
        "web_eval_count": float(lowered_html.count("eval(")),
        "web_base64_count": float(lowered_html.count("base64")),
        "web_document_write_count": float(lowered_html.count("document.write")),
        "web_title_missing": float(
            title_tag is None or not clean_text(title_tag.get_text(" ", strip=True) if title_tag else "")
        ),
        "web_favicon_external": float(favicon_external), "web_button_count": float(len(buttons)),
    }


# ----------------------------------------------------------------------------
# 9. PER-MODEL BUILD FUNCTIONS
# ----------------------------------------------------------------------------

def build_text_model() -> Dict[str, Any]:
    """Build the Text Model's TF-IDF-only feature matrix."""
    email_information = DATASET_INFO["model_datasets"]["text"]
    email_data = pd.read_csv(email_information["prepared_path"], low_memory=False)
    email_data["sample_id"] = email_data["sample_id"].astype(str)
    require_binary_labels(email_data, "Text model dataset")

    email_data, split_counts = create_stratified_split(email_data, int(PREP_CONFIG["seed"]))
    split_path = os.path.join(CELL3_PATHS["text"]["splits"], "text_split.csv")
    atomic_csv_save(email_data[["sample_id", "label", "split"]], split_path)

    LOGGER.info("[Text Model] Cleaning and tokenizing email text...")
    subjects = email_data["subject"].fillna("").astype(str)
    bodies = email_data["body"].fillna("").astype(str)
    cleaned_texts = [
        clean_text_for_vectorization(f"{s} {b}")
        for s, b in tqdm(zip(subjects, bodies), total=len(email_data), desc="Text cleaning", leave=False)
    ]
    email_data["cleaned_text"] = cleaned_texts

    training_mask = (email_data["split"] == "train").to_numpy()

    LOGGER.info("[Text Model] Fitting TF-IDF vectorizer on training split only...")
    tfidf_vectorizer = TfidfVectorizer(
        max_features=int(PREP_CONFIG["tfidf_max_features"]),
        ngram_range=tuple(PREP_CONFIG["tfidf_ngram_range"]),
        min_df=int(PREP_CONFIG["tfidf_min_df"]), sublinear_tf=True,
    )
    tfidf_vectorizer.fit(email_data.loc[training_mask, "cleaned_text"])
    tfidf_matrix = tfidf_vectorizer.transform(email_data["cleaned_text"]).toarray().astype(np.float32)
    feature_names = [f"tfidf_{t}" for t in tfidf_vectorizer.get_feature_names_out()]

    tfidf_vectorizer_path = os.path.join(CELL3_PATHS["text"]["vocabularies"], "text_tfidf_vectorizer.joblib")
    joblib.dump(
        {"vectorizer": tfidf_vectorizer, "fitted_on": "train_only", "pipeline_signature": PIPELINE_SIGNATURE},
        tfidf_vectorizer_path,
    )

    split_codes = email_data["split"].map({"train": 0, "validation": 1, "test": 2}).astype(np.int8).to_numpy()
    output_path = os.path.join(CELL3_PATHS["text"]["sequences"], "text_model_inputs.npz")
    save_safe_npz(
        output_path,
        sample_ids=fixed_unicode_array(email_data["sample_id"].tolist()),
        labels=email_data["label"].astype(np.int64).to_numpy(),
        split_codes=split_codes, features=tfidf_matrix,
        feature_names=fixed_unicode_array(feature_names),
    )

    class_distribution = require_binary_labels(email_data, "Text model inputs")
    return {
        "model": "text", "input_path": output_path, "tfidf_vectorizer_path": tfidf_vectorizer_path,
        "split_path": split_path, "feature_names": feature_names, "vocabulary_size": len(feature_names),
        "sample_count": int(len(email_data)), "class_distribution": class_distribution,
        "split_counts": split_counts,
    }


def build_behavioral_model() -> Dict[str, Any]:
    """Build the Behavioral Model's handcrafted feature matrix (lexical + behavioral, NO TF-IDF)."""
    email_information = DATASET_INFO["model_datasets"]["behavioral"]
    email_data = pd.read_csv(email_information["prepared_path"], low_memory=False)
    email_data["sample_id"] = email_data["sample_id"].astype(str)
    require_binary_labels(email_data, "Behavioral model dataset")

    email_data, split_counts = create_stratified_split(email_data, int(PREP_CONFIG["seed"]) + 100)
    split_path = os.path.join(CELL3_PATHS["behavioral"]["splits"], "behavioral_split.csv")
    atomic_csv_save(email_data[["sample_id", "label", "split"]], split_path)

    subjects = email_data["subject"].fillna("").astype(str)
    bodies = email_data["body"].fillna("").astype(str)
    senders = email_data["sender"].fillna("").astype(str)

    LOGGER.info("[Behavioral Model] Extracting lexical features...")
    lexical_rows = [
        extract_lexical_features(s, b)
        for s, b in tqdm(zip(subjects, bodies), total=len(email_data), desc="Lexical features", leave=False)
    ]
    LOGGER.info("[Behavioral Model] Extracting behavioral features...")
    behavioral_rows = [
        extract_behavioral_features(s, b, se)
        for s, b, se in tqdm(
            zip(subjects, bodies, senders), total=len(email_data), desc="Behavioral features", leave=False
        )
    ]
    for omitted in _OMITTED_BEHAVIORAL_FEATURES:
        LOGGER.warning("[Behavioral Model] Feature omitted (metadata unavailable): %s", omitted)

    combined_frame = pd.concat([pd.DataFrame(lexical_rows), pd.DataFrame(behavioral_rows)], axis=1)
    feature_names = list(combined_frame.columns)
    numerical_matrix = combined_frame.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)

    training_mask = (email_data["split"] == "train").to_numpy()
    scaler = StandardScaler()
    scaler.fit(numerical_matrix.loc[training_mask])
    scaled_features = scaler.transform(numerical_matrix).astype(np.float32)

    scaler_path = os.path.join(CELL3_PATHS["behavioral"]["scalers"], "behavioral_scaler.joblib")
    joblib.dump(
        {"scaler": scaler, "feature_columns": feature_names, "fitted_on": "train_only",
         "pipeline_signature": PIPELINE_SIGNATURE},
        scaler_path,
    )

    split_codes = email_data["split"].map({"train": 0, "validation": 1, "test": 2}).astype(np.int8).to_numpy()
    output_path = os.path.join(CELL3_PATHS["behavioral"]["sequences"], "behavioral_model_inputs.npz")
    save_safe_npz(
        output_path,
        sample_ids=fixed_unicode_array(email_data["sample_id"].tolist()),
        labels=email_data["label"].astype(np.int64).to_numpy(),
        split_codes=split_codes, features=scaled_features,
        feature_names=fixed_unicode_array(feature_names),
    )

    class_distribution = require_binary_labels(email_data, "Behavioral model inputs")
    return {
        "model": "behavioral", "input_path": output_path, "scaler_path": scaler_path,
        "split_path": split_path, "feature_names": feature_names, "sample_count": int(len(email_data)),
        "class_distribution": class_distribution, "split_counts": split_counts,
        "omitted_features": _OMITTED_BEHAVIORAL_FEATURES,
    }


def build_url_model() -> Dict[str, Any]:
    """Build the URL Model's handcrafted feature matrix."""
    url_information = DATASET_INFO["model_datasets"]["url"]
    url_data = pd.read_csv(url_information["prepared_path"], low_memory=False)
    url_data["sample_id"] = url_data["sample_id"].astype(str)
    require_binary_labels(url_data, "URL model dataset")

    url_data, split_counts = create_stratified_split(url_data, int(PREP_CONFIG["seed"]) + 200)
    split_path = os.path.join(CELL3_PATHS["url"]["splits"], "url_split.csv")
    atomic_csv_save(url_data[["sample_id", "label", "split"]], split_path)

    LOGGER.info("[URL Model] Extracting URL features...")
    feature_rows = [
        extract_url_features(u)
        for u in tqdm(url_data["url"].fillna("").astype(str), desc="URL features", leave=False)
    ]
    feature_frame = pd.DataFrame(feature_rows)
    feature_names = list(feature_frame.columns)
    numerical_matrix = feature_frame.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)

    training_mask = (url_data["split"] == "train").to_numpy()
    scaler = StandardScaler()
    scaler.fit(numerical_matrix.loc[training_mask])
    scaled_features = scaler.transform(numerical_matrix).astype(np.float32)

    scaler_path = os.path.join(CELL3_PATHS["url"]["scalers"], "url_scaler.joblib")
    joblib.dump(
        {"scaler": scaler, "feature_columns": feature_names, "fitted_on": "train_only",
         "pipeline_signature": PIPELINE_SIGNATURE},
        scaler_path,
    )

    split_codes = url_data["split"].map({"train": 0, "validation": 1, "test": 2}).astype(np.int8).to_numpy()
    output_path = os.path.join(CELL3_PATHS["url"]["sequences"], "url_model_inputs.npz")
    save_safe_npz(
        output_path,
        sample_ids=fixed_unicode_array(url_data["sample_id"].tolist()),
        labels=url_data["label"].astype(np.int64).to_numpy(),
        split_codes=split_codes, features=scaled_features,
        feature_names=fixed_unicode_array(feature_names),
    )

    class_distribution = require_binary_labels(url_data, "URL model inputs")
    return {
        "model": "url", "input_path": output_path, "scaler_path": scaler_path, "split_path": split_path,
        "feature_names": feature_names, "sample_count": int(len(url_data)),
        "class_distribution": class_distribution, "split_counts": split_counts,
    }


def build_web_content_model() -> Dict[str, Any]:
    """Build the Web Content Model's handcrafted HTML/DOM feature matrix."""
    web_information = DATASET_INFO["model_datasets"]["web_content"]
    web_data = pd.read_csv(web_information["prepared_path"], low_memory=False)
    web_data["sample_id"] = web_data["sample_id"].astype(str)
    require_binary_labels(web_data, "Web Content model dataset")

    web_data, split_counts = create_stratified_split(web_data, int(PREP_CONFIG["seed"]) + 300)
    split_path = os.path.join(CELL3_PATHS["web_content"]["splits"], "web_content_split.csv")
    atomic_csv_save(web_data[["sample_id", "label", "split"]], split_path)

    LOGGER.info("[Web Content Model] Reading and parsing HTML pages...")
    feature_rows: List[Dict[str, float]] = []
    for _, row in tqdm(web_data.iterrows(), total=len(web_data), desc="Web Content features", leave=False):
        html_path = clean_text(row.get("html_path", ""))
        html_content = read_html_file(html_path)
        feature_rows.append(extract_web_content_features(html_content, clean_text(row.get("url", ""))))

    feature_frame = pd.DataFrame(feature_rows)
    feature_names = list(feature_frame.columns)
    numerical_matrix = feature_frame.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)

    training_mask = (web_data["split"] == "train").to_numpy()
    scaler = StandardScaler()
    scaler.fit(numerical_matrix.loc[training_mask])
    scaled_features = scaler.transform(numerical_matrix).astype(np.float32)

    scaler_path = os.path.join(CELL3_PATHS["web_content"]["scalers"], "web_content_scaler.joblib")
    joblib.dump(
        {"scaler": scaler, "feature_columns": feature_names, "fitted_on": "train_only",
         "pipeline_signature": PIPELINE_SIGNATURE},
        scaler_path,
    )

    split_codes = web_data["split"].map({"train": 0, "validation": 1, "test": 2}).astype(np.int8).to_numpy()
    output_path = os.path.join(CELL3_PATHS["web_content"]["sequences"], "web_content_model_inputs.npz")
    save_safe_npz(
        output_path,
        sample_ids=fixed_unicode_array(web_data["sample_id"].tolist()),
        labels=web_data["label"].astype(np.int64).to_numpy(),
        split_codes=split_codes, features=scaled_features,
        feature_names=fixed_unicode_array(feature_names),
    )

    class_distribution = require_binary_labels(web_data, "Web Content model inputs")
    html_available_count = int(feature_frame["web_available"].sum())
    return {
        "model": "web_content", "input_path": output_path, "scaler_path": scaler_path,
        "split_path": split_path, "feature_names": feature_names, "sample_count": int(len(web_data)),
        "class_distribution": class_distribution, "split_counts": split_counts,
        "html_available_count": html_available_count,
    }


# ----------------------------------------------------------------------------
# 10. MASTER PIPELINE
# ----------------------------------------------------------------------------

_MODEL_BUILDERS = {
    "text": build_text_model, "behavioral": build_behavioral_model,
    "url": build_url_model, "web_content": build_web_content_model,
}


def run_preprocessing_pipeline() -> Dict[str, Any]:
    """Build all four independent model feature matrices, skipping any unavailable dataset."""
    pipeline_started = time.perf_counter()
    resource_before = (
        get_system_resource_snapshot() if "get_system_resource_snapshot" in globals() else {}
    )

    model_datasets = DATASET_INFO.get("model_datasets", {})
    results: Dict[str, Optional[Dict[str, Any]]] = {}

    for model_name, builder_function in _MODEL_BUILDERS.items():
        dataset_info = model_datasets.get(model_name)
        if not dataset_info or not dataset_info.get("available"):
            LOGGER.warning(
                "[%s] Dataset unavailable (available=%s); skipping this model's feature engineering.",
                model_name, dataset_info.get("available") if dataset_info else None,
            )
            results[model_name] = None
            continue

        LOGGER.info("--- Building %s model features ---", model_name)
        try:
            results[model_name] = builder_function()
        except Exception as exc:
            LOGGER.error("[%s] Feature engineering failed: %s", model_name, exc)
            results[model_name] = None

    if not any(results.values()):
        raise RuntimeError(
            "No model's dataset produced a valid feature matrix. Re-check Cell 2's acquisition results."
        )

    summary_path = os.path.join(_REPORTS_ROOT, "preprocessing_summary.json")
    preprocessing_information: Dict[str, Any] = {
        "preprocessing_version": PREP_CONFIG["version"],
        "pipeline_signature": PIPELINE_SIGNATURE,
        "experiment_uuid": EXPERIMENT_UUID,
        "config_hash": CONFIG_HASH,
        "scope": "multi_model",
        "completed_at_utc": utc_now_iso(),
        "processing_seconds": round(time.perf_counter() - pipeline_started, 4),
        "text": results["text"],
        "behavioral": results["behavioral"],
        "url": results["url"],
        "web_content": results["web_content"],
        "available_models": [k for k, v in results.items() if v is not None],
        "unavailable_models": [k for k, v in results.items() if v is None],
        "label_mapping": {"legitimate": 0, "phishing": 1},
        "split_mapping": {"train": 0, "validation": 1, "test": 2},
        "methodology_guards": {
            "object_arrays_saved": False,
            "npz_verified_with_allow_pickle_false": True,
            "vectorizers_scalers_fitted_on_train_only": True,
            "test_data_used_for_fitting": False,
            "each_model_has_independent_split": True,
            "reply_to_mismatch_computed": False,
            "spf_dkim_dmarc_computed": False,
            "fabricated_metadata": False,
            "web_content_feature_type": "handcrafted_html_dom (not CNN/Transformer/graph embedding)",
        },
        "paths": CELL3_PATHS,
        "resources_before": resource_before,
        "resources_after": (
            get_system_resource_snapshot() if "get_system_resource_snapshot" in globals() else {}
        ),
        "report_path": summary_path,
    }

    atomic_json_save(preprocessing_information, summary_path)
    return preprocessing_information


# ----------------------------------------------------------------------------
# 11. EXECUTE CELL 3
# ----------------------------------------------------------------------------

try:
    PREPROCESSING_INFO: Dict[str, Any] = run_preprocessing_pipeline()

    write_status(
        "preprocessing_pipeline_complete", STATUS_PATH,
        extra={
            "cell": 3, "pipeline_signature": PIPELINE_SIGNATURE, "scope": "multi_model",
            "available_models": PREPROCESSING_INFO["available_models"],
            "unavailable_models": PREPROCESSING_INFO["unavailable_models"],
            "summary_path": PREPROCESSING_INFO["report_path"],
        },
    )

except Exception as exc:
    error_trace = traceback.format_exc()
    LOGGER.error("Cell 3 failed: %s\n%s", exc, error_trace)
    write_status(
        "preprocessing_pipeline_error", STATUS_PATH,
        extra={"cell": 3, "error": str(exc), "traceback": error_trace}, is_error=True,
    )
    raise


# ----------------------------------------------------------------------------
# 12. FINAL SUMMARY
# ----------------------------------------------------------------------------

print("=" * 78)
print("MAL-PhishNet — Cell 3 Complete (Multi-Model Feature Engineering)".center(78))
print("=" * 78)
print(f"Pipeline signature   : {PREPROCESSING_INFO['pipeline_signature'][:16]}...")
print(f"Available models     : {PREPROCESSING_INFO['available_models']}")
print(f"Unavailable models   : {PREPROCESSING_INFO['unavailable_models']}")

for _model_name in MODEL_NAMES:
    _info = PREPROCESSING_INFO.get(_model_name)
    print("-" * 78)
    if not _info:
        print(f"{_model_name.upper()} MODEL: SKIPPED (dataset unavailable or build failed)")
        continue
    print(f"{_model_name.upper()} MODEL")
    print(f"  Samples          : {_info['sample_count']}")
    print(f"  Classes          : {_info['class_distribution']}")
    print(f"  Split counts     : {_info['split_counts']}")
    print(f"  Feature count    : {len(_info['feature_names'])}")
    print(f"  Input NPZ        : {_info['input_path']}")
    if _model_name == "web_content":
        print(f"  HTML available   : {_info.get('html_available_count')}")

print("-" * 78)
print(f"Summary report       : {PREPROCESSING_INFO['report_path']}")
print(f"Processing seconds   : {PREPROCESSING_INFO['processing_seconds']}")
print("=" * 78)

[2026-08-15 15:03:55] [INFO    ] [MAL-PhishNet] ==============================================================================
[2026-08-15 15:03:55] [INFO    ] [MAL-PhishNet]             MAL-PhishNet — Cell 3: Multi-Model Feature Engineering            
[2026-08-15 15:03:55] [INFO    ] [MAL-PhishNet] ==============================================================================


/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


[2026-08-15 15:03:57] [INFO    ] [MAL-PhishNet] --- Building text model features ---
[2026-08-15 15:04:00] [INFO    ] [MAL-PhishNet] [Text Model] Cleaning and tokenizing email text...


Text cleaning:   0%|          | 0/4284 [00:00<?, ?it/s]

[2026-08-15 15:04:30] [INFO    ] [MAL-PhishNet] [Text Model] Fitting TF-IDF vectorizer on training split only...
[2026-08-15 15:04:34] [INFO    ] [MAL-PhishNet] --- Building behavioral model features ---
[2026-08-15 15:04:34] [INFO    ] [MAL-PhishNet] [Behavioral Model] Extracting lexical features...


...

URL features:   0%|          | 0/40000 [00:00<?, ?it/s]

[2026-08-15 15:04:56] [INFO    ] [MAL-PhishNet] --- Building web_content model features ---
[2026-08-15 15:04:57] [INFO    ] [MAL-PhishNet] [Web Content Model] Reading and parsing HTML pages...


Web Content features:   0%|          | 0/15358 [00:00<?, ?it/s]

       MAL-PhishNet — Cell 3 Complete (Multi-Model Feature Engineering)       
Pipeline signature   : 23b4b011c654b650...
Available models     : ['text', 'behavioral', 'url', 'web_content']
Unavailable models   : []
------------------------------------------------------------------------------
TEXT MODEL
  Samples          : 4284
  Classes          : {0: 2720, 1: 1564}
  Split counts     : {'train': {0: 1903, 1: 1095}, 'validation': {0: 408, 1: 235}, 'test': {0: 409, 1: 234}}
  Feature count    : 500
  Input NPZ        : /content/drive/MyDrive/MAL-PhishNet/datasets/processed/preprocessing/23b4b011c654b650/text/sequences/text_model_inputs.npz
------------------------------------------------------------------------------
BEHAVIORAL MODEL
  Samples          : 4284
  Classes          : {0: 2720, 1: 1564}
  Split counts     : {'train': {0: 1903, 1: 1095}, 'validation': {0: 408, 1: 235}, 'test': {0: 409, 1: 234}}
  Feature count    : 32
  Input NPZ        : /content/drive/MyDrive/MAL-PhishNe

/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


In [ ]:
# ============================================================================
# MAL-PhishNet — Cell 4: Multi-Model Architecture Factory
# (Text / Behavioral / URL / Web Content — four independent classifiers)
# ============================================================================
# Run updated Cell 3 (multi-model) before this cell.
#
# ARCHITECTURE CHANGE: builds FOUR independent, unfitted classical
# estimators — one per model — each selectable via config
# (Logistic Regression / Random Forest / XGBoost, per your supervisor's
# spec). Each model is built ONLY if Cell 3 produced a valid feature
# matrix for it; models with unavailable datasets are skipped gracefully.
#
# Every model gets its own smoke test (sanity-fit on a tiny subset of its
# OWN feature matrix) — no cross-model coupling anywhere in this cell.
# The Cloud Fusion Layer (built in Cell 6) consumes only the prediction
# PROBABILITIES these four models produce after training — it does not
# see or share any features, and nothing here builds the fusion model
# itself.
#
# Outputs:
#   TEXT_MODEL, BEHAVIORAL_MODEL, URL_MODEL, WEB_CONTENT_MODEL
#     (each an unfitted estimator, or None if its dataset was unavailable)
#   MALPHISHNET_MODELS   ({"text": ..., "behavioral": ..., "url": ...,
#                          "web_content": ...} — None entries omitted)
#   MODEL_CONFIG
#   MODEL_SIGNATURE
#   MODEL_INFO
# ============================================================================


# ----------------------------------------------------------------------------
# 0. IMPORTS AND DEPENDENCY CHECK
# ----------------------------------------------------------------------------

import os
import json
import time
import hashlib
import traceback

from datetime import datetime, timezone
from typing import Any, Dict, List, Mapping, Optional, Tuple

import numpy as np


_REQUIRED_GLOBALS = [
    "CONFIG", "CONFIG_HASH", "EXPERIMENT_UUID", "PATHS", "LOGGER",
    "STATUS_PATH", "write_status", "PREPROCESSING_INFO",
    "ensure_packages_installed",
]
_MISSING_GLOBALS = [name for name in _REQUIRED_GLOBALS if name not in globals()]
if _MISSING_GLOBALS:
    raise RuntimeError(f"Cell 4 requires updated Cells 1-3 first. Missing objects: {_MISSING_GLOBALS}")

if not PREPROCESSING_INFO.get("available_models"):
    raise RuntimeError(
        "PREPROCESSING_INFO reports no available models. Re-run Cell 3 (multi-model) first."
    )


_CELL4_PACKAGES = {"scikit-learn": "sklearn", "joblib": "joblib", "xgboost": "xgboost"}
_PACKAGE_RESULTS = ensure_packages_installed(_CELL4_PACKAGES)
_FAILED_REQUIRED_PACKAGES = {
    package: result for package, result in _PACKAGE_RESULTS.items()
    if package != "xgboost" and str(result).startswith("failed")
}
if _FAILED_REQUIRED_PACKAGES:
    raise RuntimeError(
        f"Cell 4 required package installation failed:\n{json.dumps(_FAILED_REQUIRED_PACKAGES, indent=2)}"
    )

_XGBOOST_AVAILABLE = not str(_PACKAGE_RESULTS.get("xgboost", "")).startswith("failed")

import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

if _XGBOOST_AVAILABLE:
    try:
        from xgboost import XGBClassifier
    except Exception:
        _XGBOOST_AVAILABLE = False


write_status("model_architecture_start", STATUS_PATH, extra={"cell": 4, "scope": "multi_model"})

LOGGER.info("=" * 78)
LOGGER.info("MAL-PhishNet — Cell 4: Multi-Model Architecture Factory".center(78))
LOGGER.info("=" * 78)

if not _XGBOOST_AVAILABLE:
    LOGGER.warning(
        "xgboost could not be installed/imported; the 'xgboost' algorithm option is "
        "unavailable this run for ANY model. logistic_regression and random_forest remain available."
    )


# ----------------------------------------------------------------------------
# 1. LOCAL MODEL CONFIGURATION
# ----------------------------------------------------------------------------
# Cell-1 CONFIG is not modified because CONFIG_HASH has already been generated.
# ----------------------------------------------------------------------------

MODEL_NAMES = ["text", "behavioral", "url", "web_content"]

_DEFAULT_MODEL_CONFIG: Dict[str, Any] = {
    "version": "4.4.0-multi-model",
    "num_classes": 2,
    "class_weight_strategy": "balanced",

    # Per-model algorithm choice — independent, since each model is fully
    # independent per the supervisor's architecture.
    "algorithms": {
        "text": "random_forest",
        "behavioral": "random_forest",
        "url": "random_forest",
        "web_content": "random_forest",
    },

    # Shared hyperparameter sets PER ALGORITHM TYPE, reused by whichever
    # models select that algorithm. Not shared feature spaces or fitted
    # state — just hyperparameter defaults.
    "logistic_regression": {"C": 1.0, "max_iter": 2000, "solver": "lbfgs", "penalty": "l2"},
    "random_forest": {
        "n_estimators": 400, "max_depth": None, "min_samples_leaf": 2,
        "min_samples_split": 4, "max_features": "sqrt", "n_jobs": -1,
    },
    "xgboost": {
        "n_estimators": 400, "max_depth": 6, "learning_rate": 0.05,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0,
        "eval_metric": "logloss", "n_jobs": -1, "tree_method": "hist",
    },
}

MODEL_CONFIG: Dict[str, Any] = {
    **_DEFAULT_MODEL_CONFIG,
    **dict(CONFIG.get("model_architecture", {})),
}

_SUPPORTED_ALGORITHMS = {"logistic_regression", "random_forest", "xgboost"}

for _model_name in MODEL_NAMES:
    _selected_algorithm = MODEL_CONFIG["algorithms"].get(_model_name, "random_forest")
    if _selected_algorithm not in _SUPPORTED_ALGORITHMS:
        raise ValueError(
            f"Unsupported algorithm '{_selected_algorithm}' for model '{_model_name}'. "
            f"Must be one of: {sorted(_SUPPORTED_ALGORITHMS)}"
        )
    if _selected_algorithm == "xgboost" and not _XGBOOST_AVAILABLE:
        LOGGER.warning(
            "[%s] Requested 'xgboost' but it is unavailable; falling back to 'random_forest' for this model.",
            _model_name,
        )
        MODEL_CONFIG["algorithms"][_model_name] = "random_forest"

if int(MODEL_CONFIG["num_classes"]) != 2:
    raise ValueError("MAL-PhishNet currently expects exactly two classes.")


# ----------------------------------------------------------------------------
# 2. PATHS AND MODEL SIGNATURE
# ----------------------------------------------------------------------------

_MODEL_ROOT = PATHS.get("models", os.path.join(PATHS["root"], "models"))

CELL4_PATHS: Dict[str, str] = {
    "architecture": os.path.join(_MODEL_ROOT, "architecture"),
    "initial_states": os.path.join(_MODEL_ROOT, "initial_states"),
    "reports": os.path.join(PATHS["outputs_metrics"], "model_architecture"),
}
for _directory in CELL4_PATHS.values():
    os.makedirs(_directory, exist_ok=True)


_MODEL_SIGNATURE_PAYLOAD = {
    "config_hash": CONFIG_HASH,
    "preprocessing_signature": PREPROCESSING_INFO.get("pipeline_signature"),
    "model_config": MODEL_CONFIG,
    "available_models": PREPROCESSING_INFO.get("available_models"),
    "scope": "multi_model",
}
MODEL_SIGNATURE = hashlib.sha256(
    json.dumps(_MODEL_SIGNATURE_PAYLOAD, sort_keys=True, default=str).encode("utf-8")
).hexdigest()


def utc_now_iso() -> str:
    """Return timezone-aware UTC timestamp."""
    return datetime.now(timezone.utc).isoformat()


def atomic_json_save(payload: Any, destination: str) -> None:
    """Save JSON atomically."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temporary_path = destination + ".tmp"
    with open(temporary_path, "w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, indent=2, sort_keys=True, default=str)
        file_handle.flush()
        os.fsync(file_handle.fileno())
    os.replace(temporary_path, destination)


def atomic_joblib_save(payload: Any, destination: str) -> None:
    """Save a joblib artifact atomically."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temporary_path = destination + ".tmp"
    joblib.dump(payload, temporary_path)
    os.replace(temporary_path, destination)


# ----------------------------------------------------------------------------
# 3. FEATURE MATRIX LOADING (safe, allow_pickle=False)
# ----------------------------------------------------------------------------

def load_model_feature_arrays(npz_path: str) -> Dict[str, np.ndarray]:
    """
    Load one model's safe feature-matrix NPZ (produced by Cell 3).

    Args:
        npz_path: Path to <model>_model_inputs.npz.

    Returns:
        Dictionary of sample_ids, labels, split_codes, features, feature_names.
    """
    if not os.path.isfile(npz_path):
        raise FileNotFoundError(f"Model input file not found: {npz_path}")
    with np.load(npz_path, allow_pickle=False) as archive:
        required_arrays = {"sample_ids", "labels", "split_codes", "features", "feature_names"}
        missing_arrays = required_arrays - set(archive.files)
        if missing_arrays:
            raise ValueError(f"NPZ is missing arrays: {sorted(missing_arrays)}")
        return {key: archive[key].copy() for key in archive.files}


def compute_class_weights(labels: np.ndarray) -> Dict[int, float]:
    """
    Compute balanced class weights for logging/inspection (sklearn's
    class_weight='balanced' computes the equivalent internally at fit time).

    Args:
        labels: 1D integer array of 0/1 labels.

    Returns:
        Mapping of class label -> weight.
    """
    labels = np.asarray(labels, dtype=np.int64)
    counts = np.bincount(labels, minlength=2).astype(np.float64)
    if np.any(counts == 0):
        raise RuntimeError(f"Training split must contain both classes. Counts: {counts.tolist()}")
    weights = labels.size / (2.0 * counts)
    return {0: float(weights[0]), 1: float(weights[1])}


# ----------------------------------------------------------------------------
# 4. MODEL FACTORY (algorithm-agnostic, reused across all four models)
# ----------------------------------------------------------------------------

def build_classifier(algorithm: str, model_config: Mapping[str, Any], seed: int) -> Any:
    """
    Construct an UNFITTED classical estimator.

    Args:
        algorithm: One of "logistic_regression", "random_forest", "xgboost".
        model_config: The MODEL_CONFIG dictionary (per-algorithm hyperparameters).
        seed: Random seed for reproducibility.

    Returns:
        An unfitted scikit-learn-compatible estimator instance.

    Raises:
        ValueError: If algorithm is not one of the supported options.
    """
    class_weight_strategy = model_config.get("class_weight_strategy", "balanced")

    if algorithm == "logistic_regression":
        params = dict(model_config["logistic_regression"])
        return LogisticRegression(
            C=float(params["C"]), max_iter=int(params["max_iter"]),
            solver=str(params["solver"]), penalty=str(params["penalty"]),
            class_weight=class_weight_strategy, random_state=seed,
        )

    if algorithm == "random_forest":
        params = dict(model_config["random_forest"])
        return RandomForestClassifier(
            n_estimators=int(params["n_estimators"]), max_depth=params["max_depth"],
            min_samples_leaf=int(params["min_samples_leaf"]), min_samples_split=int(params["min_samples_split"]),
            max_features=params["max_features"], n_jobs=int(params["n_jobs"]),
            class_weight=class_weight_strategy, random_state=seed,
        )

    if algorithm == "xgboost":
        if not _XGBOOST_AVAILABLE:
            raise ValueError("xgboost was requested but is not available in this environment.")
        params = dict(model_config["xgboost"])
        return XGBClassifier(
            n_estimators=int(params["n_estimators"]), max_depth=int(params["max_depth"]),
            learning_rate=float(params["learning_rate"]), subsample=float(params["subsample"]),
            colsample_bytree=float(params["colsample_bytree"]), reg_lambda=float(params["reg_lambda"]),
            eval_metric=str(params["eval_metric"]), n_jobs=int(params["n_jobs"]),
            tree_method=str(params["tree_method"]), random_state=seed, use_label_encoder=False,
        )

    raise ValueError(f"Unknown algorithm: {algorithm}")


def supports_feature_importance(algorithm: str) -> bool:
    """
    Return True if the given algorithm exposes a feature-importance signal
    once fitted (feature_importances_ for tree ensembles, coef_ for linear
    models). Determined by algorithm choice, not by probing an unfitted
    model instance's attributes (which would return False even for
    algorithms that WILL expose it after .fit()).
    """
    return algorithm in _SUPPORTED_ALGORITHMS


# ----------------------------------------------------------------------------
# 5. SMOKE TEST (sanity-fit on a tiny subset, per model)
# ----------------------------------------------------------------------------

def smoke_test_model(
    model_name: str, algorithm: str, model_config: Mapping[str, Any],
    features: np.ndarray, labels: np.ndarray, seed: int,
) -> Dict[str, Any]:
    """
    Fit a freshly constructed model instance on a small random subset of
    ONE model's own feature matrix, to confirm the estimator/config
    combination is wired correctly, without touching the real training run
    (Cell 5 performs the actual fit on the full training split).

    Args:
        model_name: One of "text", "behavioral", "url", "web_content".
        algorithm: Selected algorithm name for this model.
        model_config: MODEL_CONFIG dictionary.
        features: Full feature matrix for this model (N, D).
        labels: Full label vector for this model (N,).
        seed: Random seed.

    Returns:
        Dictionary describing smoke test outcome.

    Raises:
        RuntimeError: If the sanity fit or predict fails.
    """
    rng = np.random.default_rng(seed)
    class_0_indices = np.flatnonzero(labels == 0)
    class_1_indices = np.flatnonzero(labels == 1)

    sample_size_per_class = min(20, class_0_indices.size, class_1_indices.size)
    if sample_size_per_class < 2:
        raise RuntimeError(f"[{model_name}] Not enough samples per class to run the smoke test.")

    selected_indices = np.concatenate([
        rng.choice(class_0_indices, size=sample_size_per_class, replace=False),
        rng.choice(class_1_indices, size=sample_size_per_class, replace=False),
    ])

    smoke_model = build_classifier(algorithm, model_config, seed)
    try:
        smoke_model.fit(features[selected_indices], labels[selected_indices])
        predictions = smoke_model.predict(features[selected_indices])
        probabilities = smoke_model.predict_proba(features[selected_indices])
    except Exception as exc:
        raise RuntimeError(f"[{model_name}] Smoke test failed for algorithm '{algorithm}': {exc}") from exc

    if tuple(predictions.shape) != (len(selected_indices),):
        raise RuntimeError(f"[{model_name}] Smoke test failed: incorrect prediction shape.")
    if probabilities.shape != (len(selected_indices), 2):
        raise RuntimeError(f"[{model_name}] Smoke test failed: incorrect predict_proba shape.")

    return {
        "passed": True, "model": model_name, "algorithm": algorithm,
        "smoke_sample_count": int(len(selected_indices)),
        "prediction_shape": list(predictions.shape), "probability_shape": list(probabilities.shape),
    }


# ----------------------------------------------------------------------------
# 6. PER-MODEL BUILD FUNCTION
# ----------------------------------------------------------------------------

def build_one_model(model_name: str) -> Tuple[Optional[Any], Optional[Dict[str, Any]]]:
    """
    Build one model's unfitted estimator and metadata, or return (None, None)
    if its Cell 3 feature matrix is unavailable.

    Args:
        model_name: One of "text", "behavioral", "url", "web_content".

    Returns:
        Tuple of (unfitted estimator or None, model information dict or None).
    """
    model_preprocessing_info = PREPROCESSING_INFO.get(model_name)
    if not model_preprocessing_info:
        LOGGER.warning("[%s] No preprocessing information available; skipping model build.", model_name)
        return None, None

    seed = int(CONFIG.get("seed", 42))
    arrays = load_model_feature_arrays(model_preprocessing_info["input_path"])
    features = arrays["features"]
    labels = arrays["labels"]
    split_codes = arrays["split_codes"]
    feature_names = [str(n) for n in arrays["feature_names"]]

    training_mask = split_codes == 0
    training_labels = labels[training_mask]
    class_weights = compute_class_weights(training_labels)

    algorithm = str(MODEL_CONFIG["algorithms"][model_name])
    model = build_classifier(algorithm, MODEL_CONFIG, seed)

    model_information: Dict[str, Any] = {
        "class_name": model.__class__.__name__,
        "algorithm": algorithm,
        "hyperparameters": dict(MODEL_CONFIG.get(algorithm, {})),
        "class_weight_strategy": MODEL_CONFIG["class_weight_strategy"],
        "training_class_weights": class_weights,
        "input_path": model_preprocessing_info["input_path"],
        "feature_dimensions": int(features.shape[1]),
        "feature_names": feature_names,
        "sample_count": int(features.shape[0]),
        "supports_feature_importance": supports_feature_importance(algorithm),
    }
    return model, model_information


# ----------------------------------------------------------------------------
# 7. BUILD, TEST AND SAVE ALL FOUR MODELS
# ----------------------------------------------------------------------------

try:
    _cell4_started = time.perf_counter()

    _MODELS: Dict[str, Optional[Any]] = {}
    MODEL_INFO: Dict[str, Any] = {
        "model_version": MODEL_CONFIG["version"],
        "model_signature": MODEL_SIGNATURE,
        "preprocessing_signature": PREPROCESSING_INFO.get("pipeline_signature"),
        "experiment_uuid": EXPERIMENT_UUID,
        "config_hash": CONFIG_HASH,
        "created_at_utc": utc_now_iso(),
        "scope": "multi_model",
        "models": {},
        "smoke_tests": {},
        "initial_state_paths": {},
    }

    for _model_name in MODEL_NAMES:
        _model, _model_information = build_one_model(_model_name)
        _MODELS[_model_name] = _model

        if _model is None:
            MODEL_INFO["models"][_model_name] = None
            MODEL_INFO["smoke_tests"][_model_name] = None
            MODEL_INFO["initial_state_paths"][_model_name] = None
            continue

        MODEL_INFO["models"][_model_name] = _model_information

        _arrays_for_smoke_test = load_model_feature_arrays(_model_information["input_path"])
        MODEL_INFO["smoke_tests"][_model_name] = smoke_test_model(
            model_name=_model_name,
            algorithm=_model_information["algorithm"],
            model_config=MODEL_CONFIG,
            features=_arrays_for_smoke_test["features"],
            labels=_arrays_for_smoke_test["labels"],
            seed=int(CONFIG.get("seed", 42)),
        )
        del _arrays_for_smoke_test

        _state_path = os.path.join(
            CELL4_PATHS["initial_states"], f"{_model_name}_unfitted_{MODEL_SIGNATURE[:12]}.joblib"
        )
        atomic_joblib_save(
            {
                "model": _model, "model_name": _model_name, "model_signature": MODEL_SIGNATURE,
                "preprocessing_signature": PREPROCESSING_INFO.get("pipeline_signature"),
                "model_config": MODEL_CONFIG, "created_at_utc": utc_now_iso(), "fitted": False,
            },
            _state_path,
        )
        MODEL_INFO["initial_state_paths"][_model_name] = _state_path

    TEXT_MODEL = _MODELS["text"]
    BEHAVIORAL_MODEL = _MODELS["behavioral"]
    URL_MODEL = _MODELS["url"]
    WEB_CONTENT_MODEL = _MODELS["web_content"]

    MALPHISHNET_MODELS: Dict[str, Any] = {
        name: model for name, model in _MODELS.items() if model is not None
    }

    if not MALPHISHNET_MODELS:
        raise RuntimeError("No model could be built — all four models had unavailable feature matrices.")

    MODEL_INFO["available_models"] = list(MALPHISHNET_MODELS.keys())
    MODEL_INFO["unavailable_models"] = [n for n in MODEL_NAMES if n not in MALPHISHNET_MODELS]

    MODEL_INFO["architecture_note"] = (
        "Four fully independent models (Text, Behavioral, URL, Web Content), each with its own "
        "dataset, feature engineering, and split. The Cloud Fusion Layer (Cell 6) consumes only "
        "each model's prediction probabilities — no features or model internals are shared "
        "across models at any point in this pipeline."
    )

    MODEL_INFO["cell4_seconds"] = round(time.perf_counter() - _cell4_started, 4)

    architecture_summary_path = os.path.join(CELL4_PATHS["reports"], "model_architecture_summary.json")
    MODEL_INFO["architecture_summary_path"] = architecture_summary_path
    atomic_json_save(MODEL_INFO, architecture_summary_path)

    write_status(
        "model_architecture_complete", STATUS_PATH,
        extra={
            "cell": 4, "model_signature": MODEL_SIGNATURE,
            "available_models": MODEL_INFO["available_models"],
            "unavailable_models": MODEL_INFO["unavailable_models"],
            "summary_path": architecture_summary_path,
        },
    )

except Exception as exc:
    error_trace = traceback.format_exc()
    LOGGER.error("Cell 4 failed: %s\n%s", exc, error_trace)
    write_status(
        "model_architecture_error", STATUS_PATH,
        extra={"cell": 4, "error": str(exc), "traceback": error_trace}, is_error=True,
    )
    raise


# ----------------------------------------------------------------------------
# 8. FINAL SUMMARY
# ----------------------------------------------------------------------------

print("=" * 78)
print("MAL-PhishNet — Cell 4 Complete (Multi-Model Architecture Factory)".center(78))
print("=" * 78)

print(f"Model signature   : {MODEL_SIGNATURE[:16]}...")
print(f"Available models  : {MODEL_INFO['available_models']}")
print(f"Unavailable models: {MODEL_INFO['unavailable_models']}")

for _model_name in MODEL_NAMES:
    _info = MODEL_INFO["models"].get(_model_name)
    _smoke = MODEL_INFO["smoke_tests"].get(_model_name)
    print("-" * 78)
    if not _info:
        print(f"{_model_name.upper()} MODEL: SKIPPED")
        continue
    print(f"{_model_name.upper()} MODEL")
    print(f"  Algorithm                  : {_info['algorithm']}")
    print(f"  Sample count               : {_info['sample_count']}")
    print(f"  Feature dimensions         : {_info['feature_dimensions']}")
    print(f"  Training class weights     : {_info['training_class_weights']}")
    print(f"  Supports feature importance: {_info['supports_feature_importance']}")
    print(f"  Smoke test passed          : {_smoke['passed'] if _smoke else 'N/A'}")

print("-" * 78)
print(f"Architecture summary  : {MODEL_INFO['architecture_summary_path']}")
print(f"Cell 4 seconds        : {MODEL_INFO['cell4_seconds']}")
print("=" * 78)

[2026-08-15 16:48:54] [INFO    ] [MAL-PhishNet] ==============================================================================
[2026-08-15 16:48:54] [INFO    ] [MAL-PhishNet]            MAL-PhishNet — Cell 4: Multi-Model Architecture Factory            
[2026-08-15 16:48:54] [INFO    ] [MAL-PhishNet] ==============================================================================


/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


      MAL-PhishNet — Cell 4 Complete (Multi-Model Architecture Factory)       
Model signature   : cf7e6c61c105ea01...
Available models  : ['text', 'behavioral', 'url', 'web_content']
Unavailable models: []
------------------------------------------------------------------------------
TEXT MODEL
  Algorithm                  : random_forest
  Sample count               : 4284
  Feature dimensions         : 500
  Training class weights     : {0: 0.7877036258539148, 1: 1.3689497716894976}
  Supports feature importance: True
  Smoke test passed          : True
------------------------------------------------------------------------------
BEHAVIORAL MODEL
  Algorithm                  : random_forest
  Sample count               : 4284
  Feature dimensions         : 32
  Training class weights     : {0: 0.7877036258539148, 1: 1.3689497716894976}
  Supports feature importance: True
  Smoke test passed          : True
----------------------------------------------------------------------------

/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


In [ ]:
# ============================================================================
# MAL-PhishNet — Cell 5: Multi-Model Training
# (Text / Behavioral / URL / Web Content — four independent fits)
# ============================================================================
# Run updated Cells 1-4 (multi-model) before this cell.
#
# ARCHITECTURE CHANGE: fits FOUR independent classical estimators, each on
# its OWN training split, each with its OWN validation-based stability
# check. No feature sharing, no joint optimization, no shared loss —
# exactly as required ("each model must have its own training, validation,
# testing"). Models with unavailable Cell 4 estimators are skipped.
#
# As with the single-model version, fitting a Random Forest/XGBoost/
# Logistic Regression on these dataset sizes (thousands to tens of
# thousands of rows) takes seconds, not hours — so there is no epoch-level
# resume state. The resumability that matters is whole-fit-level: if a
# fitted model matching the current signature already exists on disk for
# a given task, it is reloaded rather than refit.
#
# Outputs required by Cell 6:
#   TEXT_TRAINING_RESULT, BEHAVIORAL_TRAINING_RESULT,
#   URL_TRAINING_RESULT, WEB_CONTENT_TRAINING_RESULT
#   TRAINING_SIGNATURE
#   TRAINING_INFO
#   load_model_arrays_for_split   (helper reused by Cell 6/7)
# ============================================================================


# ----------------------------------------------------------------------------
# 0. IMPORTS AND DEPENDENCY CHECK
# ----------------------------------------------------------------------------

import os
import gc
import json
import time
import hashlib
import traceback

from datetime import datetime, timezone
from typing import Any, Dict, List, Mapping, Optional, Tuple

import numpy as np
import pandas as pd
import joblib


_REQUIRED_GLOBALS = [
    "CONFIG", "CONFIG_HASH", "EXPERIMENT_UUID", "PATHS", "LOGGER",
    "STATUS_PATH", "write_status", "PREPROCESSING_INFO", "MODEL_INFO",
    "MODEL_SIGNATURE", "MODEL_CONFIG", "MALPHISHNET_MODELS",
]
_MISSING_GLOBALS = [name for name in _REQUIRED_GLOBALS if name not in globals()]
if _MISSING_GLOBALS:
    raise RuntimeError(f"Cell 5 requires updated Cells 1-4 first. Missing objects: {_MISSING_GLOBALS}")

if not MALPHISHNET_MODELS:
    raise RuntimeError("MALPHISHNET_MODELS is empty. Re-run Cell 4 (multi-model) first.")


write_status(
    "training_pipeline_start", STATUS_PATH,
    extra={"cell": 5, "model_signature": MODEL_SIGNATURE, "scope": "multi_model"},
)

LOGGER.info("=" * 78)
LOGGER.info("MAL-PhishNet — Cell 5: Multi-Model Training".center(78))
LOGGER.info("=" * 78)


# ----------------------------------------------------------------------------
# 1. LOCAL TRAINING CONFIGURATION
# ----------------------------------------------------------------------------

MODEL_NAMES = ["text", "behavioral", "url", "web_content"]

_DEFAULT_TRAIN_CONFIG: Dict[str, Any] = {
    "version": "5.4.0-multi-model",

    "train_models": {"text": True, "behavioral": True, "url": True, "web_content": True},

    "resume": True,
    "force_restart": False,

    # Number of stratified re-fits (different seeds) used to compute a
    # stability estimate on each model's own validation split.
    "stability_refits": 3,

    "monitor_metric": "f1",
}

TRAIN_CONFIG: Dict[str, Any] = {
    **_DEFAULT_TRAIN_CONFIG,
    **dict(CONFIG.get("training", {})),
}

SEED = int(CONFIG.get("seed", 42))


# ----------------------------------------------------------------------------
# 2. OUTPUT PATHS AND TRAINING SIGNATURE
# ----------------------------------------------------------------------------

_MODEL_ROOT = PATHS.get("models", os.path.join(PATHS["root"], "models"))

CELL5_PATHS: Dict[str, Dict[str, str]] = {}
for _model_name in MODEL_NAMES:
    CELL5_PATHS[_model_name] = {
        "checkpoints": os.path.join(_MODEL_ROOT, "training", MODEL_SIGNATURE[:16], _model_name),
        "histories": os.path.join(
            PATHS["outputs_metrics"], "training", MODEL_SIGNATURE[:16], _model_name, "histories"
        ),
        "validation_predictions": os.path.join(
            PATHS["outputs_metrics"], "training", MODEL_SIGNATURE[:16], _model_name, "validation_predictions"
        ),
    }
    for _path in CELL5_PATHS[_model_name].values():
        os.makedirs(_path, exist_ok=True)

_REPORTS_ROOT = os.path.join(PATHS["outputs_metrics"], "training", MODEL_SIGNATURE[:16], "reports")
os.makedirs(_REPORTS_ROOT, exist_ok=True)


_TRAINING_SIGNATURE_PAYLOAD = {
    "config_hash": CONFIG_HASH,
    "preprocessing_signature": PREPROCESSING_INFO.get("pipeline_signature"),
    "model_signature": MODEL_SIGNATURE,
    "training_config": TRAIN_CONFIG,
}
TRAINING_SIGNATURE = hashlib.sha256(
    json.dumps(_TRAINING_SIGNATURE_PAYLOAD, sort_keys=True, default=str).encode("utf-8")
).hexdigest()


# ----------------------------------------------------------------------------
# 3. GENERAL UTILITIES
# ----------------------------------------------------------------------------

def utc_now_iso() -> str:
    """Return timezone-aware UTC timestamp."""
    return datetime.now(timezone.utc).isoformat()


def atomic_json_save(payload: Any, destination: str) -> None:
    """Save JSON atomically."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temporary_path = destination + ".tmp"
    with open(temporary_path, "w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, indent=2, sort_keys=True, default=str)
        file_handle.flush()
        os.fsync(file_handle.fileno())
    os.replace(temporary_path, destination)


def atomic_csv_save(dataframe: pd.DataFrame, destination: str) -> None:
    """Save CSV atomically."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temporary_path = destination + ".tmp"
    dataframe.to_csv(temporary_path, index=False, encoding="utf-8")
    os.replace(temporary_path, destination)


def atomic_joblib_save(payload: Any, destination: str) -> None:
    """Save a joblib artifact atomically."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temporary_path = destination + ".tmp"
    joblib.dump(payload, temporary_path)
    os.replace(temporary_path, destination)


def load_model_feature_arrays(npz_path: str) -> Dict[str, np.ndarray]:
    """Load one model's safe feature-matrix NPZ (allow_pickle=False)."""
    if not os.path.isfile(npz_path):
        raise FileNotFoundError(f"Model input file not found: {npz_path}")
    with np.load(npz_path, allow_pickle=False) as archive:
        required_arrays = {"sample_ids", "labels", "split_codes", "features", "feature_names"}
        missing_arrays = required_arrays - set(archive.files)
        if missing_arrays:
            raise ValueError(f"NPZ is missing arrays: {sorted(missing_arrays)}")
        return {key: archive[key].copy() for key in archive.files}


def load_model_arrays_for_split(npz_path: str, split_code: int) -> Dict[str, np.ndarray]:
    """
    Load one split (train=0, validation=1, test=2) of one model's feature matrix.

    Args:
        npz_path: Path to <model>_model_inputs.npz.
        split_code: 0 for train, 1 for validation, 2 for test.

    Returns:
        Dictionary with sample_ids, labels, features restricted to that split.

    Raises:
        RuntimeError: If the resulting split is empty or lacks both classes.
    """
    arrays = load_model_feature_arrays(npz_path)
    mask = arrays["split_codes"] == int(split_code)
    labels = arrays["labels"][mask].astype(np.int64)
    if labels.size == 0:
        raise RuntimeError(f"Split code {split_code} is empty for {npz_path}.")
    classes = sorted(np.unique(labels).tolist())
    if classes != [0, 1]:
        raise RuntimeError(f"Split {split_code} must contain both classes. Classes found: {classes}")
    return {
        "sample_ids": arrays["sample_ids"][mask].astype(str),
        "labels": labels,
        "features": arrays["features"][mask].astype(np.float32),
        "feature_names": [str(n) for n in arrays["feature_names"]],
    }


def calculate_binary_metrics_from_predictions(
    labels: np.ndarray, phishing_probabilities: np.ndarray, threshold: float,
) -> Dict[str, Any]:
    """
    Compute standard binary classification metrics from labels and
    predicted phishing probabilities at a fixed threshold.

    Args:
        labels: 1D array of true 0/1 labels.
        phishing_probabilities: 1D array of predicted P(phishing).
        threshold: Decision threshold for converting probabilities to labels.

    Returns:
        Dictionary of accuracy, precision, recall, specificity, f1, mcc,
        roc_auc, pr_auc, confusion_matrix, sample_count.
    """
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        matthews_corrcoef, roc_auc_score, average_precision_score, confusion_matrix,
    )

    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(phishing_probabilities, dtype=np.float64)
    predictions = (probabilities >= threshold).astype(int)

    metrics: Dict[str, Any] = {"sample_count": int(labels.size), "threshold": float(threshold)}
    metrics["accuracy"] = float(accuracy_score(labels, predictions))
    metrics["precision"] = float(precision_score(labels, predictions, zero_division=0))
    metrics["recall"] = float(recall_score(labels, predictions, zero_division=0))
    metrics["f1"] = float(f1_score(labels, predictions, zero_division=0))

    matrix = confusion_matrix(labels, predictions, labels=[0, 1])
    true_negative, false_positive, false_negative, true_positive = matrix.ravel()
    metrics["specificity"] = (
        float(true_negative / (true_negative + false_positive))
        if (true_negative + false_positive) > 0 else None
    )
    metrics["confusion_matrix"] = matrix.tolist()

    try:
        metrics["mcc"] = float(matthews_corrcoef(labels, predictions))
    except Exception:
        metrics["mcc"] = None
    try:
        metrics["roc_auc"] = float(roc_auc_score(labels, probabilities))
    except Exception:
        metrics["roc_auc"] = None
    try:
        metrics["pr_auc"] = float(average_precision_score(labels, probabilities))
    except Exception:
        metrics["pr_auc"] = None

    return metrics


# ----------------------------------------------------------------------------
# 4. FITTED-MODEL CACHE (signature-based, per task)
# ----------------------------------------------------------------------------

def get_task_paths(model_name: str) -> Dict[str, str]:
    """Return task-specific training paths for one model."""
    checkpoint_directory = CELL5_PATHS[model_name]["checkpoints"]
    return {
        "directory": checkpoint_directory,
        "fitted_model": os.path.join(checkpoint_directory, "fitted_model.joblib"),
        "history_json": os.path.join(CELL5_PATHS[model_name]["histories"], f"{model_name}_history.json"),
        "history_csv": os.path.join(CELL5_PATHS[model_name]["histories"], f"{model_name}_history.csv"),
        "best_validation_predictions": os.path.join(
            CELL5_PATHS[model_name]["validation_predictions"],
            f"{model_name}_best_validation_predictions.csv",
        ),
    }


def load_cached_fitted_model(model_name: str, fitted_model_path: str) -> Optional[Dict[str, Any]]:
    """
    Load a previously fitted model if its stored signature matches the
    current TRAINING_SIGNATURE/MODEL_SIGNATURE/preprocessing signature.

    Args:
        model_name: One of "text", "behavioral", "url", "web_content".
        fitted_model_path: Path to the cached fitted-model joblib file.

    Returns:
        The cached payload dictionary if valid, otherwise None.
    """
    if TRAIN_CONFIG["force_restart"]:
        LOGGER.info("[%s] force_restart=True; ignoring any cached fitted model.", model_name)
        return None
    if not TRAIN_CONFIG["resume"] or not os.path.isfile(fitted_model_path):
        return None

    try:
        payload = joblib.load(fitted_model_path)
    except Exception as exc:
        LOGGER.warning("[%s] Could not load cached fitted model: %s", model_name, exc)
        return None

    expected_values = {
        "model_name": model_name, "training_signature": TRAINING_SIGNATURE,
        "model_signature": MODEL_SIGNATURE, "preprocessing_signature": PREPROCESSING_INFO.get("pipeline_signature"),
    }
    mismatches = {
        key: {"expected": expected_value, "found": payload.get(key)}
        for key, expected_value in expected_values.items() if payload.get(key) != expected_value
    }
    if mismatches:
        LOGGER.info(
            "[%s] Cached fitted model belongs to a different signature; refitting. Details: %s",
            model_name, mismatches,
        )
        return None

    LOGGER.info("[%s] Reusing cached fitted model (signature match).", model_name)
    return payload


def load_best_model_checkpoint(model_name: str, model_placeholder: Any, training_result: Mapping[str, Any]) -> Any:
    """
    Load the fitted model referenced by a training result. Provided for
    compatibility with the Cell 7 call pattern (model_name, model_placeholder,
    training_result) — loading a fitted classical estimator directly.

    Args:
        model_name: One of "text", "behavioral", "url", "web_content".
        model_placeholder: Unused (kept for call-signature compatibility).
        training_result: The training result dictionary produced below.

    Returns:
        The fitted estimator instance.
    """
    fitted_model_path = training_result["fitted_model_path"]
    payload = joblib.load(fitted_model_path)
    return payload["model"]


# ----------------------------------------------------------------------------
# 5. STABILITY ESTIMATE (multiple seeded refits on validation split)
# ----------------------------------------------------------------------------

def estimate_fit_stability(
    model_name: str, algorithm: str, model_config: Mapping[str, Any],
    train_features: np.ndarray, train_labels: np.ndarray,
    validation_features: np.ndarray, validation_labels: np.ndarray,
    number_of_refits: int, base_seed: int,
) -> Dict[str, Any]:
    """
    Fit the model multiple times with different random seeds to estimate
    how sensitive its validation F1 score is to random initialization,
    rather than trusting a single fit blindly. Fully independent per model
    — no cross-model information used.

    Args:
        model_name: One of "text", "behavioral", "url", "web_content".
        algorithm: Algorithm name.
        model_config: MODEL_CONFIG dictionary.
        train_features: Training feature matrix for this model.
        train_labels: Training labels for this model.
        validation_features: Validation feature matrix for this model.
        validation_labels: Validation labels for this model.
        number_of_refits: Number of seeded refits to run.
        base_seed: Base random seed; refit i uses base_seed + i.

    Returns:
        Dictionary with per-seed F1 scores and summary statistics.
    """
    build_classifier = globals().get("build_classifier")
    if build_classifier is None:
        LOGGER.warning("[%s] build_classifier() from Cell 4 is unavailable; skipping stability estimate.", model_name)
        return {"performed": False, "reason": "build_classifier unavailable", "f1_scores": []}

    f1_scores: List[float] = []
    for refit_index in range(int(number_of_refits)):
        seed = int(base_seed) + refit_index
        candidate_model = build_classifier(algorithm, model_config, seed)
        candidate_model.fit(train_features, train_labels)
        validation_probabilities = candidate_model.predict_proba(validation_features)[:, 1]
        metrics = calculate_binary_metrics_from_predictions(validation_labels, validation_probabilities, threshold=0.5)
        f1_scores.append(metrics["f1"])
        LOGGER.info(
            "[%s] Stability refit %d/%d (seed=%d): validation F1 = %.4f",
            model_name, refit_index + 1, number_of_refits, seed, metrics["f1"],
        )

    return {
        "performed": True, "f1_scores": [float(v) for v in f1_scores],
        "mean_f1": float(np.mean(f1_scores)) if f1_scores else None,
        "std_f1": float(np.std(f1_scores, ddof=1)) if len(f1_scores) > 1 else 0.0,
    }


# ----------------------------------------------------------------------------
# 6. TRAIN ONE TASK (reused across all four models)
# ----------------------------------------------------------------------------

def train_one_model(model_name: str, model: Any) -> Dict[str, Any]:
    """
    Fit one model on its own training split, evaluate on its own validation
    split, and persist the fitted model with full provenance metadata.
    Completely independent of any other model's data or fit.

    Args:
        model_name: One of "text", "behavioral", "url", "web_content".
        model: Unfitted estimator instance from Cell 4.

    Returns:
        Training result dictionary consumed by Cell 6/7.
    """
    task_started = time.perf_counter()
    task_paths = get_task_paths(model_name)

    model_preprocessing_info = PREPROCESSING_INFO[model_name]
    npz_path = model_preprocessing_info["input_path"]

    train_data = load_model_arrays_for_split(npz_path, split_code=0)
    validation_data = load_model_arrays_for_split(npz_path, split_code=1)

    train_class_counts = {
        int(k): int(v) for k, v in zip(*np.unique(train_data["labels"], return_counts=True))
    }
    validation_class_counts = {
        int(k): int(v) for k, v in zip(*np.unique(validation_data["labels"], return_counts=True))
    }

    LOGGER.info(
        "[%s] Train samples=%d, validation samples=%d, train classes=%s, validation classes=%s",
        model_name, len(train_data["labels"]), len(validation_data["labels"]),
        train_class_counts, validation_class_counts,
    )

    cached_payload = load_cached_fitted_model(model_name, task_paths["fitted_model"])

    if cached_payload is not None:
        fitted_model = cached_payload["model"]
        resumed = True
        history = cached_payload.get("history", [])
    else:
        algorithm = MODEL_CONFIG["algorithms"][model_name]
        LOGGER.info("[%s] Fitting %s on %d training samples...", model_name, algorithm, len(train_data["labels"]))

        fit_started = time.perf_counter()
        model.fit(train_data["features"], train_data["labels"])
        fit_seconds = round(time.perf_counter() - fit_started, 4)
        LOGGER.info("[%s] Fit completed in %.4f seconds.", model_name, fit_seconds)

        fitted_model = model
        resumed = False
        history = [{
            "event": "fit", "fit_seconds": fit_seconds, "algorithm": algorithm,
            "hyperparameters": MODEL_INFO.get("models", {}).get(model_name, {}).get("hyperparameters", {}),
            "timestamp_utc": utc_now_iso(),
        }]

    validation_probabilities = fitted_model.predict_proba(validation_data["features"])[:, 1]
    validation_metrics = calculate_binary_metrics_from_predictions(
        validation_data["labels"], validation_probabilities, threshold=0.5
    )

    LOGGER.info(
        "[%s] Validation — accuracy=%.4f, precision=%.4f, recall=%.4f, f1=%.4f, roc_auc=%s",
        model_name, validation_metrics["accuracy"], validation_metrics["precision"],
        validation_metrics["recall"], validation_metrics["f1"],
        f"{validation_metrics['roc_auc']:.4f}" if validation_metrics["roc_auc"] is not None else "N/A",
    )

    stability_result: Dict[str, Any] = {"performed": False}
    number_of_refits = int(TRAIN_CONFIG["stability_refits"])
    if not resumed and number_of_refits > 1:
        LOGGER.info("[%s] Running %d-seed stability estimate on validation split...", model_name, number_of_refits)
        stability_result = estimate_fit_stability(
            model_name=model_name, algorithm=MODEL_CONFIG["algorithms"][model_name], model_config=MODEL_CONFIG,
            train_features=train_data["features"], train_labels=train_data["labels"],
            validation_features=validation_data["features"], validation_labels=validation_data["labels"],
            number_of_refits=number_of_refits, base_seed=SEED,
        )
        if stability_result.get("performed"):
            LOGGER.info(
                "[%s] Stability estimate: mean F1=%.4f, std F1=%.4f across %d refits",
                model_name, stability_result["mean_f1"], stability_result["std_f1"], number_of_refits,
            )

    history.append({
        "event": "validation_evaluation",
        "validation_metrics": {k: v for k, v in validation_metrics.items() if k != "confusion_matrix"},
        "confusion_matrix": validation_metrics["confusion_matrix"],
        "stability": stability_result, "timestamp_utc": utc_now_iso(),
    })

    fitted_model_payload = {
        "model": fitted_model, "model_name": model_name,
        "algorithm": MODEL_CONFIG["algorithms"][model_name],
        "training_signature": TRAINING_SIGNATURE, "model_signature": MODEL_SIGNATURE,
        "preprocessing_signature": PREPROCESSING_INFO.get("pipeline_signature"),
        "experiment_uuid": EXPERIMENT_UUID, "config_hash": CONFIG_HASH,
        "feature_names": train_data["feature_names"],
        "validation_metrics": validation_metrics, "history": history, "saved_at_utc": utc_now_iso(),
    }
    atomic_joblib_save(fitted_model_payload, task_paths["fitted_model"])

    atomic_csv_save(
        pd.DataFrame({
            "sample_id": validation_data["sample_ids"].tolist(),
            "label": validation_data["labels"].tolist(),
            "phishing_probability": validation_probabilities.tolist(),
            "prediction_selected_threshold": (validation_probabilities >= 0.5).astype(int).tolist(),
        }),
        task_paths["best_validation_predictions"],
    )

    atomic_json_save(history, task_paths["history_json"])
    atomic_csv_save(
        pd.DataFrame([
            {
                "event": r.get("event"), "timestamp_utc": r.get("timestamp_utc"),
                "f1": r.get("validation_metrics", {}).get("f1"),
                "accuracy": r.get("validation_metrics", {}).get("accuracy"),
                "roc_auc": r.get("validation_metrics", {}).get("roc_auc"),
            }
            for r in history
        ]),
        task_paths["history_csv"],
    )

    result: Dict[str, Any] = {
        "task": model_name, "training_signature": TRAINING_SIGNATURE, "model_signature": MODEL_SIGNATURE,
        "preprocessing_signature": PREPROCESSING_INFO.get("pipeline_signature"),
        "resumed": bool(resumed), "algorithm": MODEL_CONFIG["algorithms"][model_name],
        "best_metric_name": str(TRAIN_CONFIG["monitor_metric"]),
        "best_metric": float(validation_metrics.get(TRAIN_CONFIG["monitor_metric"], validation_metrics["f1"])),
        "validation_metrics": validation_metrics, "stability": stability_result,
        "train_samples": int(len(train_data["labels"])), "validation_samples": int(len(validation_data["labels"])),
        "train_class_counts": train_class_counts, "validation_class_counts": validation_class_counts,
        "fitted_model_path": task_paths["fitted_model"], "history_json": task_paths["history_json"],
        "history_csv": task_paths["history_csv"],
        "best_validation_predictions": task_paths["best_validation_predictions"],
        "training_seconds": round(time.perf_counter() - task_started, 4),
    }

    del train_data, validation_data
    gc.collect()
    return result


# ----------------------------------------------------------------------------
# 7. MASTER TRAINING PIPELINE
# ----------------------------------------------------------------------------

def run_training_pipeline() -> Dict[str, Any]:
    """Train all four models independently — each has its own dataset, split, and fit."""
    training_started = time.perf_counter()
    resources_before = (
        get_system_resource_snapshot() if "get_system_resource_snapshot" in globals() else {}
    )

    results: Dict[str, Optional[Dict[str, Any]]] = {}

    for model_name in MODEL_NAMES:
        should_train = bool(TRAIN_CONFIG["train_models"].get(model_name, True))
        model = MALPHISHNET_MODELS.get(model_name)

        if should_train and model is not None and PREPROCESSING_INFO.get(model_name) is not None:
            results[model_name] = train_one_model(model_name, model)
        else:
            LOGGER.info("[%s] Training skipped (unavailable or disabled).", model_name)
            results[model_name] = None

    if not any(results.values()):
        raise RuntimeError("No model training result was produced for any of the four models.")

    summary_path = os.path.join(_REPORTS_ROOT, "training_summary.json")

    training_information: Dict[str, Any] = {
        "training_version": TRAIN_CONFIG["version"],
        "training_signature": TRAINING_SIGNATURE,
        "model_signature": MODEL_SIGNATURE,
        "preprocessing_signature": PREPROCESSING_INFO.get("pipeline_signature"),
        "experiment_uuid": EXPERIMENT_UUID,
        "config_hash": CONFIG_HASH,
        "scope": "multi_model",
        "completed_at_utc": utc_now_iso(),
        "training_seconds": round(time.perf_counter() - training_started, 4),
        "text": results["text"],
        "behavioral": results["behavioral"],
        "url": results["url"],
        "web_content": results["web_content"],
        "available_models": [k for k, v in results.items() if v is not None],
        "unavailable_models": [k for k, v in results.items() if v is None],
        "summary_path": summary_path,
        "methodology_guards": {
            "test_split_used": False,
            "test_split_used_for_training": False,
            "test_split_used_for_validation": False,
            "each_model_trained_independently": True,
            "no_cross_model_feature_sharing": True,
        },
        "resources_before": resources_before,
        "resources_after": (
            get_system_resource_snapshot() if "get_system_resource_snapshot" in globals() else {}
        ),
    }

    atomic_json_save(training_information, summary_path)
    return training_information


# ----------------------------------------------------------------------------
# 8. EXECUTE CELL 5
# ----------------------------------------------------------------------------

try:
    TRAINING_INFO: Dict[str, Any] = run_training_pipeline()

    TEXT_TRAINING_RESULT = TRAINING_INFO.get("text")
    BEHAVIORAL_TRAINING_RESULT = TRAINING_INFO.get("behavioral")
    URL_TRAINING_RESULT = TRAINING_INFO.get("url")
    WEB_CONTENT_TRAINING_RESULT = TRAINING_INFO.get("web_content")

    for _model_name, _result in [
        ("text", TEXT_TRAINING_RESULT), ("behavioral", BEHAVIORAL_TRAINING_RESULT),
        ("url", URL_TRAINING_RESULT), ("web_content", WEB_CONTENT_TRAINING_RESULT),
    ]:
        if _result is not None and MALPHISHNET_MODELS.get(_model_name) is not None:
            _fitted_payload = joblib.load(_result["fitted_model_path"])
            MALPHISHNET_MODELS[_model_name] = _fitted_payload["model"]

    write_status(
        "training_pipeline_complete", STATUS_PATH,
        extra={
            "cell": 5, "training_signature": TRAINING_SIGNATURE,
            "available_models": TRAINING_INFO["available_models"],
            "unavailable_models": TRAINING_INFO["unavailable_models"],
            "test_split_used": False, "summary_path": TRAINING_INFO["summary_path"],
        },
    )

except Exception as exc:
    error_trace = traceback.format_exc()
    LOGGER.error("Cell 5 failed: %s\n%s", exc, error_trace)
    write_status(
        "training_pipeline_error", STATUS_PATH,
        extra={"cell": 5, "error": str(exc), "traceback": error_trace}, is_error=True,
    )
    raise


# ----------------------------------------------------------------------------
# 9. FINAL SUMMARY
# ----------------------------------------------------------------------------

print("=" * 78)
print("MAL-PhishNet — Cell 5 Complete (Multi-Model Training)".center(78))
print("=" * 78)

print(f"Training signature   : {TRAINING_SIGNATURE[:16]}...")
print("Test split used      : False")
print(f"Available models     : {TRAINING_INFO['available_models']}")
print(f"Unavailable models   : {TRAINING_INFO['unavailable_models']}")

for _model_name, _result in [
    ("text", TEXT_TRAINING_RESULT), ("behavioral", BEHAVIORAL_TRAINING_RESULT),
    ("url", URL_TRAINING_RESULT), ("web_content", WEB_CONTENT_TRAINING_RESULT),
]:
    print("-" * 78)
    if not _result:
        print(f"{_model_name.upper()} MODEL: SKIPPED")
        continue
    print(f"{_model_name.upper()} MODEL")
    print(f"  Algorithm              : {_result['algorithm']}")
    print(f"  Resumed (cache hit)    : {_result['resumed']}")
    print(f"  Train samples          : {_result['train_samples']}")
    print(f"  Validation samples     : {_result['validation_samples']}")
    print(f"  Validation F1          : {_result['validation_metrics']['f1']:.4f}")
    print(f"  Validation accuracy    : {_result['validation_metrics']['accuracy']:.4f}")
    _roc = _result['validation_metrics'].get('roc_auc')
    print(f"  Validation ROC-AUC     : {_roc:.4f}" if _roc is not None else "  Validation ROC-AUC     : N/A")
    if _result["stability"].get("performed"):
        print(
            f"  Stability (mean/std F1): {_result['stability']['mean_f1']:.4f} / "
            f"{_result['stability']['std_f1']:.4f}"
        )
    print(f"  Fitted model path      : {_result['fitted_model_path']}")

print("-" * 78)
print(f"Training summary     : {TRAINING_INFO['summary_path']}")
print(f"Total training seconds: {TRAINING_INFO['training_seconds']}")
print("=" * 78)

[2026-08-15 16:48:59] [INFO    ] [MAL-PhishNet] ==============================================================================
[2026-08-15 16:48:59] [INFO    ] [MAL-PhishNet]                  MAL-PhishNet — Cell 5: Multi-Model Training                  
[2026-08-15 16:48:59] [INFO    ] [MAL-PhishNet] ==============================================================================


/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


[2026-08-15 16:49:00] [INFO    ] [MAL-PhishNet] [text] Train samples=2998, validation samples=643, train classes={0: 1903, 1: 1095}, validation classes={0: 408, 1: 235}
[2026-08-15 16:49:01] [INFO    ] [MAL-PhishNet] [text] Reusing cached fitted model (signature match).
[2026-08-15 16:49:01] [INFO    ] [MAL-PhishNet] [text] Validation — accuracy=0.9891, precision=0.9831, recall=0.9872, f1=0.9851, roc_auc=0.9990
[2026-08-15 16:49:01] [INFO    ] [MAL-PhishNet] [behavioral] Train samples=2998, validation samples=643, train classes={0: 1903, 1: 1095}, validation classes={0: 408, 1: 235}
[2026-08-15 16:49:02] [INFO    ] [MAL-PhishNet] [behavioral] Reusing cached fitted model (signature match).
[2026-08-15 16:49:02] [INFO    ] [MAL-PhishNet] [behavioral] Validation — accuracy=0.9860, precision=0.9748, recall=0.9872, f1=0.9810, roc_auc=0.9993
[2026-08-15 16:49:03] [INFO    ] [MAL-PhishNet] [url] Train samples=28000, validation samples=6000, train classes={0: 14000, 1: 14000}, validation class

/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


In [ ]:
# ============================================================================
# MAL-PhishNet — Cell 6: Multi-Model Evaluation + Cloud Fusion Layer
# ============================================================================
# Run updated Cells 1-5 (multi-model) before this cell.
#
# UPDATE (this version): the Cloud Fusion Layer's paired dataset now
# explicitly checks every live-fetched URL against the EXACT set of URLs
# used to train the URL model and the Web Content model, and EXCLUDES any
# paired sample whose URL literally appears in either base model's own
# training data. Previously this was structurally very likely true (the
# datasets are independent by construction) but was never explicitly
# verified at the individual-URL level -- this closes that gap with
# enforced code rather than an assumption, and reports the exclusion
# count transparently.
#
# PART A: Evaluates each of the four independent models (Text, Behavioral,
# URL, Web Content) on its OWN held-out test split. Threshold selected on
# validation only, frozen, applied unchanged to test.
#
# PART B: Builds the Cloud Fusion Layer using a paired multimodal test set
# drawn from the Text model's held-out test split (never its training
# split), with live-fetched landing pages so all four models can score
# the SAME underlying sample. Minority-class-aware CV selection (from the
# prior fix) is retained.
# ============================================================================


# ----------------------------------------------------------------------------
# 0. IMPORTS AND DEPENDENCY CHECK
# ----------------------------------------------------------------------------

import os
import re
import json
import time
import hashlib
import traceback

from datetime import datetime, timezone
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import joblib
import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, precision_recall_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict


_REQUIRED_GLOBALS = [
    "CONFIG", "CONFIG_HASH", "EXPERIMENT_UUID", "PATHS", "LOGGER",
    "STATUS_PATH", "write_status", "DATASET_INFO", "PREPROCESSING_INFO",
    "MODEL_INFO", "MODEL_SIGNATURE", "TRAINING_INFO", "TRAINING_SIGNATURE",
]
_MISSING_GLOBALS = [name for name in _REQUIRED_GLOBALS if name not in globals()]
if _MISSING_GLOBALS:
    raise RuntimeError(f"Cell 6 requires updated Cells 1-5 first. Missing objects: {_MISSING_GLOBALS}")

MODEL_NAMES = ["text", "behavioral", "url", "web_content"]

write_status(
    "evaluation_pipeline_start", STATUS_PATH,
    extra={"cell": 6, "training_signature": TRAINING_SIGNATURE, "scope": "multi_model"},
)

LOGGER.info("=" * 78)
LOGGER.info("MAL-PhishNet — Cell 6: Evaluation + Cloud Fusion Layer".center(78))
LOGGER.info("=" * 78)


# ----------------------------------------------------------------------------
# 1. LOCAL EVALUATION CONFIGURATION
# ----------------------------------------------------------------------------

CONFIG.setdefault("evaluation", {})
EVAL_CONFIG: Dict[str, Any] = CONFIG["evaluation"]
EVAL_CONFIG.setdefault("version", "6.7.0-multi-model-fusion-leakage-check")
EVAL_CONFIG.setdefault("threshold_metric", "f1")
EVAL_CONFIG.setdefault("threshold_grid_size", 200)
EVAL_CONFIG.setdefault("threshold_min", 0.01)
EVAL_CONFIG.setdefault("threshold_max", 0.99)

EVAL_CONFIG["fusion_max_candidate_emails"] = 450
EVAL_CONFIG["fusion_fetch_timeout_seconds"] = 10
EVAL_CONFIG.setdefault("fusion_fetch_retries", 1)
EVAL_CONFIG.setdefault("fusion_min_paired_samples", 30)
EVAL_CONFIG.setdefault("fusion_min_samples_per_class", 8)
EVAL_CONFIG.setdefault("fusion_test_fraction", 0.30)
EVAL_CONFIG.setdefault("fusion_cv_folds", 5)
EVAL_CONFIG["fusion_min_minority_for_holdout"] = 25

if EVAL_CONFIG["threshold_metric"] not in {"f1", "youden_j"}:
    raise ValueError("evaluation.threshold_metric must be 'f1' or 'youden_j'.")

SEED = int(CONFIG.get("seed", 42))

LOGGER.info(
    "[fusion config] max_candidate_emails=%d, fetch_timeout_seconds=%d, min_minority_for_holdout=%d",
    EVAL_CONFIG["fusion_max_candidate_emails"], EVAL_CONFIG["fusion_fetch_timeout_seconds"],
    EVAL_CONFIG["fusion_min_minority_for_holdout"],
)


# ----------------------------------------------------------------------------
# 2. PATHS AND EVALUATION SIGNATURE
# ----------------------------------------------------------------------------

_EVALUATION_SIGNATURE_PAYLOAD = {
    "config_hash": CONFIG_HASH,
    "preprocessing_signature": PREPROCESSING_INFO.get("pipeline_signature"),
    "model_signature": MODEL_SIGNATURE,
    "training_signature": TRAINING_SIGNATURE,
    "evaluation_config": EVAL_CONFIG,
}
EVALUATION_SIGNATURE = hashlib.sha256(
    json.dumps(_EVALUATION_SIGNATURE_PAYLOAD, sort_keys=True, default=str).encode("utf-8")
).hexdigest()

_EVAL_ROOT = os.path.join(PATHS["outputs_metrics"], "evaluation", EVALUATION_SIGNATURE[:16])
CELL6_PATHS: Dict[str, Any] = {"root": _EVAL_ROOT, "fusion": os.path.join(_EVAL_ROOT, "fusion")}
for _model_name in MODEL_NAMES:
    CELL6_PATHS[_model_name] = {
        "predictions": os.path.join(_EVAL_ROOT, _model_name, "predictions"),
        "reports": os.path.join(_EVAL_ROOT, _model_name, "reports"),
    }
    for _p in CELL6_PATHS[_model_name].values():
        os.makedirs(_p, exist_ok=True)
os.makedirs(CELL6_PATHS["fusion"], exist_ok=True)


# ----------------------------------------------------------------------------
# 3. GENERAL UTILITIES
# ----------------------------------------------------------------------------

def utc_now_iso() -> str:
    """Return timezone-aware UTC timestamp."""
    return datetime.now(timezone.utc).isoformat()


def atomic_json_save(payload: Any, destination: str) -> None:
    """Save JSON atomically."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temporary_path = destination + ".tmp"
    with open(temporary_path, "w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, indent=2, sort_keys=True, default=str)
        file_handle.flush()
        os.fsync(file_handle.fileno())
    os.replace(temporary_path, destination)


def atomic_csv_save(dataframe: pd.DataFrame, destination: str) -> None:
    """Save CSV atomically."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temporary_path = destination + ".tmp"
    dataframe.to_csv(temporary_path, index=False, encoding="utf-8")
    os.replace(temporary_path, destination)


def load_model_feature_arrays(npz_path: str) -> Dict[str, np.ndarray]:
    """Load one model's safe feature-matrix NPZ (allow_pickle=False)."""
    if not os.path.isfile(npz_path):
        raise FileNotFoundError(f"Model input file not found: {npz_path}")
    with np.load(npz_path, allow_pickle=False) as archive:
        required = {"sample_ids", "labels", "split_codes", "features", "feature_names"}
        missing = required - set(archive.files)
        if missing:
            raise ValueError(f"NPZ is missing arrays: {sorted(missing)}")
        return {key: archive[key].copy() for key in archive.files}


def load_model_arrays_for_split(npz_path: str, split_code: int) -> Dict[str, np.ndarray]:
    """Load one split (0=train, 1=validation, 2=test) of one model's feature matrix."""
    arrays = load_model_feature_arrays(npz_path)
    mask = arrays["split_codes"] == int(split_code)
    labels = arrays["labels"][mask].astype(np.int64)
    if labels.size == 0:
        raise RuntimeError(f"Split code {split_code} is empty for {npz_path}.")
    classes = sorted(np.unique(labels).tolist())
    if classes != [0, 1]:
        raise RuntimeError(f"Split {split_code} must contain both classes. Found: {classes}")
    return {
        "sample_ids": arrays["sample_ids"][mask].astype(str), "labels": labels,
        "features": arrays["features"][mask].astype(np.float32),
        "feature_names": [str(n) for n in arrays["feature_names"]],
    }


def calculate_binary_metrics(
    labels: np.ndarray, phishing_probabilities: np.ndarray, threshold: float,
) -> Tuple[Dict[str, Any], np.ndarray]:
    """Compute standard binary classification metrics at a fixed threshold."""
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(phishing_probabilities, dtype=np.float64)
    predictions = (probabilities >= threshold).astype(int)

    metrics: Dict[str, Any] = {"sample_count": int(labels.size), "threshold": float(threshold)}
    metrics["accuracy"] = float(accuracy_score(labels, predictions))
    metrics["precision"] = float(precision_score(labels, predictions, zero_division=0))
    metrics["recall"] = float(recall_score(labels, predictions, zero_division=0))
    metrics["f1"] = float(f1_score(labels, predictions, zero_division=0))

    matrix = confusion_matrix(labels, predictions, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    metrics["specificity"] = float(tn / (tn + fp)) if (tn + fp) > 0 else None
    metrics["confusion_matrix"] = matrix.tolist()

    try:
        metrics["mcc"] = float(matthews_corrcoef(labels, predictions))
    except Exception:
        metrics["mcc"] = None
    try:
        metrics["roc_auc"] = float(roc_auc_score(labels, probabilities))
    except Exception:
        metrics["roc_auc"] = None
    try:
        metrics["pr_auc"] = float(average_precision_score(labels, probabilities))
    except Exception:
        metrics["pr_auc"] = None

    return metrics, predictions


def select_threshold_on_validation(
    validation_labels: np.ndarray, validation_probabilities: np.ndarray,
    metric_name: str, grid_size: int, threshold_min: float, threshold_max: float,
) -> Dict[str, Any]:
    """Select a decision threshold by maximizing the requested metric on VALIDATION only."""
    candidate_thresholds = np.linspace(threshold_min, threshold_max, grid_size)
    validation_labels = np.asarray(validation_labels, dtype=np.int64)
    validation_probabilities = np.asarray(validation_probabilities, dtype=np.float64)

    scores: List[float] = []
    for candidate_threshold in candidate_thresholds:
        predictions = (validation_probabilities >= candidate_threshold).astype(int)
        if metric_name == "f1":
            score = f1_score(validation_labels, predictions, zero_division=0)
        else:
            matrix = confusion_matrix(validation_labels, predictions, labels=[0, 1])
            tn, fp, fn, tp = matrix.ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
            score = sensitivity + specificity - 1.0
        scores.append(float(score))

    best_index = int(np.argmax(scores))
    return {
        "metric_used": metric_name, "threshold": float(candidate_thresholds[best_index]),
        "validation_score_at_threshold": float(scores[best_index]),
        "grid_size": int(grid_size),
    }


# ----------------------------------------------------------------------------
# 4. PART A — INDEPENDENT PER-MODEL EVALUATION (reused across all four)
# ----------------------------------------------------------------------------

def evaluate_one_model(model_name: str) -> Optional[Dict[str, Any]]:
    """
    Evaluate one model on its own test split, using a threshold selected
    exclusively from its own validation split.
    """
    training_result = TRAINING_INFO.get(model_name)
    if training_result is None:
        LOGGER.warning("[%s] No training result available; skipping evaluation.", model_name)
        return None

    task_started = time.perf_counter()
    fitted_payload = joblib.load(training_result["fitted_model_path"])
    fitted_model = fitted_payload["model"]

    npz_path = PREPROCESSING_INFO[model_name]["input_path"]
    validation_data = load_model_arrays_for_split(npz_path, split_code=1)
    test_data = load_model_arrays_for_split(npz_path, split_code=2)

    LOGGER.info(
        "[%s] Selecting threshold on validation split only (metric=%s, samples=%d)...",
        model_name, EVAL_CONFIG["threshold_metric"], len(validation_data["labels"]),
    )
    validation_probabilities = fitted_model.predict_proba(validation_data["features"])[:, 1]
    threshold_selection = select_threshold_on_validation(
        validation_labels=validation_data["labels"], validation_probabilities=validation_probabilities,
        metric_name=EVAL_CONFIG["threshold_metric"], grid_size=int(EVAL_CONFIG["threshold_grid_size"]),
        threshold_min=float(EVAL_CONFIG["threshold_min"]), threshold_max=float(EVAL_CONFIG["threshold_max"]),
    )
    selected_threshold = threshold_selection["threshold"]
    LOGGER.info(
        "[%s] Selected threshold = %.4f (validation %s = %.4f). Applying to test split.",
        model_name, selected_threshold, EVAL_CONFIG["threshold_metric"],
        threshold_selection["validation_score_at_threshold"],
    )

    LOGGER.info("[%s] Evaluating on test split (%d samples)...", model_name, len(test_data["labels"]))
    test_probabilities = fitted_model.predict_proba(test_data["features"])[:, 1]
    test_metrics, test_predictions = calculate_binary_metrics(
        test_data["labels"], test_probabilities, selected_threshold
    )

    LOGGER.info(
        "[%s] Test — accuracy=%.4f, precision=%.4f, recall=%.4f, f1=%.4f, roc_auc=%s, pr_auc=%s",
        model_name, test_metrics["accuracy"], test_metrics["precision"], test_metrics["recall"],
        test_metrics["f1"],
        f"{test_metrics['roc_auc']:.4f}" if test_metrics["roc_auc"] is not None else "N/A",
        f"{test_metrics['pr_auc']:.4f}" if test_metrics["pr_auc"] is not None else "N/A",
    )

    prediction_path = os.path.join(CELL6_PATHS[model_name]["predictions"], f"{model_name}_test_predictions.csv")
    atomic_csv_save(
        pd.DataFrame({
            "sample_id": test_data["sample_ids"].tolist(), "label": test_data["labels"].tolist(),
            "phishing_probability": test_probabilities.tolist(),
            "prediction_selected_threshold": test_predictions.tolist(),
        }),
        prediction_path,
    )

    confusion_matrix_path = os.path.join(CELL6_PATHS[model_name]["reports"], "confusion_matrix.csv")
    pd.DataFrame(
        test_metrics["confusion_matrix"], index=["true_legitimate", "true_phishing"],
        columns=["predicted_legitimate", "predicted_phishing"],
    ).to_csv(confusion_matrix_path, encoding="utf-8")

    classification_report_dict = classification_report(
        test_data["labels"], test_predictions, target_names=["Legitimate", "Phishing"],
        output_dict=True, zero_division=0,
    )
    classification_report_csv_path = os.path.join(CELL6_PATHS[model_name]["reports"], "classification_report.csv")
    pd.DataFrame(classification_report_dict).transpose().to_csv(classification_report_csv_path, encoding="utf-8")
    classification_report_json_path = os.path.join(CELL6_PATHS[model_name]["reports"], "classification_report.json")
    atomic_json_save(classification_report_dict, classification_report_json_path)

    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(test_data["labels"], test_probabilities)

    return {
        "task": model_name, "evaluation_signature": EVALUATION_SIGNATURE,
        "algorithm": training_result["algorithm"], "threshold_selection": threshold_selection,
        "test_metrics": test_metrics, "test_samples": int(len(test_data["labels"])),
        "test_class_counts": {int(k): int(v) for k, v in zip(*np.unique(test_data["labels"], return_counts=True))},
        "prediction_path": prediction_path, "confusion_matrix_path": confusion_matrix_path,
        "classification_report_csv_path": classification_report_csv_path,
        "classification_report_json_path": classification_report_json_path,
        "precision_recall_curve": {
            "precision": [float(v) for v in precision_curve], "recall": [float(v) for v in recall_curve],
            "thresholds": [float(v) for v in pr_thresholds],
        },
        "methodology_guards": {
            "threshold_selected_on_test": False, "threshold_selected_on_validation": True,
            "model_retrained_on_test": False,
        },
        "evaluation_seconds": round(time.perf_counter() - task_started, 4),
    }


# ----------------------------------------------------------------------------
# 5. PART B — CLOUD FUSION LAYER: PAIRED MULTIMODAL TEST SET CONSTRUCTION
# ----------------------------------------------------------------------------

_URL_PATTERN = re.compile(r"https?://[^\s<>'\"\]\)]+|www\.[^\s<>'\"\]\)]+", re.IGNORECASE)


def _resolve_helper(name: str):
    """Resolve a Cell 3 helper function from the global namespace, or None if unavailable."""
    return globals().get(name)


def find_first_url(text: str) -> Optional[str]:
    """Find the first URL-looking substring in a block of text."""
    match = _URL_PATTERN.search(text or "")
    if not match:
        return None
    url = match.group(0)
    if not url.lower().startswith(("http://", "https://")):
        url = "http://" + url
    return url


def normalize_url_for_leakage_check(url: str) -> str:
    """
    Normalize a URL for exact-match leakage checking (case-insensitive,
    scheme/www-prefix/trailing-slash insensitive), used ONLY to detect
    whether a fusion-evaluation URL literally coincides with a URL the
    URL model or Web Content model was TRAINED on -- not used for any
    feature computation.
    """
    text = str(url).strip().lower()
    text = re.sub(r"^https?://", "", text)
    text = re.sub(r"^www\.", "", text)
    return text.rstrip("/")


def load_base_model_training_urls() -> Dict[str, set]:
    """
    Load the EXACT set of URLs used to train the URL model and the Web
    Content model, so the fusion paired dataset can be checked for (and
    stripped of) any sample whose URL literally appeared in either base
    model's own training data. This is the one theoretically possible
    leakage vector in this pipeline that dataset independence alone does
    not automatically rule out -- Text/Behavioral leakage is already
    structurally impossible (the paired set is restricted to the TEXT
    model's own held-out TEST split before anything else happens).

    Returns:
        Dictionary with keys "url" and "web_content", each a set of
        normalized URL strings used in that model's TRAINING split.
    """
    training_url_sets: Dict[str, set] = {"url": set(), "web_content": set()}

    try:
        url_corpus = pd.read_csv(DATASET_INFO["model_datasets"]["url"]["prepared_path"], low_memory=False)
        url_split = pd.read_csv(PREPROCESSING_INFO["url"]["split_path"], low_memory=False)
        url_corpus["sample_id"] = url_corpus["sample_id"].astype(str)
        url_split["sample_id"] = url_split["sample_id"].astype(str)
        train_ids = set(url_split.loc[url_split["split"] == "train", "sample_id"])
        train_urls = url_corpus.loc[url_corpus["sample_id"].isin(train_ids), "url"].astype(str)
        training_url_sets["url"] = {normalize_url_for_leakage_check(u) for u in train_urls}
        LOGGER.info("[fusion] Loaded %d normalized training URLs for leakage check (URL model).",
                    len(training_url_sets["url"]))
    except Exception as exc:
        LOGGER.warning("[fusion] Could not load URL model's training URLs for leakage check: %s", exc)

    try:
        web_corpus = pd.read_csv(DATASET_INFO["model_datasets"]["web_content"]["prepared_path"], low_memory=False)
        web_split = pd.read_csv(PREPROCESSING_INFO["web_content"]["split_path"], low_memory=False)
        web_corpus["sample_id"] = web_corpus["sample_id"].astype(str)
        web_split["sample_id"] = web_split["sample_id"].astype(str)
        train_ids = set(web_split.loc[web_split["split"] == "train", "sample_id"])
        train_urls = web_corpus.loc[web_corpus["sample_id"].isin(train_ids), "url"].astype(str)
        training_url_sets["web_content"] = {normalize_url_for_leakage_check(u) for u in train_urls}
        LOGGER.info("[fusion] Loaded %d normalized training URLs for leakage check (Web Content model).",
                    len(training_url_sets["web_content"]))
    except Exception as exc:
        LOGGER.warning("[fusion] Could not load Web Content model's training URLs for leakage check: %s", exc)

    return training_url_sets


def create_fusion_http_session() -> requests.Session:
    """Create a short-timeout, low-retry HTTP session for live URL fetching."""
    retry_strategy = Retry(
        total=int(EVAL_CONFIG["fusion_fetch_retries"]), connect=1, read=1, status=0,
        backoff_factor=0.5, raise_on_status=False,
    )
    session = requests.Session()
    adapter = HTTPAdapter(max_retries=retry_strategy, pool_connections=10, pool_maxsize=10)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
        ),
    })
    return session


_FUSION_SESSION = create_fusion_http_session()


def fetch_url_content(url: str) -> Tuple[bool, str, str]:
    """Attempt to live-fetch one URL's landing page HTML, tolerating expected failure modes."""
    try:
        response = _FUSION_SESSION.get(
            url, timeout=int(EVAL_CONFIG["fusion_fetch_timeout_seconds"]),
            verify=False, allow_redirects=True,
        )
        if response.status_code != 200 or not response.text or len(response.text.strip()) < 50:
            return False, "", f"http_status_{response.status_code}_or_empty_body"
        return True, response.text, ""
    except requests.exceptions.SSLError as exc:
        return False, "", f"ssl_error: {exc}"
    except requests.exceptions.Timeout:
        return False, "", "timeout"
    except requests.exceptions.ConnectionError as exc:
        return False, "", f"connection_error: {exc}"
    except Exception as exc:
        return False, "", f"unexpected_error: {exc}"


def vectorize_numerical_features(feature_dict: Mapping[str, float], feature_names: Sequence[str]) -> np.ndarray:
    """Build a feature array in the exact column order a saved scaler expects."""
    return np.asarray([float(feature_dict.get(name, 0.0)) for name in feature_names], dtype=np.float32)


def build_fusion_paired_dataset() -> Dict[str, Any]:
    """
    Construct the paired multimodal fusion evaluation set: emails from the
    Text model's held-out test split that contain a URL, with that URL's
    landing page live-fetched, scored by all four models on the SAME
    underlying sample. Any sample whose URL literally appears in the URL
    model's or Web Content model's own training data is EXCLUDED and
    counted separately, closing the one previously-unverified leakage gap.

    Returns:
        Dictionary with the paired dataframe (or None), fetch statistics,
        and a fully disclosed accounting of every stage of attrition.
    """
    stats: Dict[str, Any] = {
        "text_test_email_count": 0, "emails_with_url_count": 0,
        "eligible_class_counts": {}, "candidate_emails_attempted": 0,
        "fetch_success_count": 0, "fetch_failure_count": 0,
        "failure_reasons": {},
        "excluded_due_to_url_model_training_overlap": 0,
        "excluded_due_to_web_content_model_training_overlap": 0,
        "final_paired_sample_count": 0,
    }

    helper_names = [
        "clean_text_for_vectorization", "extract_lexical_features", "extract_behavioral_features",
        "extract_url_features", "extract_web_content_features",
    ]
    helpers = {name: _resolve_helper(name) for name in helper_names}
    missing_helpers = [name for name, fn in helpers.items() if fn is None]
    if missing_helpers:
        LOGGER.warning(
            "Cloud Fusion Layer cannot be built — Cell 3 helper function(s) not in memory this "
            "session: %s. Re-run Cell 3 in this same kernel session before Cell 6 if you need fusion.",
            missing_helpers,
        )
        return {"available": False, "reason": f"missing_cell3_helpers: {missing_helpers}", "stats": stats}

    required_models = ["text", "behavioral", "url", "web_content"]
    missing_models = [m for m in required_models if TRAINING_INFO.get(m) is None]
    if missing_models:
        LOGGER.warning(
            "Cloud Fusion Layer requires all four models to be trained; missing: %s. Skipping fusion.",
            missing_models,
        )
        return {"available": False, "reason": f"missing_trained_models: {missing_models}", "stats": stats}

    # Load fitted models once. These are frozen, already-trained estimators
    # -- .predict_proba() is the only method called on them anywhere below.
    fitted_models = {
        name: joblib.load(TRAINING_INFO[name]["fitted_model_path"])["model"] for name in required_models
    }

    text_vectorizer_bundle = joblib.load(PREPROCESSING_INFO["text"]["tfidf_vectorizer_path"])
    tfidf_vectorizer = text_vectorizer_bundle["vectorizer"]

    behavioral_scaler_bundle = joblib.load(PREPROCESSING_INFO["behavioral"]["scaler_path"])
    behavioral_scaler = behavioral_scaler_bundle["scaler"]
    behavioral_feature_names = behavioral_scaler_bundle["feature_columns"]

    url_scaler_bundle = joblib.load(PREPROCESSING_INFO["url"]["scaler_path"])
    url_scaler = url_scaler_bundle["scaler"]
    url_feature_names = url_scaler_bundle["feature_columns"]

    web_scaler_bundle = joblib.load(PREPROCESSING_INFO["web_content"]["scaler_path"])
    web_scaler = web_scaler_bundle["scaler"]
    web_feature_names = web_scaler_bundle["feature_columns"]

    # NEW: explicit training-URL sets for the leakage-exclusion check.
    training_url_sets = load_base_model_training_urls()

    # Load Text model's test-split emails WITH their original subject/body.
    text_split_frame = pd.read_csv(PREPROCESSING_INFO["text"]["split_path"], low_memory=False)
    test_sample_ids = set(text_split_frame.loc[text_split_frame["split"] == "test", "sample_id"].astype(str))

    email_corpus_path = DATASET_INFO["model_datasets"]["text"]["prepared_path"]
    email_corpus_frame = pd.read_csv(email_corpus_path, low_memory=False)
    email_corpus_frame["sample_id"] = email_corpus_frame["sample_id"].astype(str)
    test_email_frame = email_corpus_frame[email_corpus_frame["sample_id"].isin(test_sample_ids)].copy()
    stats["text_test_email_count"] = int(len(test_email_frame))

    test_email_frame["subject"] = test_email_frame["subject"].fillna("").astype(str)
    test_email_frame["body"] = test_email_frame["body"].fillna("").astype(str)
    test_email_frame["sender"] = test_email_frame["sender"].fillna("").astype(str)
    test_email_frame["candidate_url"] = (test_email_frame["subject"] + "\n" + test_email_frame["body"]).map(
        find_first_url
    )

    emails_with_url = test_email_frame[test_email_frame["candidate_url"].notna()].copy()
    stats["emails_with_url_count"] = int(len(emails_with_url))
    stats["eligible_class_counts"] = {
        int(k): int(v) for k, v in emails_with_url["label"].value_counts().sort_index().items()
    }

    if emails_with_url.empty:
        LOGGER.warning("No emails in the Text model's test split contain a detectable URL. Fusion skipped.")
        return {"available": False, "reason": "no_emails_with_urls_in_test_split", "stats": stats}

    max_candidates = int(EVAL_CONFIG["fusion_max_candidate_emails"])
    per_class_cap = max(1, max_candidates // 2)
    sampled_parts = []
    for label_value in (0, 1):
        class_subset = emails_with_url[emails_with_url["label"] == label_value]
        sample_size = min(per_class_cap, len(class_subset))
        if sample_size > 0:
            sampled_parts.append(class_subset.sample(n=sample_size, random_state=SEED))
    candidate_frame = pd.concat(sampled_parts, ignore_index=True) if sampled_parts else emails_with_url.iloc[0:0]
    stats["candidate_emails_attempted"] = int(len(candidate_frame))

    LOGGER.info(
        "[fusion] Eligible pool: %d emails-with-URLs (classes: %s). Attempting to live-fetch %d "
        "candidates (per-class cap=%d, overall cap=%d)...",
        len(emails_with_url), stats["eligible_class_counts"], len(candidate_frame), per_class_cap, max_candidates,
    )

    paired_rows: List[Dict[str, Any]] = []
    for _, row in candidate_frame.iterrows():
        url = str(row["candidate_url"])
        normalized_url = normalize_url_for_leakage_check(url)

        # NEW: explicit leakage exclusion. If this exact URL (normalized)
        # was used to TRAIN the URL model or the Web Content model, this
        # sample cannot honestly be used to EVALUATE those same models --
        # exclude it here, before spending a fetch attempt on it.
        if normalized_url in training_url_sets["url"]:
            stats["excluded_due_to_url_model_training_overlap"] += 1
            continue
        if normalized_url in training_url_sets["web_content"]:
            stats["excluded_due_to_web_content_model_training_overlap"] += 1
            continue

        success, html_text, failure_reason = fetch_url_content(url)

        if not success:
            stats["fetch_failure_count"] += 1
            stats["failure_reasons"][failure_reason] = stats["failure_reasons"].get(failure_reason, 0) + 1
            continue
        stats["fetch_success_count"] += 1

        subject, body, sender = row["subject"], row["body"], row["sender"]

        cleaned_text = helpers["clean_text_for_vectorization"](f"{subject} {body}")
        text_features = tfidf_vectorizer.transform([cleaned_text]).toarray().astype(np.float32)
        p_text = float(fitted_models["text"].predict_proba(text_features)[0, 1])

        lexical = helpers["extract_lexical_features"](subject, body)
        behavioral = helpers["extract_behavioral_features"](subject, body, sender)
        combined_beh = {**lexical, **behavioral}
        behavioral_array = vectorize_numerical_features(combined_beh, behavioral_feature_names).reshape(1, -1)
        behavioral_scaled = behavioral_scaler.transform(behavioral_array).astype(np.float32)
        p_behavioral = float(fitted_models["behavioral"].predict_proba(behavioral_scaled)[0, 1])

        url_features_dict = helpers["extract_url_features"](url)
        url_array = vectorize_numerical_features(url_features_dict, url_feature_names).reshape(1, -1)
        url_scaled = url_scaler.transform(url_array).astype(np.float32)
        p_url = float(fitted_models["url"].predict_proba(url_scaled)[0, 1])

        web_features_dict = helpers["extract_web_content_features"](html_text, url)
        web_array = vectorize_numerical_features(web_features_dict, web_feature_names).reshape(1, -1)
        web_scaled = web_scaler.transform(web_array).astype(np.float32)
        p_web_content = float(fitted_models["web_content"].predict_proba(web_scaled)[0, 1])

        paired_rows.append({
            "sample_id": row["sample_id"], "url": url, "label": int(row["label"]),
            "p_text": p_text, "p_url": p_url, "p_behavioral": p_behavioral, "p_web_content": p_web_content,
        })

    stats["final_paired_sample_count"] = len(paired_rows)
    paired_frame = pd.DataFrame(paired_rows)

    paired_dataset_path = os.path.join(CELL6_PATHS["fusion"], "fusion_paired_dataset.csv")
    atomic_csv_save(paired_frame, paired_dataset_path)

    LOGGER.info(
        "[fusion] Fetch results: %d succeeded / %d attempted (%d failed). Failure breakdown: %s",
        stats["fetch_success_count"], stats["candidate_emails_attempted"],
        stats["fetch_failure_count"], stats["failure_reasons"],
    )
    LOGGER.info(
        "[fusion] Excluded due to training-set overlap: %d (URL model), %d (Web Content model).",
        stats["excluded_due_to_url_model_training_overlap"], stats["excluded_due_to_web_content_model_training_overlap"],
    )

    return {
        "available": not paired_frame.empty, "paired_dataframe": paired_frame,
        "paired_dataset_path": paired_dataset_path, "stats": stats,
    }


# ----------------------------------------------------------------------------
# 6. PART B — TRAIN AND EVALUATE THE FUSION META-CLASSIFIER
# ----------------------------------------------------------------------------

def train_and_evaluate_fusion_layer(fusion_construction: Dict[str, Any]) -> Dict[str, Any]:
    """
    Fit a Logistic Regression meta-classifier on [p_text, p_url, p_behavioral,
    p_web_content] -> label, using whichever paired samples survived live
    fetching and the training-overlap exclusion check. Uses a held-out
    split ONLY if the minority class is large enough; otherwise falls back
    to stratified k-fold cross-validation.
    """
    if not fusion_construction.get("available"):
        return {
            "available": False, "reason": fusion_construction.get("reason", "unknown"),
            "stats": fusion_construction.get("stats", {}),
        }

    paired_frame = fusion_construction["paired_dataframe"]
    stats = dict(fusion_construction["stats"])

    class_counts = paired_frame["label"].value_counts().to_dict()
    stats["paired_class_counts"] = {int(k): int(v) for k, v in class_counts.items()}

    min_total = int(EVAL_CONFIG["fusion_min_paired_samples"])
    min_per_class = int(EVAL_CONFIG["fusion_min_samples_per_class"])

    if len(paired_frame) < min_total or len(class_counts) < 2 or min(class_counts.values()) < min_per_class:
        LOGGER.warning(
            "[fusion] Only %d paired samples survived live-fetching + leakage exclusion (class counts: %s); "
            "below the minimum required (%d total, %d per class). Reporting the paired dataset for "
            "inspection, but NOT computing fusion metrics from an inadequate sample.",
            len(paired_frame), stats["paired_class_counts"], min_total, min_per_class,
        )
        return {
            "available": False, "reason": "insufficient_paired_samples_after_live_fetch_and_leakage_exclusion",
            "stats": stats, "paired_dataset_path": fusion_construction["paired_dataset_path"],
        }

    feature_columns = ["p_text", "p_url", "p_behavioral", "p_web_content"]
    X = paired_frame[feature_columns].to_numpy(dtype=np.float64)
    y = paired_frame["label"].to_numpy(dtype=np.int64)

    minority_class_count = int(min(class_counts.values()))
    minority_holdout_floor = int(EVAL_CONFIG["fusion_min_minority_for_holdout"])
    use_holdout = minority_class_count >= minority_holdout_floor

    LOGGER.info(
        "[fusion] Minority class count = %d (floor for holdout = %d) -> using %s.",
        minority_class_count, minority_holdout_floor, "holdout split" if use_holdout else "cross-validation",
    )

    if use_holdout:
        X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
            X, y, np.arange(len(y)), test_size=float(EVAL_CONFIG["fusion_test_fraction"]),
            random_state=SEED, stratify=y,
        )
        fusion_model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
        fusion_model.fit(X_train, y_train)
        test_probabilities = fusion_model.predict_proba(X_test)[:, 1]
        evaluation_mode = "holdout_split"
        evaluated_sample_ids = paired_frame.iloc[idx_test]["sample_id"].tolist()
        evaluated_labels = y_test
    else:
        n_splits = min(int(EVAL_CONFIG["fusion_cv_folds"]), minority_class_count)
        n_splits = max(2, n_splits)
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        fusion_model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
        test_probabilities = cross_val_predict(fusion_model, X, y, cv=cv, method="predict_proba")[:, 1]
        fusion_model.fit(X, y)
        evaluation_mode = f"stratified_{n_splits}fold_cv_out_of_fold_predictions"
        evaluated_sample_ids = paired_frame["sample_id"].tolist()
        evaluated_labels = y

    fusion_metrics, fusion_predictions = calculate_binary_metrics(evaluated_labels, test_probabilities, threshold=0.5)

    fusion_model_path = os.path.join(CELL6_PATHS["fusion"], "fusion_logistic_regression.joblib")
    joblib.dump(
        {"model": fusion_model, "feature_columns": feature_columns, "evaluation_mode": evaluation_mode},
        fusion_model_path,
    )

    fusion_predictions_path = os.path.join(CELL6_PATHS["fusion"], "fusion_predictions.csv")
    atomic_csv_save(
        pd.DataFrame({
            "sample_id": evaluated_sample_ids, "label": evaluated_labels.tolist(),
            "fusion_probability": test_probabilities.tolist(),
            "fusion_prediction": fusion_predictions.tolist(),
        }),
        fusion_predictions_path,
    )

    mean_probabilities = paired_frame[feature_columns].mean(axis=1).to_numpy()
    baseline_metrics, _ = calculate_binary_metrics(y, mean_probabilities, threshold=0.5)

    return {
        "available": True, "evaluation_mode": evaluation_mode,
        "minority_class_count": minority_class_count,
        "minority_holdout_floor": minority_holdout_floor,
        "stats": stats,
        "paired_dataset_path": fusion_construction["paired_dataset_path"],
        "fusion_model_path": fusion_model_path, "fusion_predictions_path": fusion_predictions_path,
        "fusion_metrics": fusion_metrics,
        "mean_probability_baseline_metrics": baseline_metrics,
        "meta_classifier_coefficients": dict(zip(feature_columns, fusion_model.coef_[0].tolist())),
    }


# ----------------------------------------------------------------------------
# 7. MASTER EVALUATION PIPELINE
# ----------------------------------------------------------------------------

def run_evaluation_pipeline() -> Dict[str, Any]:
    """Evaluate all four models independently, then build and evaluate the fusion layer."""
    pipeline_started = time.perf_counter()
    resources_before = (
        get_system_resource_snapshot() if "get_system_resource_snapshot" in globals() else {}
    )

    per_model_results: Dict[str, Optional[Dict[str, Any]]] = {}
    for model_name in MODEL_NAMES:
        LOGGER.info("--- Evaluating %s model ---", model_name)
        try:
            per_model_results[model_name] = evaluate_one_model(model_name)
        except Exception as exc:
            LOGGER.error("[%s] Evaluation failed: %s", model_name, exc)
            per_model_results[model_name] = None

    if not any(per_model_results.values()):
        raise RuntimeError("No model produced a valid test evaluation.")

    LOGGER.info("--- Building Cloud Fusion Layer paired multimodal test set ---")
    fusion_construction = build_fusion_paired_dataset()
    fusion_result = train_and_evaluate_fusion_layer(fusion_construction)

    summary_path = os.path.join(CELL6_PATHS["root"], "evaluation_summary.json")
    evaluation_information: Dict[str, Any] = {
        "evaluation_version": EVAL_CONFIG["version"], "evaluation_signature": EVALUATION_SIGNATURE,
        "training_signature": TRAINING_SIGNATURE, "model_signature": MODEL_SIGNATURE,
        "preprocessing_signature": PREPROCESSING_INFO.get("pipeline_signature"),
        "experiment_uuid": EXPERIMENT_UUID, "config_hash": CONFIG_HASH, "scope": "multi_model",
        "completed_at_utc": utc_now_iso(),
        "evaluation_seconds": round(time.perf_counter() - pipeline_started, 4),
        "text": per_model_results["text"], "behavioral": per_model_results["behavioral"],
        "url": per_model_results["url"], "web_content": per_model_results["web_content"],
        "fusion": fusion_result,
        "summary_path": summary_path,
        "methodology_guards": {
            "threshold_selected_on_test": False, "threshold_selected_on_validation": True,
            "model_retrained_on_test": False,
            "fusion_uses_paired_same_email_samples": True,
            "fusion_metrics_fabricated_if_insufficient_samples": False,
            "fusion_evaluation_mode_based_on_minority_class_count": True,
            "fusion_excludes_samples_overlapping_base_model_training_urls": True,
        },
        "resources_before": resources_before,
        "resources_after": (
            get_system_resource_snapshot() if "get_system_resource_snapshot" in globals() else {}
        ),
    }

    atomic_json_save(evaluation_information, summary_path)
    return evaluation_information


# ----------------------------------------------------------------------------
# 8. EXECUTE CELL 6
# ----------------------------------------------------------------------------

try:
    EVALUATION_INFO: Dict[str, Any] = run_evaluation_pipeline()

    write_status(
        "evaluation_pipeline_complete", STATUS_PATH,
        extra={
            "cell": 6, "evaluation_signature": EVALUATION_SIGNATURE,
            "available_models": [m for m in MODEL_NAMES if EVALUATION_INFO.get(m) is not None],
            "fusion_available": EVALUATION_INFO["fusion"].get("available", False),
            "summary_path": EVALUATION_INFO["summary_path"],
        },
    )

except Exception as exc:
    error_trace = traceback.format_exc()
    LOGGER.error("Cell 6 failed: %s\n%s", exc, error_trace)
    write_status(
        "evaluation_pipeline_error", STATUS_PATH,
        extra={"cell": 6, "error": str(exc), "traceback": error_trace}, is_error=True,
    )
    raise


# ----------------------------------------------------------------------------
# 9. FINAL SUMMARY
# ----------------------------------------------------------------------------

print("=" * 78)
print("MAL-PhishNet — Cell 6 Complete (Evaluation + Cloud Fusion Layer)".center(78))
print("=" * 78)

for _model_name in MODEL_NAMES:
    _result = EVALUATION_INFO.get(_model_name)
    print("-" * 78)
    if not _result:
        print(f"{_model_name.upper()} MODEL: SKIPPED")
        continue
    _m = _result["test_metrics"]
    print(f"{_model_name.upper()} MODEL")
    print(f"  Threshold        : {_result['threshold_selection']['threshold']:.4f}")
    print(f"  Test samples     : {_result['test_samples']} — classes {_result['test_class_counts']}")
    print(f"  Accuracy/Prec/Rec/F1: {_m['accuracy']:.4f} / {_m['precision']:.4f} / {_m['recall']:.4f} / {_m['f1']:.4f}")
    print(f"  ROC-AUC / PR-AUC : {_m.get('roc_auc')} / {_m.get('pr_auc')}")
    print(f"  Predictions CSV  : {_result['prediction_path']}")

print("-" * 78)
print("CLOUD FUSION LAYER")
_fusion = EVALUATION_INFO["fusion"]
print(f"  Config used      : max_candidates={EVAL_CONFIG['fusion_max_candidate_emails']}, "
      f"timeout={EVAL_CONFIG['fusion_fetch_timeout_seconds']}s, "
      f"min_minority_for_holdout={EVAL_CONFIG['fusion_min_minority_for_holdout']}")
print(f"  Available        : {_fusion.get('available')}")
if _fusion.get("available"):
    print(f"  Evaluation mode  : {_fusion['evaluation_mode']}")
    print(f"  Minority class count: {_fusion.get('minority_class_count')} "
          f"(holdout requires >= {_fusion.get('minority_holdout_floor')})")
    print(f"  Eligible pool    : {_fusion['stats'].get('emails_with_url_count')} "
          f"(classes {_fusion['stats'].get('eligible_class_counts')})")
    print(f"  Candidates tried : {_fusion['stats'].get('candidate_emails_attempted')}")
    print(f"  Excluded (URL model training overlap)         : "
          f"{_fusion['stats'].get('excluded_due_to_url_model_training_overlap')}")
    print(f"  Excluded (Web Content model training overlap) : "
          f"{_fusion['stats'].get('excluded_due_to_web_content_model_training_overlap')}")
    print(f"  Fetch success    : {_fusion['stats'].get('fetch_success_count')}")
    print(f"  Paired samples   : {_fusion['stats']['final_paired_sample_count']} "
          f"(classes {_fusion['stats'].get('paired_class_counts')})")
    _fm = _fusion["fusion_metrics"]
    print(f"  Accuracy/Prec/Rec/F1: {_fm['accuracy']:.4f} / {_fm['precision']:.4f} / "
          f"{_fm['recall']:.4f} / {_fm['f1']:.4f}")
    print(f"  ROC-AUC / PR-AUC : {_fm.get('roc_auc')} / {_fm.get('pr_auc')}")
    print(f"  Baseline (mean-prob) F1: {_fusion['mean_probability_baseline_metrics']['f1']:.4f}")
    print(f"  Meta-classifier coefficients: {_fusion['meta_classifier_coefficients']}")
else:
    print(f"  Reason           : {_fusion.get('reason')}")
    print(f"  Stats            : {_fusion.get('stats')}")

print("-" * 78)
print(f"Evaluation summary : {EVALUATION_INFO['summary_path']}")
print(f"Evaluation seconds : {EVALUATION_INFO['evaluation_seconds']}")
print("=" * 78)

[2026-08-15 16:49:14] [INFO    ] [MAL-PhishNet] ==============================================================================
[2026-08-15 16:49:14] [INFO    ] [MAL-PhishNet]             MAL-PhishNet — Cell 6: Evaluation + Cloud Fusion Layer            
[2026-08-15 16:49:14] [INFO    ] [MAL-PhishNet] ==============================================================================
[2026-08-15 16:49:14] [INFO    ] [MAL-PhishNet] [fusion config] max_candidate_emails=450, fetch_timeout_seconds=10, min_minority_for_holdout=25


/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


[2026-08-15 16:49:15] [INFO    ] [MAL-PhishNet] --- Evaluating text model ---
[2026-08-15 16:49:16] [INFO    ] [MAL-PhishNet] [text] Selecting threshold on validation split only (metric=f1, samples=643)...
[2026-08-15 16:49:16] [INFO    ] [MAL-PhishNet] [text] Selected threshold = 0.4729 (validation f1 = 0.9851). Applying to test split.
[2026-08-15 16:49:16] [INFO    ] [MAL-PhishNet] [text] Evaluating on test split (643 samples)...
[2026-08-15 16:49:16] [INFO    ] [MAL-PhishNet] [text] Test — accuracy=0.9829, precision=0.9705, recall=0.9829, f1=0.9766, roc_auc=0.9985, pr_auc=0.9976
[2026-08-15 16:49:17] [INFO    ] [MAL-PhishNet] --- Evaluating behavioral model ---
[2026-08-15 16:49:17] [INFO    ] [MAL-PhishNet] [behavioral] Selecting threshold on validation split only (metric=f1, samples=643)...
[2026-08-15 16:49:18] [INFO    ] [MAL-PhishNet] [behavioral] Selected threshold = 0.5025 (validation f1 = 0.9831). Applying to test split.
[2026-08-15 16:49:18] [INFO    ] [MAL-PhishNet] [behav

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.gamestop.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.eff.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'freshrpms.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: 

[2026-08-15 16:59:04] [INFO    ] [MAL-PhishNet] [fusion] Fetch results: 69 succeeded / 262 attempted (185 failed). Failure breakdown: {'http_status_404_or_empty_body': 37, 'connection_error: HTTPConnectionPool(host=\'www.newsisfree.com\', port=80): Max retries exceeded with url: /click/-6,8572789,215/ (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x7f84903ed7c0>: Failed to resolve \'www.newsisfree.com\' ([Errno -3] Temporary failure in name resolution)"))': 1, 'connection_error: HTTPConnectionPool(host=\'messenger.msn.com\', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x7f84903ef6e0>: Failed to resolve \'messenger.msn.com\' ([Errno -2] Name or service not known)"))': 1, 'http_status_403_or_empty_body': 14, 'connection_error: HTTPConnectionPool(host=\'www.newsisfree.com\', port=80): Max retries exceeded with url: /click/-0,8357897,215/ (Caused by NameResolutionError("<urllib3.conne

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.monkey.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


       MAL-PhishNet — Cell 6 Complete (Evaluation + Cloud Fusion Layer)       
------------------------------------------------------------------------------
TEXT MODEL
  Threshold        : 0.4729
  Test samples     : 643 — classes {0: 409, 1: 234}
  Accuracy/Prec/Rec/F1: 0.9829 / 0.9705 / 0.9829 / 0.9766
  ROC-AUC / PR-AUC : 0.9985162894698347 / 0.9976465265791352
  Predictions CSV  : /content/drive/MyDrive/MAL-PhishNet/outputs/metrics/evaluation/b924aa197ed7d8ff/text/predictions/text_test_predictions.csv
------------------------------------------------------------------------------
BEHAVIORAL MODEL
  Threshold        : 0.5025
  Test samples     : 643 — classes {0: 409, 1: 234}
  Accuracy/Prec/Rec/F1: 0.9844 / 0.9746 / 0.9829 / 0.9787
  ROC-AUC / PR-AUC : 0.9981819321672624 / 0.9964036474932265
  Predictions CSV  : /content/drive/MyDrive/MAL-PhishNet/outputs/metrics/evaluation/b924aa197ed7d8ff/behavioral/predictions/behavioral_test_predictions.csv
-------------------------------------

/tmp/ipykernel_863/3319101159.py:1037: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


In [ ]:
# ============================================================================
# MAL-PhishNet — Cell 7B: EMAIL-ONLY Paper Report
# "Email Phishing Detection — Nazario + SpamAssassin"
# ============================================================================
# Purpose: produce paper-ready results for THIS paper's first experiment
# (Textual + Lexical + Behavioral features, Random Forest, email dataset
# only), strictly excluding URL / Web Content / Cloud Fusion.
#
# Does NOT retrain anything. Reuses EVALUATION_INFO from Cell 6 if it is
# still in memory; otherwise loads the cached evaluation_summary.json and
# the cached per-model prediction CSVs written by Cell 6. Run Cells 1-6
# once (already done); this cell can be re-run on its own afterward.
#
# IMPORTANT SCOPE NOTE (do not remove): the current implementation trains
# "text" (TF-IDF) and "behavioral" (lexical+behavioral folded together) as
# TWO INDEPENDENT Random Forest models — there is no single fused feature
# matrix / single classifier producing one combined number. This cell
# reports both models side by side rather than inventing a fused metric.
# If a single fused number is required, that is a methodology change to
# Cells 3-5 (feature fusion before training), not a reporting change here.
# ============================================================================

import os
import json
import warnings
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix

EMAIL_MODELS = ["text", "behavioral"]  # the ONLY models that belong to this paper
DATASET_LABEL = "Email Phishing Detection — Nazario + SpamAssassin"

# ----------------------------------------------------------------------------
# 1. LOCATE / LOAD EVALUATION RESULTS (cached, no retraining)
# ----------------------------------------------------------------------------

def _load_evaluation_info():
    if "EVALUATION_INFO" in globals():
        return globals()["EVALUATION_INFO"]
    # Fallback: read the cached summary written at the end of Cell 6.
    candidate_paths = []
    if "CELL6_PATHS" in globals():
        candidate_paths.append(os.path.join(CELL6_PATHS["root"], "evaluation_summary.json"))
    if "PATHS" in globals():
        candidate_paths.append(
            os.path.join(PATHS.get("root", ""), "outputs", "metrics", "evaluation")
        )
    for p in candidate_paths:
        if p and p.endswith(".json") and os.path.isfile(p):
            with open(p, "r", encoding="utf-8") as fh:
                return json.load(fh)
        if p and os.path.isdir(p):
            # search subfolders for evaluation_summary.json (keyed by evaluation signature)
            for root, _, files in os.walk(p):
                if "evaluation_summary.json" in files:
                    with open(os.path.join(root, "evaluation_summary.json"), "r", encoding="utf-8") as fh:
                        return json.load(fh)
    raise RuntimeError(
        "Could not find EVALUATION_INFO in memory or evaluation_summary.json on disk. "
        "Run Cell 6 at least once before this cell."
    )

EVALUATION_INFO = _load_evaluation_info()

for _m in EMAIL_MODELS:
    if EVALUATION_INFO.get(_m) is None:
        raise RuntimeError(f"[{_m}] has no evaluation result in EVALUATION_INFO — cannot build paper report.")

print("=" * 78)
print(DATASET_LABEL.center(78))
print("=" * 78)
print("Models included : text (TF-IDF/textual), behavioral (lexical+behavioral)")
print("Models excluded : url, web_content, fusion  <-- NOT part of this paper")
print("-" * 78)

# ----------------------------------------------------------------------------
# 2. OUTPUT FOLDERS (separate from the multi-model report, unambiguous names)
# ----------------------------------------------------------------------------

_ROOT_BASE = PATHS.get("root", "/content/drive/MyDrive/MAL-PhishNet") if "PATHS" in globals() else "."
EMAIL_REPORT_ROOT = os.path.join(_ROOT_BASE, "Email_Experiment_Report_NazarioSpamAssassin")
EMAIL_REPORT_PATHS = {
    "root": EMAIL_REPORT_ROOT,
    "figures": os.path.join(EMAIL_REPORT_ROOT, "Figures"),
    "tables": os.path.join(EMAIL_REPORT_ROOT, "Tables"),
    "reports": os.path.join(EMAIL_REPORT_ROOT, "Reports"),
}
for _p in EMAIL_REPORT_PATHS.values():
    os.makedirs(_p, exist_ok=True)

# ----------------------------------------------------------------------------
# 3. LOAD PER-MODEL TEST PREDICTIONS (cached CSVs from Cell 6)
# ----------------------------------------------------------------------------

def _load_predictions(model_name: str) -> pd.DataFrame:
    result = EVALUATION_INFO[model_name]
    path = result["prediction_path"]
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"[{model_name}] Cached predictions CSV not found at {path}. "
            "Re-run Cell 6 to regenerate it (do not re-run Cells 1-5)."
        )
    return pd.read_csv(path)

EMAIL_PREDICTIONS = {m: _load_predictions(m) for m in EMAIL_MODELS}

# ----------------------------------------------------------------------------
# 4. METRICS SUMMARY TABLE (paper-ready, verbatim from Cell 6 — nothing invented)
# ----------------------------------------------------------------------------

_rows = []
for model_name in EMAIL_MODELS:
    r = EVALUATION_INFO[model_name]
    m = r["test_metrics"]
    _rows.append({
        "Model": {"text": "Textual (TF-IDF)", "behavioral": "Lexical + Behavioral"}[model_name],
        "Dataset": "Nazario + SpamAssassin",
        "Classifier": r["algorithm"],
        "Test samples": r["test_samples"],
        "Threshold": round(r["threshold_selection"]["threshold"], 4),
        "Accuracy": round(m["accuracy"], 4),
        "Precision": round(m["precision"], 4),
        "Recall": round(m["recall"], 4),
        "F1-score": round(m["f1"], 4),
        "ROC-AUC": round(m["roc_auc"], 4) if m.get("roc_auc") is not None else None,
        "PR-AUC": round(m["pr_auc"], 4) if m.get("pr_auc") is not None else None,
    })

email_metrics_table = pd.DataFrame(_rows)
metrics_csv_path = os.path.join(EMAIL_REPORT_PATHS["tables"], "Table_Email_Experiment_Metrics.csv")
email_metrics_table.to_csv(metrics_csv_path, index=False)
print(email_metrics_table.to_string(index=False))
print(f"\nSaved: {metrics_csv_path}")

# ----------------------------------------------------------------------------
# 5. CONFUSION MATRICES (correctly captioned, per model)
# ----------------------------------------------------------------------------

for model_name in EMAIL_MODELS:
    df = EMAIL_PREDICTIONS[model_name]
    cm = confusion_matrix(df["label"], df["prediction_selected_threshold"])
    label_display = {"text": "Textual (TF-IDF) Model", "behavioral": "Lexical + Behavioral Model"}[model_name]

    fig, ax = plt.subplots(figsize=(5, 4.5), constrained_layout=True)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"Figure — Confusion Matrix\n{label_display} | {DATASET_LABEL} | Test split", fontsize=10)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Legitimate", "Phishing"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Legitimate", "Phishing"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    out_path = os.path.join(EMAIL_REPORT_PATHS["figures"], f"confusion_matrix_{model_name}_email.png")
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"Saved: {out_path}")

# ----------------------------------------------------------------------------
# 6. ROC CURVES (correctly captioned, per model)
# ----------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
for model_name in EMAIL_MODELS:
    df = EMAIL_PREDICTIONS[model_name]
    fpr, tpr, _ = roc_curve(df["label"], df["phishing_probability"])
    roc_auc_value = auc(fpr, tpr)
    label_display = {"text": "Textual (TF-IDF)", "behavioral": "Lexical + Behavioral"}[model_name]
    ax.plot(fpr, tpr, label=f"{label_display} (AUC = {roc_auc_value:.4f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"Figure — ROC Curve\n{DATASET_LABEL} | Test split", fontsize=10)
ax.legend(loc="lower right", fontsize=8)
roc_path = os.path.join(EMAIL_REPORT_PATHS["figures"], "roc_curve_email_experiment.png")
fig.savefig(roc_path, dpi=200)
plt.close(fig)
print(f"Saved: {roc_path}")

# ----------------------------------------------------------------------------
# 7. PAPER-READY JSON (exact final numbers, traceable to source artifacts)
# ----------------------------------------------------------------------------

paper_results = {
    "experiment_name": DATASET_LABEL,
    "dataset": {
        "phishing_source": "Nazario Phishing Email Corpus (Zenodo record 8339691)",
        "legitimate_source": "SpamAssassin Public Corpus (easy_ham + hard_ham)",
        "phishing_count_after_cleaning": 1564,
        "legitimate_count_after_cleaning": 2720,
        "total": 4284,
        "split": {"train": 2998, "validation": 643, "test": 643},
    },
    "classifier": "Random Forest (identical hyperparameters for both feature groups; "
                  "each feature group trained as an independent model — no single "
                  "fused feature matrix exists in the current implementation)",
    "results": {m: EVALUATION_INFO[m]["test_metrics"] for m in EMAIL_MODELS},
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_evaluation_signature": EVALUATION_INFO.get("evaluation_signature"),
    "excluded_from_this_paper": ["url", "web_content", "fusion"],
}
json_path = os.path.join(EMAIL_REPORT_PATHS["reports"], "email_experiment_paper_results.json")
with open(json_path, "w", encoding="utf-8") as fh:
    json.dump(paper_results, fh, indent=2, default=str)
print(f"Saved: {json_path}")

print("-" * 78)
print("Cell 7B complete. Use Table_Email_Experiment_Metrics.csv and")
print("email_experiment_paper_results.json as the ONLY source of results/")
print("figures for this paper's email experiment.")
print("=" * 78)

              Email Phishing Detection — Nazario + SpamAssassin               
Models included : text (TF-IDF/textual), behavioral (lexical+behavioral)
Models excluded : url, web_content, fusion  <-- NOT part of this paper
------------------------------------------------------------------------------
               Model                Dataset    Classifier  Test samples  Threshold  Accuracy  Precision  Recall  F1-score  ROC-AUC  PR-AUC
    Textual (TF-IDF) Nazario + SpamAssassin random_forest           643     0.4729    0.9829     0.9705  0.9829    0.9766   0.9985  0.9976
Lexical + Behavioral Nazario + SpamAssassin random_forest           643     0.5025    0.9844     0.9746  0.9829    0.9787   0.9982  0.9964

Saved: /content/drive/MyDrive/MAL-PhishNet/Email_Experiment_Report_NazarioSpamAssassin/Tables/Table_Email_Experiment_Metrics.csv
Saved: /content/drive/MyDrive/MAL-PhishNet/Email_Experiment_Report_NazarioSpamAssassin/Figures/confusion_matrix_text_email.png
Saved: /content/drive/My

In [3]:
# ============================================================================
# MAL-PhishNet — Cell 7B: EMAIL-ONLY Paper Report
# "Email Phishing Detection — Nazario + SpamAssassin"
# ============================================================================
# Purpose: produce / package paper-ready results for THIS paper's first
# experiment (Textual + Lexical + Behavioral features, Random Forest, email
# dataset only), strictly excluding URL / Web Content / Cloud Fusion.
#
# BEHAVIOR (in order):
#   1. First check whether the paper-ready artifacts from a PRIOR successful
#      run of this cell already exist on disk, at the exact paths this cell
#      itself uses. If they exist, LOAD them directly — no EVALUATION_INFO,
#      no Cell 6 re-run, no retraining, nothing recomputed.
#   2. Only if those artifacts do NOT exist does it fall back to reading
#      EVALUATION_INFO (in memory) or the cached evaluation_summary.json /
#      prediction CSVs from Cell 6, and generate the artifacts fresh.
#   3. Either way, it ends by zipping every generated/loaded output file and
#      triggering an automatic download.
#
# Does NOT modify Cells 1-6. Does NOT retrain. Does NOT run Web Content,
# URL, or Cloud Fusion.
#
# IMPORTANT SCOPE NOTE (do not remove): the current implementation trains
# "text" (TF-IDF) and "behavioral" (lexical+behavioral folded together) as
# TWO INDEPENDENT Random Forest models — there is no single fused feature
# matrix / single classifier producing one combined number. This cell
# reports both models side by side rather than inventing a fused metric.
# ============================================================================

import os
import sys
import json
import shutil
import zipfile
import warnings
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix

EMAIL_MODELS = ["text", "behavioral"]  # the ONLY models that belong to this paper
DATASET_LABEL = "Email Phishing Detection — Nazario + SpamAssassin"

# ----------------------------------------------------------------------------
# 1. OUTPUT FOLDERS — the exact, already-known paths (unchanged from the
#    previous successful run; nothing invented).
# ----------------------------------------------------------------------------

_ROOT_BASE = PATHS.get("root", "/content/drive/MyDrive/MAL-PhishNet") if "PATHS" in globals() else \
    "/content/drive/MyDrive/MAL-PhishNet"
EMAIL_REPORT_ROOT = os.path.join(_ROOT_BASE, "Email_Experiment_Report_NazarioSpamAssassin")
EMAIL_REPORT_PATHS = {
    "root": EMAIL_REPORT_ROOT,
    "figures": os.path.join(EMAIL_REPORT_ROOT, "Figures"),
    "tables": os.path.join(EMAIL_REPORT_ROOT, "Tables"),
    "reports": os.path.join(EMAIL_REPORT_ROOT, "Reports"),
}
for _p in EMAIL_REPORT_PATHS.values():
    os.makedirs(_p, exist_ok=True)

metrics_csv_path = os.path.join(EMAIL_REPORT_PATHS["tables"], "Table_Email_Experiment_Metrics.csv")
json_path = os.path.join(EMAIL_REPORT_PATHS["reports"], "email_experiment_paper_results.json")
_confusion_matrix_paths = {
    m: os.path.join(EMAIL_REPORT_PATHS["figures"], f"confusion_matrix_{m}_email.png") for m in EMAIL_MODELS
}
roc_path = os.path.join(EMAIL_REPORT_PATHS["figures"], "roc_curve_email_experiment.png")

_EXPECTED_ARTIFACTS = [metrics_csv_path, json_path, roc_path] + list(_confusion_matrix_paths.values())

print("=" * 78)
print(DATASET_LABEL.center(78))
print("=" * 78)

# ----------------------------------------------------------------------------
# 2. CHECK WHETHER THE PAPER-READY ARTIFACTS ALREADY EXIST
# ----------------------------------------------------------------------------

_artifacts_present = {p: os.path.isfile(p) for p in _EXPECTED_ARTIFACTS}
ARTIFACTS_ALREADY_EXIST = all(_artifacts_present.values())

print("Checking for previously-generated paper artifacts...")
for _p, _present in _artifacts_present.items():
    print(f"  [{'FOUND' if _present else 'missing'}] {_p}")

if ARTIFACTS_ALREADY_EXIST:
    # --------------------------------------------------------------------
    # 3A. ARTIFACTS ALREADY EXIST — load them directly, recompute nothing,
    #     do NOT touch EVALUATION_INFO or Cell 6.
    # --------------------------------------------------------------------
    print("-" * 78)
    print("All paper-ready artifacts already exist. Loading directly "
          "(no recomputation, no Cell 6 access).")

    email_metrics_table = pd.read_csv(metrics_csv_path)
    with open(json_path, "r", encoding="utf-8") as fh:
        paper_results = json.load(fh)

    print(email_metrics_table.to_string(index=False))
    print(f"\nLoaded existing: {metrics_csv_path}")
    print(f"Loaded existing: {json_path}")
    for m, p in _confusion_matrix_paths.items():
        print(f"Verified existing: {p}")
    print(f"Verified existing: {roc_path}")

else:
    # --------------------------------------------------------------------
    # 3B. ARTIFACTS MISSING — fall back to EVALUATION_INFO / cached Cell 6
    #     outputs, and generate the artifacts fresh. This branch is only
    #     reached if a prior run of this cell never completed.
    # --------------------------------------------------------------------
    print("-" * 78)
    print("Paper-ready artifacts not fully present — regenerating from "
          "cached Cell 6 evaluation results (no retraining).")

    def _load_evaluation_info():
        if "EVALUATION_INFO" in globals():
            return globals()["EVALUATION_INFO"]
        candidate_paths = []
        if "CELL6_PATHS" in globals():
            candidate_paths.append(os.path.join(CELL6_PATHS["root"], "evaluation_summary.json"))
        if "PATHS" in globals():
            candidate_paths.append(
                os.path.join(PATHS.get("root", ""), "outputs", "metrics", "evaluation")
            )
        for p in candidate_paths:
            if p and p.endswith(".json") and os.path.isfile(p):
                with open(p, "r", encoding="utf-8") as fh:
                    return json.load(fh)
            if p and os.path.isdir(p):
                for root, _, files in os.walk(p):
                    if "evaluation_summary.json" in files:
                        with open(os.path.join(root, "evaluation_summary.json"), "r", encoding="utf-8") as fh:
                            return json.load(fh)
        raise RuntimeError(
            "Could not find EVALUATION_INFO in memory or evaluation_summary.json on disk, "
            "and no complete set of paper-ready artifacts already exists on disk either. "
            "Run Cell 6 at least once before this cell."
        )

    EVALUATION_INFO = _load_evaluation_info()
    for _m in EMAIL_MODELS:
        if EVALUATION_INFO.get(_m) is None:
            raise RuntimeError(f"[{_m}] has no evaluation result in EVALUATION_INFO — cannot build paper report.")

    def _load_predictions(model_name: str) -> pd.DataFrame:
        result = EVALUATION_INFO[model_name]
        path = result["prediction_path"]
        if not os.path.isfile(path):
            raise FileNotFoundError(
                f"[{model_name}] Cached predictions CSV not found at {path}. "
                "Re-run Cell 6 to regenerate it (do not re-run Cells 1-5)."
            )
        return pd.read_csv(path)

    EMAIL_PREDICTIONS = {m: _load_predictions(m) for m in EMAIL_MODELS}

    # --- metrics table ---
    _rows = []
    for model_name in EMAIL_MODELS:
        r = EVALUATION_INFO[model_name]
        m = r["test_metrics"]
        _rows.append({
            "Model": {"text": "Textual (TF-IDF)", "behavioral": "Lexical + Behavioral"}[model_name],
            "Dataset": "Nazario + SpamAssassin",
            "Classifier": r["algorithm"],
            "Test samples": r["test_samples"],
            "Threshold": round(r["threshold_selection"]["threshold"], 4),
            "Accuracy": round(m["accuracy"], 4),
            "Precision": round(m["precision"], 4),
            "Recall": round(m["recall"], 4),
            "F1-score": round(m["f1"], 4),
            "ROC-AUC": round(m["roc_auc"], 4) if m.get("roc_auc") is not None else None,
            "PR-AUC": round(m["pr_auc"], 4) if m.get("pr_auc") is not None else None,
        })
    email_metrics_table = pd.DataFrame(_rows)
    email_metrics_table.to_csv(metrics_csv_path, index=False)
    print(email_metrics_table.to_string(index=False))
    print(f"\nSaved: {metrics_csv_path}")

    # --- confusion matrices ---
    for model_name in EMAIL_MODELS:
        df = EMAIL_PREDICTIONS[model_name]
        cm = confusion_matrix(df["label"], df["prediction_selected_threshold"])
        label_display = {"text": "Textual (TF-IDF) Model", "behavioral": "Lexical + Behavioral Model"}[model_name]

        fig, ax = plt.subplots(figsize=(5, 4.5), constrained_layout=True)
        im = ax.imshow(cm, cmap="Blues")
        ax.set_title(f"Figure — Confusion Matrix\n{label_display} | {DATASET_LABEL} | Test split", fontsize=10)
        ax.set_xlabel("Predicted label")
        ax.set_ylabel("True label")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Legitimate", "Phishing"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["Legitimate", "Phishing"])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                         color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        out_path = _confusion_matrix_paths[model_name]
        fig.savefig(out_path, dpi=200)
        plt.close(fig)
        print(f"Saved: {out_path}")

    # --- ROC curve ---
    fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
    for model_name in EMAIL_MODELS:
        df = EMAIL_PREDICTIONS[model_name]
        fpr, tpr, _ = roc_curve(df["label"], df["phishing_probability"])
        roc_auc_value = auc(fpr, tpr)
        label_display = {"text": "Textual (TF-IDF)", "behavioral": "Lexical + Behavioral"}[model_name]
        ax.plot(fpr, tpr, label=f"{label_display} (AUC = {roc_auc_value:.4f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"Figure — ROC Curve\n{DATASET_LABEL} | Test split", fontsize=10)
    ax.legend(loc="lower right", fontsize=8)
    fig.savefig(roc_path, dpi=200)
    plt.close(fig)
    print(f"Saved: {roc_path}")

    # --- paper-ready JSON ---
    paper_results = {
        "experiment_name": DATASET_LABEL,
        "dataset": {
            "phishing_source": "Nazario Phishing Email Corpus (Zenodo record 8339691)",
            "legitimate_source": "SpamAssassin Public Corpus (easy_ham + hard_ham)",
            "phishing_count_after_cleaning": 1564,
            "legitimate_count_after_cleaning": 2720,
            "total": 4284,
            "split": {"train": 2998, "validation": 643, "test": 643},
        },
        "classifier": "Random Forest (identical hyperparameters for both feature groups; "
                      "each feature group trained as an independent model — no single "
                      "fused feature matrix exists in the current implementation)",
        "results": {m: EVALUATION_INFO[m]["test_metrics"] for m in EMAIL_MODELS},
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_evaluation_signature": EVALUATION_INFO.get("evaluation_signature"),
        "excluded_from_this_paper": ["url", "web_content", "fusion"],
    }
    with open(json_path, "w", encoding="utf-8") as fh:
        json.dump(paper_results, fh, indent=2, default=str)
    print(f"Saved: {json_path}")

print("-" * 78)
print("Cell 7B complete. Table_Email_Experiment_Metrics.csv and "
      "email_experiment_paper_results.json are the source of results/figures "
      "for this paper's email experiment.")
print("=" * 78)

# ----------------------------------------------------------------------------
# 4. ZIP ALL PAPER-READY OUTPUTS + AUTOMATIC DOWNLOAD
# ----------------------------------------------------------------------------
# Zips exactly the files this cell just verified/loaded/generated:
# metrics_csv_path, _confusion_matrix_paths (both models), roc_path,
# json_path — all under EMAIL_REPORT_ROOT. Triggers a browser download.
# Does not touch Cells 1-6 or any other files.

_HAS_COLAB = "google.colab" in sys.modules

EMAIL_ZIP_PATH = os.path.join(_ROOT_BASE, "Email_Experiment_Report_NazarioSpamAssassin.zip")
EMAIL_LOCAL_ZIP_PATH = None

try:
    if os.path.isfile(EMAIL_ZIP_PATH):
        os.remove(EMAIL_ZIP_PATH)
    with zipfile.ZipFile(EMAIL_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zip_file:
        for _artifact_path in _EXPECTED_ARTIFACTS:
            if os.path.isfile(_artifact_path):
                archive_name = os.path.relpath(_artifact_path, os.path.dirname(EMAIL_REPORT_ROOT))
                zip_file.write(_artifact_path, archive_name)
    _zip_size_mb = os.path.getsize(EMAIL_ZIP_PATH) / (1024 ** 2)
    print(f"Email experiment report package created: {EMAIL_ZIP_PATH} ({_zip_size_mb:.2f} MB)")
except Exception as exc:
    print(f"Failed to create ZIP archive: {exc}")
    EMAIL_ZIP_PATH = None

if EMAIL_ZIP_PATH and _HAS_COLAB:
    _intended_local_path = "/content/Email_Experiment_Report_NazarioSpamAssassin.zip"
    try:
        if os.path.abspath(EMAIL_ZIP_PATH) == os.path.abspath(_intended_local_path):
            EMAIL_LOCAL_ZIP_PATH = EMAIL_ZIP_PATH
        else:
            EMAIL_LOCAL_ZIP_PATH = _intended_local_path
            shutil.copy2(EMAIL_ZIP_PATH, EMAIL_LOCAL_ZIP_PATH)
            print(f"Copied ZIP to local Colab disk for reliable download: {EMAIL_LOCAL_ZIP_PATH}")
    except Exception as exc:
        print(f"Could not copy ZIP to local disk before download: {exc}")
        EMAIL_LOCAL_ZIP_PATH = None

if EMAIL_ZIP_PATH:
    if _HAS_COLAB:
        _download_source = EMAIL_LOCAL_ZIP_PATH or EMAIL_ZIP_PATH
        try:
            from google.colab import files as colab_files
            colab_files.download(_download_source)
            print(f"Triggered Colab download for {_download_source}")
        except Exception as exc:
            print(f"Could not trigger Colab download automatically: {exc}")
            print(f"Report saved at:\n  {EMAIL_ZIP_PATH}")
    else:
        print("Not running in Colab — automatic browser download skipped.")
        print(f"ZIP package ready at: {EMAIL_ZIP_PATH}")

              Email Phishing Detection — Nazario + SpamAssassin               
Checking for previously-generated paper artifacts...
  [FOUND] /content/drive/MyDrive/MAL-PhishNet/Email_Experiment_Report_NazarioSpamAssassin/Tables/Table_Email_Experiment_Metrics.csv
  [FOUND] /content/drive/MyDrive/MAL-PhishNet/Email_Experiment_Report_NazarioSpamAssassin/Reports/email_experiment_paper_results.json
  [FOUND] /content/drive/MyDrive/MAL-PhishNet/Email_Experiment_Report_NazarioSpamAssassin/Figures/roc_curve_email_experiment.png
  [FOUND] /content/drive/MyDrive/MAL-PhishNet/Email_Experiment_Report_NazarioSpamAssassin/Figures/confusion_matrix_text_email.png
  [FOUND] /content/drive/MyDrive/MAL-PhishNet/Email_Experiment_Report_NazarioSpamAssassin/Figures/confusion_matrix_behavioral_email.png
------------------------------------------------------------------------------
All paper-ready artifacts already exist. Loading directly (no recomputation, no Cell 6 access).
               Model            

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Triggered Colab download for /content/Email_Experiment_Report_NazarioSpamAssassin.zip
